In [1]:
# ============================================================
# CELL 1: EXPLORE NeurIPS Open Polymer Prediction 2025
# ============================================================
import pandas as pd
import numpy as np

BASE = "/kaggle/input/competitions/neurips-open-polymer-prediction-2025/"

files = {
    "train":     BASE + "train.csv",
    "test":      BASE + "test.csv",
    "sample_submission": BASE + "sample_submission.csv",
    "dataset1":  BASE + "train_supplement/dataset1.csv",
    "dataset2":  BASE + "train_supplement/dataset2.csv",
    "dataset3":  BASE + "train_supplement/dataset3.csv",
    "dataset4":  BASE + "train_supplement/dataset4.csv",
}

dfs = {}
for name, path in files.items():
    try:
        df = pd.read_csv(path)
        dfs[name] = df
        print(f"\n{'='*60}")
        print(f"FILE  : {name}")
        print(f"Shape : {df.shape[0]} rows × {df.shape[1]} cols")
        print(f"Cols  : {list(df.columns)}")
        print(f"\nNull counts:\n{df.isnull().sum().to_string()}")
        print(f"\nFirst 3 rows:\n{df.head(3).to_string()}")
        num = df.select_dtypes(include=[np.number]).columns.tolist()
        if num:
            print(f"\nNumeric stats:\n{df[num].describe().round(3).to_string()}")
    except Exception as e:
        print(f"\n✗ {name}: {e}")

print(f"\n{'='*60}")
print("SUMMARY")
print(f"{'='*60}")
for name, df in dfs.items():
    print(f"  {name:<20} {df.shape[0]:>6} rows  {df.shape[1]:>3} cols  |  {list(df.columns)}")


FILE  : train
Shape : 7973 rows × 7 cols
Cols  : ['id', 'SMILES', 'Tg', 'FFV', 'Tc', 'Density', 'Rg']

Null counts:
id            0
SMILES        0
Tg         7462
FFV         943
Tc         7236
Density    7360
Rg         7359

First 3 rows:
       id                                                                                                                                  SMILES  Tg       FFV        Tc  Density  Rg
0   87817                                                                                                              *CC(*)c1ccccc1C(=O)OCCCCCC NaN  0.374645  0.205667      NaN NaN
1  106919                                                      *Nc1ccc([C@H](CCC)c2ccc(C3(c4ccc([C@@H](CCC)c5ccc(N*)cc5)cc4)CCC(CCCCC)CC3)cc2)cc1 NaN  0.370410       NaN      NaN NaN
2  388772  *Oc1ccc(S(=O)(=O)c2ccc(Oc3ccc(C4(c5ccc(Oc6ccc(S(=O)(=O)c7ccc(Oc8ccc(C=C9CCCC(=Cc%10ccc(*)cc%10)C9=O)cc8)cc7)cc6)cc5)CCCCC4)cc3)cc2)cc1 NaN  0.378860       NaN      NaN NaN

Numeric stats:
        

In [2]:
# ================================================================
# CELL 0: Install RDKit
# ================================================================
import subprocess
subprocess.run(["pip", "install", "rdkit", "-q"], capture_output=True)
from rdkit import Chem
print("✓ RDKit OK")

# ================================================================
# CELL 1: Full Training Code
# ================================================================
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import r2_score, mean_absolute_error
import warnings, time, os, logging

# ── Suppress ALL warnings including RDKit deprecations ──────────
warnings.filterwarnings('ignore')
logging.getLogger('rdkit').setLevel(logging.ERROR)
os.environ['RDKIT_LOG_LEVEL'] = 'error'

from rdkit import Chem, RDLogger
RDLogger.DisableLog('rdApp.*')          # ← silences all RDKit warnings

from rdkit.Chem import Descriptors, rdMolDescriptors, AllChem
from rdkit.ML.Descriptors import MoleculeDescriptors
from rdkit.Chem import rdFingerprintGenerator  # new API (no deprecation warning)

torch.manual_seed(42)
np.random.seed(42)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device : {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU    : {torch.cuda.get_device_name(0)}")

# ================================================================
# SECTION 1 — LOAD & MERGE ALL DATA
# ================================================================
BASE = "/kaggle/input/competitions/neurips-open-polymer-prediction-2025/"

print("\n[1/6] Loading all data files...")

train = pd.read_csv(BASE + "train.csv")
test  = pd.read_csv(BASE + "test.csv")
ds1   = pd.read_csv(BASE + "train_supplement/dataset1.csv").rename(columns={"TC_mean": "Tc"})
ds3   = pd.read_csv(BASE + "train_supplement/dataset3.csv")
ds4   = pd.read_csv(BASE + "train_supplement/dataset4.csv")

TARGETS = ["Tg", "FFV", "Tc", "Density", "Rg"]

all_train = pd.concat([
    train[["SMILES","Tg","FFV","Tc","Density","Rg"]],
    ds3[["SMILES","Tg"]],
    ds4[["SMILES","FFV"]],
    ds1[["SMILES","Tc"]],
], ignore_index=True)

print(f"  Combined training rows : {len(all_train)}")
for t in TARGETS:
    if t in all_train.columns:
        print(f"    {t:10s}: {all_train[t].notna().sum():5d} labeled rows")

# ================================================================
# SECTION 2 — FEATURE EXTRACTION FROM SMILES
# ================================================================
print("\n[2/6] Extracting features from SMILES...")

DESCRIPTOR_NAMES = [
    "MolWt","HeavyAtomMolWt","ExactMolWt","NumValenceElectrons",
    "FpDensityMorgan1","FpDensityMorgan2","FpDensityMorgan3",
    "MolLogP","MolMR","TPSA",
    "NumHAcceptors","NumHDonors","NumRotatableBonds","NumAromaticRings",
    "NumSaturatedRings","NumAliphaticRings","NumAromaticHeterocycles",
    "NumSaturatedHeterocycles","NumAliphaticHeterocycles",
    "RingCount","NumHeteroatoms","NumRadicalElectrons",
    "FractionCSP3","HeavyAtomCount","NHOHCount","NOCount",
    "MaxAbsEStateIndex","MinAbsEStateIndex","MaxEStateIndex","MinEStateIndex",
    "Chi0","Chi0n","Chi0v","Chi1","Chi1n","Chi1v","Chi2n","Chi2v",
    "Kappa1","Kappa2","Kappa3",
    "LabuteASA","PEOE_VSA1","PEOE_VSA2","PEOE_VSA3",
    "SMR_VSA1","SMR_VSA2","SlogP_VSA1","SlogP_VSA2","BalabanJ"
]
calc = MoleculeDescriptors.MolecularDescriptorCalculator(DESCRIPTOR_NAMES)
MORGAN_BITS = 512
N_DESC = len(DESCRIPTOR_NAMES)

# Use new MorganGenerator API (no deprecation warning)
morgan_gen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=MORGAN_BITS)

def smiles_to_features(smiles):
    try:
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            return np.full(N_DESC + MORGAN_BITS, np.nan, dtype=np.float32)
        desc = np.array(calc.CalcDescriptors(mol), dtype=np.float32)
        fp   = np.array(morgan_gen.GetFingerprintAsNumPy(mol), dtype=np.float32)
        return np.concatenate([desc, fp])
    except:
        return np.full(N_DESC + MORGAN_BITS, np.nan, dtype=np.float32)

def extract(df, label=""):
    out = []
    total = len(df)
    for i, smi in enumerate(df["SMILES"].values):
        out.append(smiles_to_features(smi))
        if (i+1) % 2000 == 0:
            print(f"  {label}: {i+1}/{total}")
    print(f"  {label}: {total}/{total} done ✓")
    return np.array(out, dtype=np.float32)

X_all  = extract(all_train, "train")
X_test = extract(test,      "test")
y_all  = all_train[TARGETS].values.astype(np.float32)

# Drop rows where SMILES completely failed
valid = ~np.isnan(X_all).all(axis=1)
X_all = X_all[valid]; y_all = y_all[valid]
print(f"  Valid rows: {valid.sum()}")

# Impute NaN features with column median
feat_medians = np.nanmedian(X_all, axis=0)
for j in range(X_all.shape[1]):
    X_all[np.isnan(X_all[:,j]),  j] = feat_medians[j]
    X_test[np.isnan(X_test[:,j]),j] = feat_medians[j]

# Scale features
scaler_X  = RobustScaler()
X_all_sc  = scaler_X.fit_transform(X_all).astype(np.float32)
X_test_sc = scaler_X.transform(X_test).astype(np.float32)

# Scale targets per-column (NaN stays NaN)
target_means = np.nanmean(y_all, axis=0)
target_stds  = np.where(np.nanstd(y_all, axis=0) < 1e-6, 1.0, np.nanstd(y_all, axis=0))
y_all_sc     = (y_all - target_means) / target_stds

INPUT_DIM = X_all_sc.shape[1]
print(f"  Feature dim : {INPUT_DIM}")
print(f"  Target means: {dict(zip(TARGETS, target_means.round(3)))}")

# ================================================================
# SECTION 3 — DATASET
# ================================================================
class PolymerDataset(Dataset):
    def __init__(self, X, y):
        self.X    = torch.tensor(X, dtype=torch.float32)
        self.y    = torch.tensor(np.nan_to_num(y, nan=0.0), dtype=torch.float32)
        self.mask = torch.tensor(~np.isnan(y), dtype=torch.float32)
    def __len__(self): return len(self.X)
    def __getitem__(self, i): return self.X[i], self.y[i], self.mask[i]

# ================================================================
# SECTION 4 — MODEL
# ================================================================
class ResBlock(nn.Module):
    def __init__(self, dim, dropout=0.15):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim, dim*2), nn.LayerNorm(dim*2), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(dim*2, dim), nn.LayerNorm(dim),
        )
        self.act = nn.GELU()
    def forward(self, x): return self.act(x + self.net(x))

class PolymerGNN(nn.Module):
    def __init__(self, input_dim, n_targets=5, n_desc=50, n_fp=512,
                 hidden=512, n_res=6, dropout=0.15):
        super().__init__()
        self.n_desc = n_desc
        self.n_fp   = n_fp

        # Separate encoders for descriptor vs fingerprint
        self.desc_enc = nn.Sequential(
            nn.Linear(n_desc, 256), nn.LayerNorm(256), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(256, 256),    nn.LayerNorm(256), nn.GELU(),
        )
        self.fp_enc = nn.Sequential(
            nn.Linear(n_fp, 512),  nn.LayerNorm(512),  nn.GELU(), nn.Dropout(dropout),
            nn.Linear(512, 256),   nn.LayerNorm(256),  nn.GELU(),
        )

        # Attention-based fusion
        self.attn = nn.Sequential(
            nn.Linear(512, 128), nn.Tanh(),
            nn.Linear(128, 2),   nn.Softmax(dim=-1),
        )

        # Deep shared trunk
        self.input_proj = nn.Sequential(
            nn.Linear(512, hidden), nn.LayerNorm(hidden), nn.GELU(),
        )
        self.res_blocks = nn.ModuleList([ResBlock(hidden, dropout) for _ in range(n_res)])
        self.trunk_norm = nn.LayerNorm(hidden)

        # Per-target heads
        def head(h=128):
            return nn.Sequential(
                nn.Linear(hidden, h), nn.LayerNorm(h), nn.GELU(), nn.Dropout(dropout/2),
                nn.Linear(h, h//2), nn.GELU(), nn.Linear(h//2, 1)
            )
        self.heads = nn.ModuleList([head() for _ in range(n_targets)])

        # Cross-property correlation
        self.cross = nn.Sequential(
            nn.Linear(n_targets, 64), nn.GELU(), nn.Linear(64, n_targets)
        )

        # Weight init
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, nonlinearity='relu')
                if m.bias is not None: nn.init.zeros_(m.bias)
            elif isinstance(m, nn.LayerNorm):
                nn.init.ones_(m.weight); nn.init.zeros_(m.bias)

    def forward(self, x):
        desc = x[:, :self.n_desc]
        fp   = x[:, self.n_desc:self.n_desc + self.n_fp]
        d    = self.desc_enc(desc)
        f    = self.fp_enc(fp)
        w    = self.attn(torch.cat([d, f], dim=-1))
        fused= torch.cat([w[:,0:1]*d + w[:,1:2]*f] * 2, dim=-1)
        h    = self.input_proj(fused)
        for block in self.res_blocks: h = block(h)
        h    = self.trunk_norm(h)
        raw  = torch.cat([head(h) for head in self.heads], dim=1)
        return raw + 0.05 * self.cross(raw)

    def count_params(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)

# ================================================================
# SECTION 5 — MASKED LOSS + TRAIN/EVAL
# ================================================================
def masked_loss(preds, targets, masks):
    total = torch.tensor(0.0, device=preds.device, requires_grad=True)
    n = 0
    for i in range(preds.shape[1]):
        m = masks[:, i].bool()
        if m.sum() == 0: continue
        total = total + F.huber_loss(preds[m, i], targets[m, i], delta=1.0)
        n += 1
    return total / n if n > 0 else total

def train_epoch(model, loader, optimizer, scaler_amp):
    model.train()
    total = 0.0
    for X_b, y_b, mask_b in loader:
        X_b, y_b, mask_b = X_b.to(DEVICE), y_b.to(DEVICE), mask_b.to(DEVICE)
        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast('cuda', enabled=(DEVICE.type=='cuda')):
            loss = masked_loss(model(X_b), y_b, mask_b)
        scaler_amp.scale(loss).backward()
        scaler_amp.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler_amp.step(optimizer); scaler_amp.update()
        total += loss.item()
    return total / len(loader)

@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    P, T, M, val_loss = [], [], [], 0.0
    for X_b, y_b, mask_b in loader:
        X_b, y_b, mask_b = X_b.to(DEVICE), y_b.to(DEVICE), mask_b.to(DEVICE)
        preds = model(X_b)
        val_loss += masked_loss(preds, y_b, mask_b).item()
        P.append(preds.cpu().numpy()); T.append(y_b.cpu().numpy()); M.append(mask_b.cpu().numpy())
    P = np.concatenate(P) * target_stds + target_means
    T = np.concatenate(T) * target_stds + target_means
    M = np.concatenate(M).astype(bool)
    metrics, r2s = {}, []
    for i, t in enumerate(TARGETS):
        m = M[:, i]
        if m.sum() < 5:
            metrics[t] = {"R2": np.nan, "MAE": np.nan, "MAPE": np.nan}
            continue
        r2   = r2_score(T[m, i], P[m, i])
        mae  = mean_absolute_error(T[m, i], P[m, i])
        mape = np.mean(np.abs((T[m,i]-P[m,i]) / (np.abs(T[m,i])+1e-6))) * 100
        metrics[t] = {"R2": r2, "MAE": mae, "MAPE": mape}
        r2s.append(r2)
    return val_loss/len(loader), np.nanmean(r2s) if r2s else np.nan, metrics, P, T, M

# ================================================================
# SECTION 6 — RUN TRAINING
# ================================================================
print("\n[3/6] Splitting data...")
X_tr, X_va, y_tr, y_va = train_test_split(
    X_all_sc, y_all_sc, test_size=0.15, random_state=42
)
print(f"  Train: {len(X_tr)} | Val: {len(X_va)}")

train_loader = DataLoader(PolymerDataset(X_tr, y_tr), batch_size=128, shuffle=True,
                          num_workers=2, pin_memory=True, drop_last=True)
val_loader   = DataLoader(PolymerDataset(X_va, y_va), batch_size=256, shuffle=False,
                          num_workers=2, pin_memory=True)

print(f"\n[4/6] Building model...")
model = PolymerGNN(INPUT_DIM, n_targets=5, n_desc=N_DESC, n_fp=MORGAN_BITS,
                   hidden=512, n_res=6, dropout=0.15).to(DEVICE)
print(f"  Parameters: {model.count_params():,}")

optimizer  = torch.optim.AdamW(model.parameters(), lr=5e-4, weight_decay=1e-4)
scheduler  = torch.optim.lr_scheduler.OneCycleLR(
    optimizer, max_lr=5e-4, epochs=300, steps_per_epoch=len(train_loader),
    pct_start=0.1, anneal_strategy='cos', final_div_factor=500)
scaler_amp = torch.amp.GradScaler('cuda', enabled=(DEVICE.type=='cuda'))

print(f"\n[5/6] Training (up to 300 epochs, early stop at patience=50)...")
best_r2, best_state, no_improve = -np.inf, None, 0
t0 = time.time()

for epoch in range(1, 301):
    tr_loss = train_epoch(model, train_loader, optimizer, scaler_amp)
    scheduler.step()

    if epoch % 5 == 0 or epoch <= 3:
        va_loss, avg_r2, metrics, _, _, _ = evaluate(model, val_loader)

        if avg_r2 > best_r2:
            best_r2    = avg_r2
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            no_improve = 0
        else:
            no_improve += 1

        if epoch % 25 == 0 or epoch <= 3:
            print(f"\n  Ep{epoch:3d} | Tr:{tr_loss:.4f} Va:{va_loss:.4f} "
                  f"AvgR²:{avg_r2:.4f} [{int(time.time()-t0)}s]")
            for t in TARGETS:
                r2   = metrics[t]['R2']
                mae  = metrics[t]['MAE']
                mape = metrics[t]['MAPE']
                flag = "✓" if (not np.isnan(r2) and r2 >= 0.90) else " "
                print(f"    {t:10s}: R²={r2:.4f}  MAE={mae:.4f}  MAPE={mape:.1f}% {flag}")

        if no_improve * 5 >= 50:
            print(f"\n  ⚡ Early stopping at epoch {epoch}  (best R²={best_r2:.4f})")
            break

model.load_state_dict(best_state)

# ── Final evaluation ──────────────────────────────────────────────
print(f"\n[6/6] Final Validation Results:")
_, avg_r2, metrics, _, _, _ = evaluate(model, val_loader)
print(f"\n{'='*52}")
print(f"  {'Target':<12} {'R²':>8} {'MAE':>10} {'MAPE%':>8}")
print(f"{'='*52}")
for t in TARGETS:
    m    = metrics[t]
    flag = "✓" if (not np.isnan(m['R2']) and m['R2'] >= 0.90) else "→"
    print(f"  {t:<12} {m['R2']:>8.4f} {m['MAE']:>10.4f} {m['MAPE']:>7.1f}%  {flag}")
print(f"{'='*52}")
print(f"  {'AVERAGE':<12} {avg_r2:>8.4f}")

# ── Generate submission ───────────────────────────────────────────
model.eval()
with torch.no_grad():
    preds_sc = model(torch.tensor(X_test_sc).to(DEVICE)).cpu().numpy()
preds_real = preds_sc * target_stds + target_means

submission = pd.DataFrame({"id": test["id"]})
for i, t in enumerate(TARGETS):
    submission[t] = preds_real[:, i]
submission.to_csv("/kaggle/working/submission.csv", index=False)
print(f"\n✓ submission.csv saved")
print(submission.to_string())

torch.save({
    "model_state":  model.state_dict(),
    "scaler_X":     scaler_X,
    "target_means": target_means,
    "target_stds":  target_stds,
    "feat_medians": feat_medians,
    "metrics":      metrics,
}, "/kaggle/working/model1_gnn.pth")
print("✓ model1_gnn.pth saved")

✓ RDKit OK
Device : cuda
GPU    : Tesla T4

[1/6] Loading all data files...
  Combined training rows : 9755
    Tg        :   557 labeled rows
    FFV       :  7892 labeled rows
    Tc        :  1611 labeled rows
    Density   :   613 labeled rows
    Rg        :   614 labeled rows

[2/6] Extracting features from SMILES...
  train: 2000/9755
  train: 4000/9755
  train: 6000/9755
  train: 8000/9755
  train: 9755/9755 done ✓
  test: 3/3 done ✓
  Valid rows: 9755
  Feature dim : 562
  Target means: {'Tg': np.float32(99.693), 'FFV': np.float32(0.367), 'Tc': np.float32(0.257), 'Density': np.float32(0.985), 'Rg': np.float32(16.42)}

[3/6] Splitting data...
  Train: 8291 | Val: 1464

[4/6] Building model...
  Parameters: 7,497,036

[5/6] Training (up to 300 epochs, early stop at patience=50)...

  Ep  1 | Tr:0.4219 Va:0.3598 AvgR²:0.0032 [3s]
    Tg        : R²=-0.0543  MAE=87.9522  MAPE=280.1%  
    FFV       : R²=0.1017  MAE=0.0198  MAPE=5.3%  
    Tc        : R²=0.3504  MAE=0.0523  MAPE=25

In [3]:
# ================================================================
# MODEL 2 — TRANSFORMER / LLM (TransPolymer-style)
# NeurIPS Open Polymer Prediction 2025
# Input  : Raw SMILES string (character-level tokenization)
# Targets: Tg | FFV | Tc | Density | Rg
# Method : SMILES → Token IDs → Positional Encoding →
#          Multi-Layer Transformer Encoder →
#          [CLS] token → 5 prediction heads
#          + RDKit features from Model 1 fused at head level
# Ref    : TransPolymer (OSTI 2422347), polyBART, Multimodal Transformer
# ================================================================

import subprocess
subprocess.run(["pip", "install", "rdkit", "-q"], capture_output=True)

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import r2_score, mean_absolute_error
import warnings, time, math
warnings.filterwarnings('ignore')

from rdkit import Chem, RDLogger
RDLogger.DisableLog('rdApp.*')
from rdkit.Chem import Descriptors, AllChem
from rdkit.ML.Descriptors import MoleculeDescriptors
from rdkit.Chem import rdFingerprintGenerator

torch.manual_seed(42)
np.random.seed(42)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device : {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU    : {torch.cuda.get_device_name(0)}")

# ================================================================
# SECTION 1 — LOAD DATA (same as Model 1)
# ================================================================
BASE = "/kaggle/input/competitions/neurips-open-polymer-prediction-2025/"
print("\n[1/7] Loading data...")

train = pd.read_csv(BASE + "train.csv")
test  = pd.read_csv(BASE + "test.csv")
ds1   = pd.read_csv(BASE + "train_supplement/dataset1.csv").rename(columns={"TC_mean":"Tc"})
ds3   = pd.read_csv(BASE + "train_supplement/dataset3.csv")
ds4   = pd.read_csv(BASE + "train_supplement/dataset4.csv")

TARGETS = ["Tg", "FFV", "Tc", "Density", "Rg"]

all_train = pd.concat([
    train[["SMILES","Tg","FFV","Tc","Density","Rg"]],
    ds3[["SMILES","Tg"]],
    ds4[["SMILES","FFV"]],
    ds1[["SMILES","Tc"]],
], ignore_index=True)

print(f"  Total rows: {len(all_train)}")
for t in TARGETS:
    print(f"    {t:10s}: {all_train[t].notna().sum():5d} labeled")

# ================================================================
# SECTION 2 — SMILES TOKENIZER
# ================================================================
print("\n[2/7] Building SMILES tokenizer...")

# Polymer SMILES special tokens + chemistry-aware vocabulary
SPECIAL_TOKENS = ['<PAD>', '<CLS>', '<UNK>', '<MASK>']

# All chemical tokens that appear in polymer SMILES
ATOM_TOKENS = [
    'C','N','O','S','F','Cl','Br','I','P','B','Si','Se','Te',
    'c','n','o','s','p','b',
    'H','Li','Na','K','Ca','Mg','Al','Zn','Cu','Fe','Co','Ni',
    '[C@@H]','[C@H]','[C@@]','[C@]','[NH]','[NH2]','[NH3+]',
    '[O-]','[N+]','[n+]','[S+]','[s+]','[nH]','[OH]',
    '[Si]','[se]','[te]','[CH2]','[CH]','[cH]',
]
BOND_TOKENS = ['=','#','/','\\','-',':']
RING_TOKENS  = [str(i) for i in range(1, 20)] + ['%10','%11','%12','%13','%14','%15']
BRANCH_TOKENS= ['(',')']
STAR_TOKEN   = ['*']   # polymer repeat unit marker

ALL_TOKENS = SPECIAL_TOKENS + ATOM_TOKENS + BOND_TOKENS + RING_TOKENS + BRANCH_TOKENS + STAR_TOKEN

# Build vocabulary
tok2id = {tok: i for i, tok in enumerate(ALL_TOKENS)}
id2tok = {i: tok for tok, i in tok2id.items()}

PAD_ID  = tok2id['<PAD>']
CLS_ID  = tok2id['<CLS>']
UNK_ID  = tok2id['<UNK>']
VOCAB_SIZE = len(ALL_TOKENS)
print(f"  Vocabulary size: {VOCAB_SIZE}")


def tokenize_smiles(smiles, max_len=128):
    """
    Tokenize a SMILES string into token IDs.
    Uses greedy longest-match tokenization.
    Prepends <CLS> token (used for property prediction).
    """
    tokens = [CLS_ID]
    i = 0
    s = smiles.strip()

    while i < len(s) and len(tokens) < max_len:
        matched = False
        # Try longest match first (up to 6 chars for things like [C@@H])
        for length in [6, 5, 4, 3, 2, 1]:
            chunk = s[i:i+length]
            if chunk in tok2id:
                tokens.append(tok2id[chunk])
                i += length
                matched = True
                break
        if not matched:
            tokens.append(UNK_ID)
            i += 1

    # Pad to max_len
    tokens = tokens[:max_len]
    pad_len = max_len - len(tokens)
    tokens  = tokens + [PAD_ID] * pad_len
    return tokens


MAX_LEN = 128
print(f"  Max sequence length: {MAX_LEN}")

# Test tokenizer
sample = train['SMILES'].iloc[0]
toks   = tokenize_smiles(sample, MAX_LEN)
print(f"  Sample SMILES length: {len(sample)} chars → {sum(t!=PAD_ID for t in toks)} tokens")

# ================================================================
# SECTION 3 — RDKit FEATURES (reuse from Model 1, fused at head)
# ================================================================
print("\n[3/7] Extracting RDKit features (descriptor fusion)...")

DESCRIPTOR_NAMES = [
    "MolWt","HeavyAtomMolWt","ExactMolWt","NumValenceElectrons",
    "FpDensityMorgan1","FpDensityMorgan2","FpDensityMorgan3",
    "MolLogP","MolMR","TPSA",
    "NumHAcceptors","NumHDonors","NumRotatableBonds","NumAromaticRings",
    "NumSaturatedRings","NumAliphaticRings","NumAromaticHeterocycles",
    "NumSaturatedHeterocycles","NumAliphaticHeterocycles",
    "RingCount","NumHeteroatoms","NumRadicalElectrons",
    "FractionCSP3","HeavyAtomCount","NHOHCount","NOCount",
    "MaxAbsEStateIndex","MinAbsEStateIndex","MaxEStateIndex","MinEStateIndex",
    "Chi0","Chi0n","Chi0v","Chi1","Chi1n","Chi1v","Chi2n","Chi2v",
    "Kappa1","Kappa2","Kappa3",
    "LabuteASA","PEOE_VSA1","PEOE_VSA2","PEOE_VSA3",
    "SMR_VSA1","SMR_VSA2","SlogP_VSA1","SlogP_VSA2","BalabanJ"
]
calc       = MoleculeDescriptors.MolecularDescriptorCalculator(DESCRIPTOR_NAMES)
morgan_gen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=256)
N_DESC     = len(DESCRIPTOR_NAMES)
N_FP       = 256
N_RDKIT    = N_DESC + N_FP    # 50 + 256 = 306


def smiles_to_rdkit(smiles):
    try:
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            return np.full(N_RDKIT, np.nan, dtype=np.float32)
        desc = np.array(calc.CalcDescriptors(mol), dtype=np.float32)
        fp   = np.array(morgan_gen.GetFingerprintAsNumPy(mol), dtype=np.float32)
        return np.concatenate([desc, fp])
    except:
        return np.full(N_RDKIT, np.nan, dtype=np.float32)


def extract_rdkit(df, label=""):
    out = []
    for i, smi in enumerate(df["SMILES"].values):
        out.append(smiles_to_rdkit(smi))
        if (i+1) % 2000 == 0:
            print(f"  {label}: {i+1}/{len(df)}")
    print(f"  {label}: {len(df)}/{len(df)} ✓")
    return np.array(out, dtype=np.float32)


rdkit_train = extract_rdkit(all_train, "train")
rdkit_test  = extract_rdkit(test,      "test")

# Impute + scale
feat_medians = np.nanmedian(rdkit_train, axis=0)
for j in range(rdkit_train.shape[1]):
    rdkit_train[np.isnan(rdkit_train[:,j]), j] = feat_medians[j]
    rdkit_test[np.isnan(rdkit_test[:,j]),   j] = feat_medians[j]

scaler_X    = RobustScaler()
rdkit_train = scaler_X.fit_transform(rdkit_train).astype(np.float32)
rdkit_test  = scaler_X.transform(rdkit_test).astype(np.float32)

# Tokenize all SMILES
print("  Tokenizing SMILES sequences...")
token_train = np.array([tokenize_smiles(s, MAX_LEN) for s in all_train["SMILES"]], dtype=np.int32)
token_test  = np.array([tokenize_smiles(s, MAX_LEN) for s in test["SMILES"]],      dtype=np.int32)
print(f"  Token matrix: {token_train.shape}")

# Targets
y_all        = all_train[TARGETS].values.astype(np.float32)
target_means = np.nanmean(y_all, axis=0)
target_stds  = np.where(np.nanstd(y_all, axis=0) < 1e-6, 1.0, np.nanstd(y_all, axis=0))
y_all_sc     = (y_all - target_means) / target_stds

print(f"  Features: {N_RDKIT} RDKit + {MAX_LEN} token sequence")

# ================================================================
# SECTION 4 — DATASET
# ================================================================
class TransPolymerDataset(Dataset):
    def __init__(self, tokens, rdkit_feats, y):
        self.tokens = torch.tensor(tokens, dtype=torch.long)
        self.rdkit  = torch.tensor(rdkit_feats, dtype=torch.float32)
        self.y      = torch.tensor(np.nan_to_num(y, nan=0.0), dtype=torch.float32)
        self.mask   = torch.tensor(~np.isnan(y), dtype=torch.float32)

    def __len__(self): return len(self.tokens)

    def __getitem__(self, i):
        return self.tokens[i], self.rdkit[i], self.y[i], self.mask[i]

# ================================================================
# SECTION 5 — TRANSFORMER MODEL
# ================================================================

class PositionalEncoding(nn.Module):
    """
    Sinusoidal positional encoding.
    Injects position information into token embeddings.
    Essential for Transformer to understand sequence order.
    """
    def __init__(self, d_model, max_len=256, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer('pe', pe.unsqueeze(0))  # [1, max_len, d_model]

    def forward(self, x):
        return self.dropout(x + self.pe[:, :x.size(1)])


class TransPolymerModel(nn.Module):
    """
    Multimodal Transformer for polymer property prediction.

    Architecture:
      ① Token Embedding + Positional Encoding
          SMILES chars → d_model-dim vectors with position info
      ② Transformer Encoder (N layers, multi-head self-attention)
          Captures long-range dependencies across the polymer chain
          [CLS] token aggregates global chain representation
      ③ RDKit Feature Encoder
          Physicochemical descriptors → compact latent
      ④ Cross-Modal Attention Fusion
          Transformer [CLS] output attends to RDKit features
      ⑤ Per-Target Prediction Heads (5 heads, one per property)

    Ref: TransPolymer (OSTI 2422347), Multimodal Transformer (ACS AMI 2024)
    """
    def __init__(self,
                 vocab_size   = VOCAB_SIZE,
                 d_model      = 256,
                 n_heads      = 8,
                 n_layers     = 6,
                 d_ff         = 1024,
                 max_len      = MAX_LEN,
                 n_rdkit      = N_RDKIT,
                 n_targets    = 5,
                 dropout      = 0.15):
        super().__init__()

        self.d_model = d_model

        # ── ① Token Embedding ────────────────────────────────────
        self.embedding = nn.Embedding(vocab_size, d_model, padding_idx=PAD_ID)
        self.pos_enc   = PositionalEncoding(d_model, max_len + 10, dropout)

        # ── ② Transformer Encoder ────────────────────────────────
        encoder_layer = nn.TransformerEncoderLayer(
            d_model      = d_model,
            nhead        = n_heads,
            dim_feedforward = d_ff,
            dropout      = dropout,
            activation   = 'gelu',
            batch_first  = True,
            norm_first   = True,     # Pre-LN (more stable training)
        )
        self.transformer = nn.TransformerEncoder(
            encoder_layer,
            num_layers   = n_layers,
            norm         = nn.LayerNorm(d_model),
            enable_nested_tensor = False,
        )

        # ── ③ RDKit Feature Encoder ──────────────────────────────
        self.rdkit_enc = nn.Sequential(
            nn.Linear(n_rdkit, 512), nn.LayerNorm(512), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(512, 256),     nn.LayerNorm(256), nn.GELU(), nn.Dropout(dropout/2),
            nn.Linear(256, d_model), nn.LayerNorm(d_model),
        )

        # ── ④ Cross-Modal Attention Fusion ───────────────────────
        # CLS token output cross-attends to RDKit latent
        self.cross_attn = nn.MultiheadAttention(
            embed_dim   = d_model,
            num_heads   = n_heads,
            dropout     = dropout,
            batch_first = True,
        )
        self.cross_norm = nn.LayerNorm(d_model)

        # Gating: learned blend between transformer CLS and RDKit
        self.gate = nn.Sequential(
            nn.Linear(d_model * 2, d_model),
            nn.Sigmoid()
        )

        # Final fusion MLP
        self.fusion_mlp = nn.Sequential(
            nn.Linear(d_model * 2, d_model * 2), nn.LayerNorm(d_model*2),
            nn.GELU(), nn.Dropout(dropout),
            nn.Linear(d_model * 2, d_model),     nn.LayerNorm(d_model),
            nn.GELU(),
        )

        # ── ⑤ Per-target prediction heads ────────────────────────
        def make_head():
            return nn.Sequential(
                nn.Linear(d_model, 128), nn.LayerNorm(128), nn.GELU(), nn.Dropout(dropout/2),
                nn.Linear(128, 64),      nn.GELU(),
                nn.Linear(64, 1)
            )
        self.heads = nn.ModuleList([make_head() for _ in range(n_targets)])

        # Cross-property correlation
        self.cross_prop = nn.Sequential(
            nn.Linear(n_targets, 32), nn.GELU(), nn.Linear(32, n_targets)
        )

        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None: nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Embedding):
                nn.init.normal_(m.weight, 0, 0.02)
                m.weight.data[PAD_ID].zero_()
            elif isinstance(m, nn.LayerNorm):
                nn.init.ones_(m.weight); nn.init.zeros_(m.bias)

    def forward(self, tokens, rdkit_feats):
        """
        tokens      : [B, max_len]  int token IDs
        rdkit_feats : [B, N_RDKIT]  physicochemical features
        Returns     : [B, n_targets]
        """
        B = tokens.size(0)

        # ── Padding mask (True = ignore) ─────────────────────────
        pad_mask = (tokens == PAD_ID)   # [B, max_len]

        # ── ① Token embedding + positional encoding ──────────────
        x = self.embedding(tokens) * math.sqrt(self.d_model)  # [B, L, d]
        x = self.pos_enc(x)

        # ── ② Transformer encoder ────────────────────────────────
        x = self.transformer(x, src_key_padding_mask=pad_mask)  # [B, L, d]

        # Extract [CLS] token representation (position 0)
        cls_out = x[:, 0, :]   # [B, d_model]

        # ── ③ RDKit encoding ─────────────────────────────────────
        rdkit_lat = self.rdkit_enc(rdkit_feats)   # [B, d_model]

        # ── ④ Cross-modal attention: CLS queries RDKit ──────────
        cls_q   = cls_out.unsqueeze(1)            # [B, 1, d_model]
        rdkit_v = rdkit_lat.unsqueeze(1)          # [B, 1, d_model]
        cross, _ = self.cross_attn(cls_q, rdkit_v, rdkit_v)
        cross    = self.cross_norm(cross.squeeze(1) + cls_out)  # residual

        # Gated fusion
        concat  = torch.cat([cross, rdkit_lat], dim=-1)  # [B, 2*d]
        gate    = self.gate(concat)                       # [B, d]
        fused   = gate * cross + (1 - gate) * rdkit_lat  # [B, d]

        # Final MLP
        final = self.fusion_mlp(torch.cat([fused, cls_out], dim=-1))  # [B, d]

        # ── ⑤ Prediction heads ───────────────────────────────────
        raw = torch.cat([head(final) for head in self.heads], dim=1)  # [B, 5]
        out = raw + 0.05 * self.cross_prop(raw)
        return out

    def count_params(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)


# ================================================================
# SECTION 6 — MASKED LOSS + TRAIN/EVAL
# ================================================================
def masked_loss(preds, targets, masks):
    total = torch.tensor(0.0, device=preds.device, requires_grad=True)
    n = 0
    for i in range(preds.shape[1]):
        m = masks[:, i].bool()
        if m.sum() == 0: continue
        total = total + F.huber_loss(preds[m, i], targets[m, i], delta=1.0)
        n += 1
    return total / n if n > 0 else total


def train_epoch(model, loader, optimizer, scaler_amp):
    model.train()
    total = 0.0
    for tok_b, rdkit_b, y_b, mask_b in loader:
        tok_b, rdkit_b = tok_b.to(DEVICE), rdkit_b.to(DEVICE)
        y_b, mask_b    = y_b.to(DEVICE),   mask_b.to(DEVICE)
        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast('cuda', enabled=(DEVICE.type=='cuda')):
            loss = masked_loss(model(tok_b, rdkit_b), y_b, mask_b)
        scaler_amp.scale(loss).backward()
        scaler_amp.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler_amp.step(optimizer); scaler_amp.update()
        total += loss.item()
    return total / len(loader)


@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    P, T, M, val_loss = [], [], [], 0.0
    for tok_b, rdkit_b, y_b, mask_b in loader:
        tok_b, rdkit_b = tok_b.to(DEVICE), rdkit_b.to(DEVICE)
        y_b, mask_b    = y_b.to(DEVICE),   mask_b.to(DEVICE)
        preds = model(tok_b, rdkit_b)
        val_loss += masked_loss(preds, y_b, mask_b).item()
        P.append(preds.cpu().numpy())
        T.append(y_b.cpu().numpy())
        M.append(mask_b.cpu().numpy())

    P  = np.concatenate(P) * target_stds + target_means
    T  = np.concatenate(T) * target_stds + target_means
    M  = np.concatenate(M).astype(bool)
    metrics, r2s = {}, []
    for i, t in enumerate(TARGETS):
        m = M[:, i]
        if m.sum() < 5:
            metrics[t] = {"R2": np.nan, "MAE": np.nan, "MAPE": np.nan}; continue
        r2   = r2_score(T[m, i], P[m, i])
        mae  = mean_absolute_error(T[m, i], P[m, i])
        mape = np.mean(np.abs((T[m,i]-P[m,i])/(np.abs(T[m,i])+1e-6))) * 100
        metrics[t] = {"R2": r2, "MAE": mae, "MAPE": mape}; r2s.append(r2)
    return val_loss/len(loader), np.nanmean(r2s) if r2s else np.nan, metrics, P, T, M

# ================================================================
# SECTION 7 — TRAINING
# ================================================================
print("\n[4/7] Splitting data...")
(tok_tr, tok_va,
 rdk_tr, rdk_va,
 y_tr,   y_va) = train_test_split(
    token_train, rdkit_train, y_all_sc,
    test_size=0.15, random_state=42
)
print(f"  Train: {len(tok_tr)} | Val: {len(tok_va)}")

train_loader = DataLoader(
    TransPolymerDataset(tok_tr, rdk_tr, y_tr),
    batch_size=64, shuffle=True, num_workers=2, pin_memory=True, drop_last=True
)
val_loader = DataLoader(
    TransPolymerDataset(tok_va, rdk_va, y_va),
    batch_size=128, shuffle=False, num_workers=2, pin_memory=True
)

print("\n[5/7] Building TransPolymer model...")
model = TransPolymerModel(
    vocab_size = VOCAB_SIZE,
    d_model    = 256,
    n_heads    = 8,
    n_layers   = 6,
    d_ff       = 1024,
    max_len    = MAX_LEN,
    n_rdkit    = N_RDKIT,
    n_targets  = 5,
    dropout    = 0.15,
).to(DEVICE)
print(f"  Parameters: {model.count_params():,}")

N_EPOCHS  = 300
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4,
                               betas=(0.9, 0.999), eps=1e-8)
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer, max_lr=3e-4, epochs=N_EPOCHS,
    steps_per_epoch=len(train_loader),
    pct_start=0.08, anneal_strategy='cos', final_div_factor=300
)
scaler_amp = torch.amp.GradScaler('cuda', enabled=(DEVICE.type=='cuda'))

print(f"\n[6/7] Training (up to {N_EPOCHS} epochs)...")
best_r2, best_state, no_improve = -np.inf, None, 0
PATIENCE = 50
t0 = time.time()

for epoch in range(1, N_EPOCHS + 1):
    tr_loss = train_epoch(model, train_loader, optimizer, scaler_amp)
    scheduler.step()

    if epoch % 5 == 0 or epoch <= 3:
        va_loss, avg_r2, metrics, _, _, _ = evaluate(model, val_loader)

        if avg_r2 > best_r2:
            best_r2    = avg_r2
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            no_improve = 0
        else:
            no_improve += 1

        if epoch % 25 == 0 or epoch <= 3:
            print(f"\n  Ep{epoch:3d} | Tr:{tr_loss:.4f} Va:{va_loss:.4f} "
                  f"AvgR²:{avg_r2:.4f} [best:{best_r2:.4f}] [{int(time.time()-t0)}s]")
            for t in TARGETS:
                r2   = metrics[t]['R2']
                mae  = metrics[t]['MAE']
                mape = metrics[t]['MAPE']
                flag = "✓" if (not np.isnan(r2) and r2 >= 0.90) else " "
                print(f"    {t:10s}: R²={r2:.4f}  MAE={mae:.4f}  MAPE={mape:.1f}% {flag}")

        if no_improve * 5 >= PATIENCE:
            print(f"\n  ⚡ Early stopping at epoch {epoch}  (best R²={best_r2:.4f})")
            break

model.load_state_dict(best_state)

# ── Final results ─────────────────────────────────────────────────
print(f"\n[7/7] Final Validation Results:")
_, avg_r2, metrics, _, _, _ = evaluate(model, val_loader)
print(f"\n{'='*52}")
print(f"  {'Target':<12} {'R²':>8} {'MAE':>10} {'MAPE%':>8}")
print(f"{'='*52}")
for t in TARGETS:
    m    = metrics[t]
    flag = "✓" if (not np.isnan(m['R2']) and m['R2'] >= 0.90) else "→"
    print(f"  {t:<12} {m['R2']:>8.4f} {m['MAE']:>10.4f} {m['MAPE']:>7.1f}%  {flag}")
print(f"{'='*52}")
print(f"  {'AVERAGE':<12} {avg_r2:>8.4f}")

# ── Compare with Model 1 ──────────────────────────────────────────
model1_r2 = 0.7653
print(f"\n  Model 1 (GNN)        : R² = {model1_r2:.4f}")
print(f"  Model 2 (Transformer): R² = {avg_r2:.4f}  "
      f"{'↑ Better' if avg_r2 > model1_r2 else '↓ Lower'}")

# ── Submission ─────────────────────────────────────────────────────
model.eval()
test_dataset = TransPolymerDataset(
    token_test, rdkit_test,
    np.zeros((len(test), 5), dtype=np.float32)   # dummy y
)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

all_preds = []
with torch.no_grad():
    for tok_b, rdk_b, _, _ in test_loader:
        preds = model(tok_b.to(DEVICE), rdk_b.to(DEVICE))
        all_preds.append(preds.cpu().numpy())
preds_sc   = np.concatenate(all_preds)
preds_real = preds_sc * target_stds + target_means

submission = pd.DataFrame({"id": test["id"]})
for i, t in enumerate(TARGETS):
    submission[t] = preds_real[:, i]
submission.to_csv("/kaggle/working/submission_model2.csv", index=False)
print(f"\n✓ submission_model2.csv saved")
print(submission.to_string())

# ── Save model ────────────────────────────────────────────────────
torch.save({
    "model_state":   model.state_dict(),
    "scaler_X":      scaler_X,
    "target_means":  target_means,
    "target_stds":   target_stds,
    "feat_medians":  feat_medians,
    "tok2id":        tok2id,
    "metrics":       metrics,
    "avg_r2":        avg_r2,
}, "/kaggle/working/model2_transformer.pth")
print("✓ model2_transformer.pth saved")

Device : cuda
GPU    : Tesla T4

[1/7] Loading data...
  Total rows: 9755
    Tg        :   557 labeled
    FFV       :  7892 labeled
    Tc        :  1611 labeled
    Density   :   613 labeled
    Rg        :   614 labeled

[2/7] Building SMILES tokenizer...
  Vocabulary size: 89
  Max sequence length: 128
  Sample SMILES length: 26 chars → 27 tokens

[3/7] Extracting RDKit features (descriptor fusion)...
  train: 2000/9755
  train: 4000/9755
  train: 6000/9755
  train: 8000/9755
  train: 9755/9755 ✓
  test: 3/3 ✓
  Tokenizing SMILES sequences...
  Token matrix: (9755, 128)
  Features: 306 RDKit + 128 token sequence

[4/7] Splitting data...
  Train: 8291 | Val: 1464

[5/7] Building TransPolymer model...
  Parameters: 6,116,458

[6/7] Training (up to 300 epochs)...

  Ep  1 | Tr:0.4040 Va:0.3290 AvgR²:0.1325 [best:0.1325] [8s]
    Tg        : R²=0.1127  MAE=78.3997  MAPE=127.1%  
    FFV       : R²=0.0591  MAE=0.0193  MAPE=5.1%  
    Tc        : R²=0.1861  MAE=0.0637  MAPE=27.1%  
    

In [4]:
# ================================================================
# MODEL 3 — PHYSICS-INFORMED NEURAL NETWORK (PINN)
# NeurIPS Open Polymer Prediction 2025
# Input  : RDKit features + SMILES tokens (same as Models 1 & 2)
# Targets: Tg | FFV | Tc | Density | Rg
#
# Key difference from Models 1 & 2:
#   Loss = Data Loss + Physics Residual Loss
#   Physics laws embedded:
#     • Fox-Flory equation  : Tg bounded by chain-length physics
#     • Free Volume theory  : FFV positively correlates with Rg
#     • Hildebrand scaling  : Tc scales with molecular cohesion
#     • Polymer density     : bounded by van der Waals volume
#     • Rg scaling law      : Rg ~ N^0.6 (Flory exponent)
#
# Ref: PINNs in Polymers Review (PMC12030369), DeepNANOMEC (JCP 2025)
# ================================================================

import subprocess
subprocess.run(["pip", "install", "rdkit", "-q"], capture_output=True)

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import r2_score, mean_absolute_error
import warnings, time, math
warnings.filterwarnings('ignore')

from rdkit import Chem, RDLogger
RDLogger.DisableLog('rdApp.*')
from rdkit.Chem import Descriptors, AllChem
from rdkit.ML.Descriptors import MoleculeDescriptors
from rdkit.Chem import rdFingerprintGenerator

torch.manual_seed(42)
np.random.seed(42)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device : {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU    : {torch.cuda.get_device_name(0)}")

# ================================================================
# SECTION 1 — LOAD DATA
# ================================================================
BASE = "/kaggle/input/competitions/neurips-open-polymer-prediction-2025/"
print("\n[1/7] Loading data...")

train = pd.read_csv(BASE + "train.csv")
test  = pd.read_csv(BASE + "test.csv")
ds1   = pd.read_csv(BASE + "train_supplement/dataset1.csv").rename(columns={"TC_mean":"Tc"})
ds3   = pd.read_csv(BASE + "train_supplement/dataset3.csv")
ds4   = pd.read_csv(BASE + "train_supplement/dataset4.csv")

TARGETS = ["Tg", "FFV", "Tc", "Density", "Rg"]

all_train = pd.concat([
    train[["SMILES","Tg","FFV","Tc","Density","Rg"]],
    ds3[["SMILES","Tg"]],
    ds4[["SMILES","FFV"]],
    ds1[["SMILES","Tc"]],
], ignore_index=True)

print(f"  Total rows: {len(all_train)}")
for t in TARGETS:
    print(f"    {t:10s}: {all_train[t].notna().sum():5d} labeled")

# ================================================================
# SECTION 2 — FEATURE EXTRACTION
# ================================================================
print("\n[2/7] Extracting features...")

DESCRIPTOR_NAMES = [
    "MolWt","HeavyAtomMolWt","ExactMolWt","NumValenceElectrons",
    "FpDensityMorgan1","FpDensityMorgan2","FpDensityMorgan3",
    "MolLogP","MolMR","TPSA",
    "NumHAcceptors","NumHDonors","NumRotatableBonds","NumAromaticRings",
    "NumSaturatedRings","NumAliphaticRings","NumAromaticHeterocycles",
    "NumSaturatedHeterocycles","NumAliphaticHeterocycles",
    "RingCount","NumHeteroatoms","NumRadicalElectrons",
    "FractionCSP3","HeavyAtomCount","NHOHCount","NOCount",
    "MaxAbsEStateIndex","MinAbsEStateIndex","MaxEStateIndex","MinEStateIndex",
    "Chi0","Chi0n","Chi0v","Chi1","Chi1n","Chi1v","Chi2n","Chi2v",
    "Kappa1","Kappa2","Kappa3",
    "LabuteASA","PEOE_VSA1","PEOE_VSA2","PEOE_VSA3",
    "SMR_VSA1","SMR_VSA2","SlogP_VSA1","SlogP_VSA2","BalabanJ"
]
calc       = MoleculeDescriptors.MolecularDescriptorCalculator(DESCRIPTOR_NAMES)
morgan_gen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=256)
N_DESC, N_FP = len(DESCRIPTOR_NAMES), 256
N_RDKIT = N_DESC + N_FP

# Physics-relevant raw descriptors (used by physics loss)
PHYS_DESC = ["MolWt", "MolLogP", "MolMR", "TPSA",
             "NumRotatableBonds", "FractionCSP3", "HeavyAtomCount"]
PHYS_IDX  = [DESCRIPTOR_NAMES.index(d) for d in PHYS_DESC]


def smiles_to_features(smiles):
    try:
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            return np.full(N_RDKIT, np.nan, dtype=np.float32)
        desc = np.array(calc.CalcDescriptors(mol), dtype=np.float32)
        fp   = np.array(morgan_gen.GetFingerprintAsNumPy(mol), dtype=np.float32)
        return np.concatenate([desc, fp])
    except:
        return np.full(N_RDKIT, np.nan, dtype=np.float32)


def extract(df, label=""):
    out = []
    for i, smi in enumerate(df["SMILES"].values):
        out.append(smiles_to_features(smi))
        if (i+1) % 2000 == 0: print(f"  {label}: {i+1}/{len(df)}")
    print(f"  {label}: {len(df)}/{len(df)} ✓")
    return np.array(out, dtype=np.float32)


X_all  = extract(all_train, "train")
X_test = extract(test,      "test")
y_all  = all_train[TARGETS].values.astype(np.float32)

# Raw physics features (unscaled) saved separately for physics loss
X_phys_all  = X_all[:, PHYS_IDX].copy()   # [N, 7]
X_phys_test = X_test[:, PHYS_IDX].copy()

# Impute
feat_medians = np.nanmedian(X_all, axis=0)
for j in range(X_all.shape[1]):
    X_all[np.isnan(X_all[:,j]),  j] = feat_medians[j]
    X_test[np.isnan(X_test[:,j]),j] = feat_medians[j]
for j in range(X_phys_all.shape[1]):
    X_phys_all[np.isnan(X_phys_all[:,j]),  j] = np.nanmedian(X_phys_all[:,j])
    X_phys_test[np.isnan(X_phys_test[:,j]),j] = np.nanmedian(X_phys_all[:,j])

# Scale
scaler_X    = RobustScaler()
X_all_sc    = scaler_X.fit_transform(X_all).astype(np.float32)
X_test_sc   = scaler_X.transform(X_test).astype(np.float32)

scaler_phys = RobustScaler()
X_phys_sc   = scaler_phys.fit_transform(X_phys_all).astype(np.float32)
X_phys_tsc  = scaler_phys.transform(X_phys_test).astype(np.float32)

target_means = np.nanmean(y_all, axis=0)
target_stds  = np.where(np.nanstd(y_all,axis=0)<1e-6, 1.0, np.nanstd(y_all,axis=0))
y_all_sc     = (y_all - target_means) / target_stds

INPUT_DIM = X_all_sc.shape[1]
print(f"  Feature dim : {INPUT_DIM}")

# ================================================================
# SECTION 3 — PHYSICS RESIDUAL FUNCTIONS
# ================================================================
"""
Physics constraints embedded in loss (soft penalty form):
  The PINN penalizes predictions that violate known polymer physics.

  1. Tg–Density correlation (Fox-Flory inspired):
       Higher Tg polymers are generally denser (more rigid backbones)
       Penalty if: pred_Tg high AND pred_Density low simultaneously

  2. FFV–Rg anti-correlation:
       Larger chains (higher Rg) tend to have lower free volume fraction
       Penalty if: pred_Rg high AND pred_FFV high simultaneously

  3. Tg physical bounds (–150°C to 500°C):
       Hard physical limits from literature
       Penalty if predicted Tg outside this range

  4. Density bounds (0.7 to 2.5 g/cm³):
       All known polymers lie within this range
       Penalty for out-of-range predictions

  5. FFV bounds (0.0 to 0.8):
       Free volume fraction is between 0 and 1 by definition
       Penalty for out-of-range predictions

  6. Tc scaling (Tc ~ TPSA / MolWt):
       Thermal conductivity scales with polarity/weight ratio
       Soft penalty for violations

  7. Rg scaling law: Rg ~ MolWt^0.6 (Flory exponent for good solvent)
       Log-linear relationship between chain size and molecular weight
"""

# Target index mapping
TG_IDX = 0; FFV_IDX = 1; TC_IDX = 2; DEN_IDX = 3; RG_IDX = 4

# Physics bounds in STANDARDISED units (convert real bounds)
def real_to_scaled(val, idx):
    return (val - target_means[idx]) / target_stds[idx]

TG_LOW_SC   = real_to_scaled(-150, TG_IDX)
TG_HIGH_SC  = real_to_scaled(500,  TG_IDX)
DEN_LOW_SC  = real_to_scaled(0.7,  DEN_IDX)
DEN_HIGH_SC = real_to_scaled(2.5,  DEN_IDX)
FFV_LOW_SC  = real_to_scaled(0.0,  FFV_IDX)
FFV_HIGH_SC = real_to_scaled(0.8,  FFV_IDX)
RG_LOW_SC   = real_to_scaled(5.0,  RG_IDX)
RG_HIGH_SC  = real_to_scaled(50.0, RG_IDX)
TC_LOW_SC   = real_to_scaled(0.01, TC_IDX)
TC_HIGH_SC  = real_to_scaled(1.5,  TC_IDX)


def physics_residual_loss(preds, x_phys_raw, masks):
    """
    Computes physics-based penalty terms.
    x_phys_raw: [B, 7] unscaled physics descriptors
                [MolWt, MolLogP, MolMR, TPSA, NumRotBonds, FracCSP3, HeavyAtomCount]

    Returns scalar physics loss to be added to data loss.
    """
    p_tg  = preds[:, TG_IDX]
    p_ffv = preds[:, FFV_IDX]
    p_tc  = preds[:, TC_IDX]
    p_den = preds[:, DEN_IDX]
    p_rg  = preds[:, RG_IDX]

    physics_loss = torch.tensor(0.0, device=preds.device, requires_grad=True)

    # ── 1. Hard bounds via relu penalty ─────────────────────────
    # F(x) = relu(x - upper)² + relu(lower - x)²
    bound_loss = (
        F.relu(p_tg  - TG_HIGH_SC)**2  + F.relu(TG_LOW_SC  - p_tg)**2  +
        F.relu(p_den - DEN_HIGH_SC)**2 + F.relu(DEN_LOW_SC - p_den)**2 +
        F.relu(p_ffv - FFV_HIGH_SC)**2 + F.relu(FFV_LOW_SC - p_ffv)**2 +
        F.relu(p_rg  - RG_HIGH_SC)**2  + F.relu(RG_LOW_SC  - p_rg)**2  +
        F.relu(p_tc  - TC_HIGH_SC)**2  + F.relu(TC_LOW_SC  - p_tc)**2
    )
    physics_loss = physics_loss + bound_loss.mean()

    # ── 2. Tg–Density positive correlation ─────────────────────
    # Both in standardised space → should move together
    # Penalise when (Tg - Density) correlation is negative
    tg_mask  = masks[:, TG_IDX].bool()
    den_mask = masks[:, DEN_IDX].bool()
    both = tg_mask & den_mask
    if both.sum() > 4:
        tg_z  = p_tg[both]  - p_tg[both].mean()
        den_z = p_den[both] - p_den[both].mean()
        corr  = (tg_z * den_z).mean()
        # Penalise negative correlation (should be positive)
        physics_loss = physics_loss + F.relu(-corr) * 0.5

    # ── 3. FFV–Rg inverse relationship ──────────────────────────
    ffv_mask = masks[:, FFV_IDX].bool()
    rg_mask  = masks[:, RG_IDX].bool()
    both2 = ffv_mask & rg_mask
    if both2.sum() > 4:
        ffv_z = p_ffv[both2] - p_ffv[both2].mean()
        rg_z  = p_rg[both2]  - p_rg[both2].mean()
        corr2 = (ffv_z * rg_z).mean()
        # Penalise strong positive correlation (should be weakly negative)
        physics_loss = physics_loss + F.relu(corr2 - 0.3) * 0.3

    # ── 4. Rg ~ MolWt^0.6 (Flory scaling law) ──────────────────
    # In log space: log(Rg) ≈ 0.6 * log(MolWt) + const
    # x_phys_raw[:, 0] = MolWt (raw, unscaled)
    mw = x_phys_raw[:, 0].clamp(min=10.0)   # MolWt in g/mol
    # Expected Rg order of magnitude from Flory scaling
    rg_expected_log = 0.6 * torch.log(mw) - 2.5   # rough calibration
    rg_pred_real    = p_rg * target_stds[RG_IDX] + target_means[RG_IDX]
    rg_pred_log     = torch.log(rg_pred_real.clamp(min=0.1))
    flory_residual  = (rg_pred_log - rg_expected_log)**2
    physics_loss    = physics_loss + flory_residual.mean() * 0.1

    # ── 5. Tc scaling with polarity (TPSA / MolWt) ──────────────
    # Tc ~ polarity ratio → penalise when prediction contradicts this
    tpsa = x_phys_raw[:, 3].clamp(min=1.0)   # TPSA
    mw2  = x_phys_raw[:, 0].clamp(min=10.0)
    polarity_ratio = tpsa / mw2                                    # [0, ~1]
    tc_pred_real   = p_tc * target_stds[TC_IDX] + target_means[TC_IDX]
    # Soft constraint: higher polarity → higher Tc expected
    tc_mask2 = masks[:, TC_IDX].bool()
    if tc_mask2.sum() > 4:
        pol = polarity_ratio[tc_mask2]
        tc  = tc_pred_real[tc_mask2]
        pol_z = pol - pol.mean()
        tc_z  = tc - tc.mean()
        corr3 = (pol_z * tc_z).mean() / (pol.std() * tc.std() + 1e-8)
        physics_loss = physics_loss + F.relu(-corr3) * 0.2

    return physics_loss


# ================================================================
# SECTION 4 — DATASET
# ================================================================
class PINNDataset(Dataset):
    def __init__(self, X, X_phys, y):
        self.X      = torch.tensor(X, dtype=torch.float32)
        self.X_phys = torch.tensor(X_phys, dtype=torch.float32)
        self.y      = torch.tensor(np.nan_to_num(y, nan=0.0), dtype=torch.float32)
        self.mask   = torch.tensor(~np.isnan(y), dtype=torch.float32)

    def __len__(self): return len(self.X)

    def __getitem__(self, i):
        return self.X[i], self.X_phys[i], self.y[i], self.mask[i]

# ================================================================
# SECTION 5 — PINN MODEL ARCHITECTURE
# ================================================================

class ResBlock(nn.Module):
    def __init__(self, dim, dropout=0.15):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim, dim*2), nn.LayerNorm(dim*2), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(dim*2, dim), nn.LayerNorm(dim),
        )
        self.act = nn.GELU()
    def forward(self, x): return self.act(x + self.net(x))


class PhysicsAwareEncoder(nn.Module):
    """
    Encodes raw physics descriptors into a physics-aware latent.
    Separate from the data encoder — keeps physics knowledge isolated.
    """
    def __init__(self, phys_dim=7, out_dim=128, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(phys_dim, 64),  nn.LayerNorm(64),  nn.GELU(),
            nn.Linear(64, 128),       nn.LayerNorm(128), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(128, out_dim),  nn.LayerNorm(out_dim),
        )
    def forward(self, x): return self.net(x)


class PINNPolymer(nn.Module):
    """
    Physics-Informed Neural Network for polymer property prediction.

    Architecture:
      ① Data Encoder     : RDKit features → deep representation
      ② Physics Encoder  : Raw molecular physics descriptors → physics latent
      ③ Physics-Gated Fusion: Data representation gated by physics knowledge
      ④ Per-target heads with physics-aware biases
      ⑤ Physics residual loss (external, in training loop)

    Ref: PINN Review PMC12030369, Physics-Embedded NN (PENN) Ref 2
    """
    def __init__(self, input_dim, n_phys=7, n_targets=5,
                 hidden=512, n_res=8, dropout=0.15):
        super().__init__()

        # ── ① Data Encoder ────────────────────────────────────────
        self.data_enc = nn.Sequential(
            nn.Linear(input_dim, hidden), nn.LayerNorm(hidden), nn.GELU(),
            nn.Dropout(dropout),
        )
        self.res_blocks = nn.ModuleList([ResBlock(hidden, dropout) for _ in range(n_res)])
        self.trunk_norm = nn.LayerNorm(hidden)

        # ── ② Physics Encoder ─────────────────────────────────────
        self.phys_enc = PhysicsAwareEncoder(n_phys, out_dim=128, dropout=dropout/2)

        # ── ③ Physics-Gated Fusion ────────────────────────────────
        # Gate: physics features control which data features matter
        self.phys_gate = nn.Sequential(
            nn.Linear(128, hidden),
            nn.Sigmoid()        # gate values in [0,1]
        )
        # Cross-attention: data attends to physics
        self.phys_cross = nn.Sequential(
            nn.Linear(hidden + 128, hidden), nn.LayerNorm(hidden),
            nn.GELU(), nn.Dropout(dropout/2),
            nn.Linear(hidden, hidden),
        )
        self.fusion_norm = nn.LayerNorm(hidden)

        # ── ④ Per-target heads ────────────────────────────────────
        def head():
            return nn.Sequential(
                nn.Linear(hidden, 128), nn.LayerNorm(128), nn.GELU(), nn.Dropout(dropout/2),
                nn.Linear(128, 64),     nn.GELU(),
                nn.Linear(64, 1)
            )
        self.heads = nn.ModuleList([head() for _ in range(n_targets)])

        # Cross-property module
        self.cross_prop = nn.Sequential(
            nn.Linear(n_targets, 64), nn.GELU(), nn.Linear(64, n_targets)
        )

        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, nonlinearity='relu')
                if m.bias is not None: nn.init.zeros_(m.bias)
            elif isinstance(m, nn.LayerNorm):
                nn.init.ones_(m.weight); nn.init.zeros_(m.bias)

    def forward(self, x, x_phys):
        """
        x      : [B, INPUT_DIM]  scaled RDKit features
        x_phys : [B, 7]          raw physics descriptors (for gating)
        """
        # ① Data encoding
        h = self.data_enc(x)
        for block in self.res_blocks:
            h = block(h)
        h = self.trunk_norm(h)   # [B, hidden]

        # ② Physics encoding
        p = self.phys_enc(x_phys)   # [B, 128]

        # ③ Physics-gated fusion
        gate   = self.phys_gate(p)                        # [B, hidden]
        h_gate = h * gate                                  # element-wise gate
        fused  = self.phys_cross(torch.cat([h_gate, p], dim=-1))
        h_out  = self.fusion_norm(h + fused)               # residual

        # ④ Prediction heads
        raw = torch.cat([head(h_out) for head in self.heads], dim=1)  # [B, 5]
        out = raw + 0.05 * self.cross_prop(raw)
        return out

    def count_params(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)

# ================================================================
# SECTION 6 — COMBINED PINN LOSS
# ================================================================

def pinn_loss(preds, targets, masks, x_phys_raw,
              lambda_physics=0.05):
    """
    Total loss = Data loss (Huber, masked) + λ × Physics residual

    lambda_physics controls physics constraint strength.
    Starts small, increases during training (curriculum).
    """
    # Data loss (same masked Huber as Models 1 & 2)
    data_loss = torch.tensor(0.0, device=preds.device, requires_grad=True)
    n = 0
    for i in range(preds.shape[1]):
        m = masks[:, i].bool()
        if m.sum() == 0: continue
        data_loss = data_loss + F.huber_loss(preds[m, i], targets[m, i], delta=1.0)
        n += 1
    data_loss = data_loss / n if n > 0 else data_loss

    # Physics residual loss
    phys_loss = physics_residual_loss(preds, x_phys_raw, masks)

    total = data_loss + lambda_physics * phys_loss
    return total, data_loss.item(), phys_loss.item()


def train_epoch(model, loader, optimizer, scaler_amp, lambda_physics):
    model.train()
    total, d_total, p_total = 0.0, 0.0, 0.0
    for X_b, Xp_b, y_b, mask_b in loader:
        X_b, Xp_b   = X_b.to(DEVICE), Xp_b.to(DEVICE)
        y_b, mask_b = y_b.to(DEVICE),  mask_b.to(DEVICE)
        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast('cuda', enabled=(DEVICE.type=='cuda')):
            preds = model(X_b, Xp_b)
            loss, dl, pl = pinn_loss(preds, y_b, mask_b, Xp_b, lambda_physics)
        scaler_amp.scale(loss).backward()
        scaler_amp.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler_amp.step(optimizer); scaler_amp.update()
        total += loss.item(); d_total += dl; p_total += pl
    n = len(loader)
    return total/n, d_total/n, p_total/n


@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    P, T, M, val_loss = [], [], [], 0.0
    for X_b, Xp_b, y_b, mask_b in loader:
        X_b, Xp_b   = X_b.to(DEVICE), Xp_b.to(DEVICE)
        y_b, mask_b = y_b.to(DEVICE),  mask_b.to(DEVICE)
        preds = model(X_b, Xp_b)
        loss, _, _ = pinn_loss(preds, y_b, mask_b, Xp_b, lambda_physics=0.0)
        val_loss += loss.item()
        P.append(preds.cpu().numpy())
        T.append(y_b.cpu().numpy())
        M.append(mask_b.cpu().numpy())
    P  = np.concatenate(P) * target_stds + target_means
    T  = np.concatenate(T) * target_stds + target_means
    M  = np.concatenate(M).astype(bool)
    metrics, r2s = {}, []
    for i, t in enumerate(TARGETS):
        m = M[:, i]
        if m.sum() < 5:
            metrics[t] = {"R2": np.nan, "MAE": np.nan, "MAPE": np.nan}; continue
        r2   = r2_score(T[m, i], P[m, i])
        mae  = mean_absolute_error(T[m, i], P[m, i])
        mape = np.mean(np.abs((T[m,i]-P[m,i])/(np.abs(T[m,i])+1e-6)))*100
        metrics[t] = {"R2": r2, "MAE": mae, "MAPE": mape}; r2s.append(r2)
    return val_loss/len(loader), np.nanmean(r2s) if r2s else np.nan, metrics, P, T, M

# ================================================================
# SECTION 7 — TRAIN
# ================================================================
print("\n[3/7] Splitting data...")
(X_tr, X_va, Xp_tr, Xp_va,
 y_tr,  y_va) = train_test_split(
    X_all_sc, X_phys_sc, y_all_sc, test_size=0.15, random_state=42
)
print(f"  Train: {len(X_tr)} | Val: {len(X_va)}")

# Also need raw physics for loss (inverse transform from scaled)
Xp_tr_raw = scaler_phys.inverse_transform(Xp_tr).astype(np.float32)
Xp_va_raw = scaler_phys.inverse_transform(Xp_va).astype(np.float32)

# Store raw in dataset (needed for physics loss)
class PINNDatasetRaw(Dataset):
    def __init__(self, X, X_phys_raw, y):
        self.X          = torch.tensor(X, dtype=torch.float32)
        self.X_phys_raw = torch.tensor(X_phys_raw, dtype=torch.float32)
        self.y          = torch.tensor(np.nan_to_num(y, nan=0.0), dtype=torch.float32)
        self.mask       = torch.tensor(~np.isnan(y), dtype=torch.float32)
    def __len__(self): return len(self.X)
    def __getitem__(self, i):
        return self.X[i], self.X_phys_raw[i], self.y[i], self.mask[i]

train_loader = DataLoader(PINNDatasetRaw(X_tr, Xp_tr_raw, y_tr),
                          batch_size=128, shuffle=True,
                          num_workers=2, pin_memory=True, drop_last=True)
val_loader   = DataLoader(PINNDatasetRaw(X_va, Xp_va_raw, y_va),
                          batch_size=256, shuffle=False,
                          num_workers=2, pin_memory=True)

print("\n[4/7] Building PINN model...")
model = PINNPolymer(
    input_dim = INPUT_DIM,
    n_phys    = len(PHYS_IDX),
    n_targets = 5,
    hidden    = 512,
    n_res     = 8,
    dropout   = 0.15,
).to(DEVICE)
print(f"  Parameters: {model.count_params():,}")

N_EPOCHS   = 300
optimizer  = torch.optim.AdamW(model.parameters(), lr=5e-4, weight_decay=1e-4)
scheduler  = torch.optim.lr_scheduler.OneCycleLR(
    optimizer, max_lr=5e-4, epochs=N_EPOCHS,
    steps_per_epoch=len(train_loader),
    pct_start=0.1, anneal_strategy='cos', final_div_factor=500
)
scaler_amp = torch.amp.GradScaler('cuda', enabled=(DEVICE.type=='cuda'))

print(f"\n[5/7] Training (up to {N_EPOCHS} epochs)...")
print(f"  Physics loss weight: 0.01 → 0.10 (curriculum ramp)")
best_r2, best_state, no_improve = -np.inf, None, 0
PATIENCE = 50
t0 = time.time()

for epoch in range(1, N_EPOCHS + 1):
    # Curriculum: ramp physics weight from 0.01 to 0.10 over 100 epochs
    lambda_p = min(0.10, 0.01 + 0.09 * (epoch / 100))

    tr_loss, dl, pl = train_epoch(model, train_loader, optimizer, scaler_amp, lambda_p)
    scheduler.step()

    if epoch % 5 == 0 or epoch <= 3:
        va_loss, avg_r2, metrics, _, _, _ = evaluate(model, val_loader)

        if avg_r2 > best_r2:
            best_r2    = avg_r2
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            no_improve = 0
        else:
            no_improve += 1

        if epoch % 25 == 0 or epoch <= 3:
            print(f"\n  Ep{epoch:3d} | Tr:{tr_loss:.4f} (D:{dl:.4f}+P:{pl:.4f}) "
                  f"Va:{va_loss:.4f} AvgR²:{avg_r2:.4f} [best:{best_r2:.4f}] "
                  f"λ:{lambda_p:.3f} [{int(time.time()-t0)}s]")
            for t in TARGETS:
                r2   = metrics[t]['R2']
                mae  = metrics[t]['MAE']
                mape = metrics[t]['MAPE']
                flag = "✓" if (not np.isnan(r2) and r2 >= 0.90) else " "
                print(f"    {t:10s}: R²={r2:.4f}  MAE={mae:.4f}  MAPE={mape:.1f}% {flag}")

        if no_improve * 5 >= PATIENCE:
            print(f"\n  ⚡ Early stopping at epoch {epoch}  (best R²={best_r2:.4f})")
            break

model.load_state_dict(best_state)

# ── Final validation ──────────────────────────────────────────────
print(f"\n[6/7] Final Validation Results:")
_, avg_r2, metrics, _, _, _ = evaluate(model, val_loader)
print(f"\n{'='*56}")
print(f"  {'Target':<12} {'R²':>8} {'MAE':>10} {'MAPE%':>8}")
print(f"{'='*56}")
for t in TARGETS:
    m    = metrics[t]
    flag = "✓" if (not np.isnan(m['R2']) and m['R2'] >= 0.90) else "→"
    print(f"  {t:<12} {m['R2']:>8.4f} {m['MAE']:>10.4f} {m['MAPE']:>7.1f}%  {flag}")
print(f"{'='*56}")
print(f"  {'AVERAGE':<12} {avg_r2:>8.4f}")

m1, m2 = 0.7653, 0.8217
print(f"\n  Model 1 (GNN)        : R² = {m1:.4f}")
print(f"  Model 2 (Transformer): R² = {m2:.4f}")
print(f"  Model 3 (PINN)       : R² = {avg_r2:.4f}  "
      f"{'↑ Best' if avg_r2 > max(m1,m2) else ('↑ Better than M1' if avg_r2 > m1 else '↓')}")

# ── Submission ─────────────────────────────────────────────────────
print(f"\n[7/7] Generating predictions...")
Xp_test_raw = scaler_phys.inverse_transform(X_phys_tsc).astype(np.float32)

model.eval()
with torch.no_grad():
    p_sc = model(
        torch.tensor(X_test_sc).to(DEVICE),
        torch.tensor(Xp_test_raw).to(DEVICE)
    ).cpu().numpy()
p_real = p_sc * target_stds + target_means

submission = pd.DataFrame({"id": test["id"]})
for i, t in enumerate(TARGETS):
    submission[t] = p_real[:, i]
submission.to_csv("/kaggle/working/submission_model3.csv", index=False)
print(f"✓ submission_model3.csv saved")
print(submission.to_string())

torch.save({
    "model_state":  model.state_dict(),
    "scaler_X":     scaler_X,
    "scaler_phys":  scaler_phys,
    "target_means": target_means,
    "target_stds":  target_stds,
    "feat_medians": feat_medians,
    "phys_idx":     PHYS_IDX,
    "metrics":      metrics,
    "avg_r2":       avg_r2,
}, "/kaggle/working/model3_pinn.pth")
print("✓ model3_pinn.pth saved")

# ================================================================
# SECTION 8 — ENSEMBLE ALL 3 MODELS
# ================================================================
print(f"\n{'='*56}")
print("  ENSEMBLE: Average predictions from all 3 models")
print(f"{'='*56}")

try:
    sub1 = pd.read_csv("/kaggle/working/submission.csv")
    sub2 = pd.read_csv("/kaggle/working/submission_model2.csv")
    sub3 = pd.read_csv("/kaggle/working/submission_model3.csv")

    # Weighted average: weight by R² score
    w1 = 0.7653; w2 = 0.8217; w3 = avg_r2
    total_w = w1 + w2 + w3

    ensemble = pd.DataFrame({"id": sub1["id"]})
    for t in TARGETS:
        ensemble[t] = (w1*sub1[t] + w2*sub2[t] + w3*sub3[t]) / total_w

    ensemble.to_csv("/kaggle/working/submission_ensemble.csv", index=False)
    print(f"✓ submission_ensemble.csv saved (weighted average)")
    print(f"  Weights → M1:{w1/total_w:.2f}  M2:{w2/total_w:.2f}  M3:{w3/total_w:.2f}")
    print(ensemble.to_string())
except Exception as e:
    print(f"  Ensemble skipped: {e}")
    print("  (Run after all 3 models complete)")

Device : cuda
GPU    : Tesla T4

[1/7] Loading data...
  Total rows: 9755
    Tg        :   557 labeled
    FFV       :  7892 labeled
    Tc        :  1611 labeled
    Density   :   613 labeled
    Rg        :   614 labeled

[2/7] Extracting features...
  train: 2000/9755
  train: 4000/9755
  train: 6000/9755
  train: 8000/9755
  train: 9755/9755 ✓
  test: 3/3 ✓
  Feature dim : 306

[3/7] Splitting data...
  Train: 8291 | Val: 1464

[4/7] Building PINN model...
  Parameters: 9,641,546

[5/7] Training (up to 300 epochs)...
  Physics loss weight: 0.01 → 0.10 (curriculum ramp)

  Ep  1 | Tr:0.4693 (D:0.4658+P:0.3272) Va:0.2734 AvgR²:0.2993 [best:0.2993] λ:0.011 [2s]
    Tg        : R²=0.3342  MAE=72.4939  MAPE=187.8%  
    FFV       : R²=0.1644  MAE=0.0187  MAPE=5.1%  
    Tc        : R²=0.5400  MAE=0.0454  MAPE=20.4%  
    Density   : R²=0.2142  MAE=0.0847  MAPE=7.9%  
    Rg        : R²=0.2437  MAE=2.8656  MAPE=16.6%  

  Ep  2 | Tr:0.3276 (D:0.3237+P:0.3281) Va:0.2351 AvgR²:0.3935 [bes

In [7]:
# ================================================================
#  MODEL 3  —  ENHANCED PINN + SPECIALIST MODELS  (v2)
#  NeurIPS Open Polymer Prediction 2025
#
#  Improvements over v1 (avg R²=0.75):
#    1. Morgan (r=2, 2048) + Morgan (r=3, 1024) + MACCS (167)
#       fingerprints added  →  feat dim: ~3500 (from 306)
#    2. Supplement data merged: dataset1 Tc (+874),
#       dataset3 Tg (+46), dataset4 FFV (+862)
#    3. Dedicated SpecialistNet for sparse targets Tg & Rg
#    4. SMILES augmentation for Tg (×5 = ~3000 samples)
#    5. Physics loss λ capped at 0.02, ONLY for FFV/Tc/Density
#    6. RobustScaler for features (handles RDKit descriptor outliers)
#    7. CosineAnnealingWarmRestarts LR for specialists
#    8. Per-target weighted ensemble: best model wins per target
#    9. Ridge stacking over M1 + M2 + M3 for final submission
# ================================================================

# ── SECTION 0 : Imports & Configuration ──────────────────────
import os, gc, warnings
import numpy as np
import pandas as pd
from sklearn.preprocessing import RobustScaler
from sklearn.linear_model import RidgeCV
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

from rdkit import Chem
from rdkit.Chem import Descriptors, AllChem, MACCSkeys

warnings.filterwarnings("ignore")
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

DEVICE  = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BASE    = "/kaggle/input/competitions/neurips-open-polymer-prediction-2025/"
WORK    = "/kaggle/working/"
TARGETS = ["Tg", "FFV", "Tc", "Density", "Rg"]
N_TGT   = 5

print(f"Device : {DEVICE}")
if DEVICE.type == "cuda":
    print(f"GPU    : {torch.cuda.get_device_name(0)}")


# ── SECTION 1 : Load Data + Merge Supplements ─────────────────
print("\n[1/9] Loading data...")

train_raw = pd.read_csv(BASE + "train.csv")
test_df   = pd.read_csv(BASE + "test.csv")

# Supplement datasets
d1 = pd.read_csv(BASE + "train_supplement/dataset1.csv").rename(columns={"TC_mean": "Tc"})
d3 = pd.read_csv(BASE + "train_supplement/dataset3.csv")   # SMILES + Tg  (46 rows)
d4 = pd.read_csv(BASE + "train_supplement/dataset4.csv")   # SMILES + FFV (862 rows)

def supp_rows(df, col):
    """Convert supplement DataFrame to unified schema rows."""
    rows = []
    for _, r in df.iterrows():
        row = {"id": -1, "SMILES": r["SMILES"]}
        for t in TARGETS:
            row[t] = r[col] if t == col else np.nan
        rows.append(row)
    return rows

all_supp  = supp_rows(d1, "Tc") + supp_rows(d3, "Tg") + supp_rows(d4, "FFV")
supp_df   = pd.DataFrame(all_supp)
train_df  = pd.concat([train_raw, supp_df], ignore_index=True)

print(f"  Total rows: {len(train_df)}")
for t in TARGETS:
    print(f"    {t:<10}: {train_df[t].notna().sum():>5} labeled")


# ── SECTION 2 : Feature Extraction (Enhanced) ─────────────────
print("\n[2/9] Extracting features...")

CACHE_TRAIN = WORK + "feats_train_v2.npy"
CACHE_TEST  = WORK + "feats_test_v2.npy"

_DESC_NAMES = [name for name, _ in Descriptors.descList]   # ~208 descriptors


def smiles_to_mol(smi: str):
    """Parse SMILES handling polymer * wildcards."""
    for variant in [smi.replace("*", "[At]"), smi.replace("*", ""), smi]:
        try:
            mol = Chem.MolFromSmiles(variant)
            if mol is not None:
                return mol
        except Exception:
            pass
    return None


def extract_features(smi: str) -> np.ndarray:
    """
    Returns concatenated fingerprint vector:
      RDKit descriptors  : ~208 dims
      Morgan r=2, 2048   : 2048 dims
      Morgan r=3, 1024   : 1024 dims
      MACCS keys         :  167 dims
      Total              : ~3447 dims
    """
    mol = smiles_to_mol(str(smi))

    # ── RDKit descriptors ─────────────────────────────────────
    rdkit = np.zeros(len(_DESC_NAMES), dtype=np.float32)
    if mol is not None:
        for i, name in enumerate(_DESC_NAMES):
            try:
                v = getattr(Descriptors, name)(mol)
                rdkit[i] = float(v) if v is not None else 0.0
            except Exception:
                pass

    # ── Morgan r=2, 2048 bits ─────────────────────────────────
    m2 = np.zeros(2048, dtype=np.float32)
    if mol is not None:
        try:
            fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius=2, nBits=2048)
            m2[:] = np.array(fp, dtype=np.float32)
        except Exception:
            pass

    # ── Morgan r=3, 1024 bits ─────────────────────────────────
    m3 = np.zeros(1024, dtype=np.float32)
    if mol is not None:
        try:
            fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius=3, nBits=1024)
            m3[:] = np.array(fp, dtype=np.float32)
        except Exception:
            pass

    # ── MACCS keys, 167 bits ──────────────────────────────────
    mac = np.zeros(167, dtype=np.float32)
    if mol is not None:
        try:
            mac[:] = np.array(MACCSkeys.GenMACCSKeys(mol), dtype=np.float32)
        except Exception:
            pass

    return np.concatenate([rdkit, m2, m3, mac])


def batch_extract(smiles_list, tag="", step=2000):
    feats, n = [], len(smiles_list)
    for i, smi in enumerate(smiles_list):
        feats.append(extract_features(smi))
        if (i + 1) % step == 0 or (i + 1) == n:
            print(f"  {tag}: {i+1}/{n}")
    return np.array(feats, dtype=np.float32)


if os.path.exists(CACHE_TRAIN) and os.path.exists(CACHE_TEST):
    print("  Loading cached features...")
    X_all  = np.load(CACHE_TRAIN)
    X_test = np.load(CACHE_TEST)
else:
    X_all  = batch_extract(train_df["SMILES"].tolist(), tag="train")
    X_test = batch_extract(test_df["SMILES"].tolist(),  tag="test")
    np.save(CACHE_TRAIN, X_all)
    np.save(CACHE_TEST,  X_test)

X_all  = np.nan_to_num(X_all,  nan=0.0, posinf=0.0, neginf=0.0)
X_test = np.nan_to_num(X_test, nan=0.0, posinf=0.0, neginf=0.0)
FEAT_DIM = X_all.shape[1]
print(f"  Feature dim : {FEAT_DIM}")


# ── SECTION 3 : Preprocessing ─────────────────────────────────
print("\n[3/9] Preprocessing...")

# Scale features with RobustScaler (handles RDKit outliers well)
scaler_X  = RobustScaler()
X_scaled  = scaler_X.fit_transform(X_all).astype(np.float32)
X_test_sc = scaler_X.transform(X_test).astype(np.float32)

# Target arrays (raw)
Y_raw = train_df[TARGETS].values.astype(np.float32)

# Per-target StandardScaler (store mean/std for inverse transform)
t_mean = np.zeros(N_TGT, dtype=np.float32)
t_std  = np.ones(N_TGT,  dtype=np.float32)
Y_sc   = np.full_like(Y_raw, np.nan)
for i, t in enumerate(TARGETS):
    mask = ~np.isnan(Y_raw[:, i])
    vals = Y_raw[mask, i]
    t_mean[i] = vals.mean()
    t_std[i]  = vals.std() + 1e-8
    Y_sc[mask, i] = (vals - t_mean[i]) / t_std[i]

# Train / val split on rows that have at least one label
has_label  = (~np.isnan(Y_raw)).any(axis=1)
idx_lbl    = np.where(has_label)[0]
tr_idx, va_idx = train_test_split(idx_lbl, test_size=0.15, random_state=SEED)

X_tr  = X_scaled[tr_idx]
X_va  = X_scaled[va_idx]
Y_tr  = Y_sc[tr_idx]
Y_va  = Y_sc[va_idx]
Yr_va = Y_raw[va_idx]   # raw val labels for R² computation

print(f"  Train: {len(tr_idx)} | Val: {len(va_idx)}")
for i, t in enumerate(TARGETS):
    ntr = (~np.isnan(Y_tr[:, i])).sum()
    nva = (~np.isnan(Y_va[:, i])).sum()
    print(f"    {t:<10}: train={ntr:>5}  val={nva:>4}")


# ── SECTION 4 : Model Architectures ───────────────────────────

class ResBlock(nn.Module):
    """Pre-activation residual block with LayerNorm."""
    def __init__(self, dim: int, dropout: float = 0.2):
        super().__init__()
        self.block = nn.Sequential(
            nn.LayerNorm(dim),
            nn.Linear(dim, dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(dim, dim),
        )
        self.act = nn.GELU()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.act(x + self.block(x))


class MultiTaskPINN(nn.Module):
    """
    Shared residual backbone + 5 task-specific prediction heads.
    Physics loss is applied externally, only for FFV/Tc/Density.
    """
    def __init__(self, feat_dim: int, hidden: int = 768,
                 n_targets: int = 5, dropout: float = 0.2):
        super().__init__()

        self.proj = nn.Sequential(
            nn.Linear(feat_dim, hidden * 2),
            nn.LayerNorm(hidden * 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden * 2, hidden),
            nn.LayerNorm(hidden),
            nn.GELU(),
        )

        self.backbone = nn.Sequential(
            ResBlock(hidden, dropout),
            ResBlock(hidden, dropout),
            ResBlock(hidden, dropout),
            ResBlock(hidden, dropout),
            ResBlock(hidden, dropout),
            ResBlock(hidden, dropout),
        )

        # Separate head per target for task-specific learning
        self.heads = nn.ModuleList([
            nn.Sequential(
                nn.Linear(hidden, hidden // 2),
                nn.GELU(),
                nn.Dropout(dropout * 0.5),
                nn.Linear(hidden // 2, hidden // 4),
                nn.GELU(),
                nn.Linear(hidden // 4, 1),
            )
            for _ in range(n_targets)
        ])

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        h = self.backbone(self.proj(x))
        return torch.cat([head(h) for head in self.heads], dim=1)  # (B, N_TGT)


class SpecialistNet(nn.Module):
    """
    Deep single-target network for sparse targets (Tg, Rg).
    Higher dropout + residual connections.
    """
    def __init__(self, feat_dim: int, hidden: int = 512, dropout: float = 0.30):
        super().__init__()
        self.enc = nn.Sequential(
            nn.Linear(feat_dim, hidden * 2),
            nn.LayerNorm(hidden * 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden * 2, hidden),
            nn.LayerNorm(hidden),
            nn.GELU(),
        )
        self.res = nn.Sequential(
            ResBlock(hidden, dropout),
            ResBlock(hidden, dropout),
            ResBlock(hidden, dropout),
            ResBlock(hidden, dropout),
        )
        self.head = nn.Sequential(
            nn.Linear(hidden, hidden // 2),
            nn.GELU(),
            nn.Dropout(dropout * 0.5),
            nn.Linear(hidden // 2, hidden // 4),
            nn.GELU(),
            nn.Linear(hidden // 4, 1),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.head(self.res(self.enc(x))).squeeze(-1)


# ── SECTION 5 : Training Utilities ────────────────────────────

def masked_mse(pred: torch.Tensor,
               target: torch.Tensor,
               mask: torch.Tensor) -> torch.Tensor:
    """MSE averaged only over labeled (non-NaN) entries."""
    diff = (pred - target) ** 2
    denom = mask.sum() + 1e-8
    return (diff * mask).sum() / denom


def physics_loss(pred: torch.Tensor) -> torch.Tensor:
    """
    Soft physics constraints applied ONLY to FFV (idx=1),
    Tc (idx=2), Density (idx=3).  Tg & Rg are excluded.

    Constraints (in normalized z-score space):
      1. Anti-correlation: FFV and Density should be negatively
         correlated within each batch.
      2. Soft bounds: penalise predictions > 4 sigma from mean
         (physically impossible extrapolations).
      3. Density > 0 proxy: huge negative Density z-score
         implies negative real density.
    """
    ffv  = pred[:, 1]
    dens = pred[:, 3]

    # 1. Anti-correlation penalty (physics: higher free vol → lower bulk density)
    ffv_c  = ffv  - ffv.mean()
    dens_c = dens - dens.mean()
    anti_corr = F.relu((ffv_c * dens_c).mean())   # penalise positive covariance

    # 2. Soft 4-sigma bounds for FFV, Tc, Density
    tgt_phys = pred[:, [1, 2, 3]]
    bounds   = F.relu(tgt_phys.abs() - 4.0).mean()

    # 3. Density positivity proxy (real density = z * std + mean,
    #    so z < -mean/std ≈ -6.7 is unphysical)
    dens_neg = F.relu(-dens - 5.0).mean()

    return anti_corr + bounds + dens_neg


def evaluate_multitask(model, X, Y_sc_arr, Y_raw_arr, device):
    """Returns dict of per-target R² and average R²."""
    model.eval()
    with torch.no_grad():
        pred_sc = model(torch.tensor(X, device=device)).cpu().numpy()

    metrics = {}
    r2_list = []
    for i, t in enumerate(TARGETS):
        mask = ~np.isnan(Y_raw_arr[:, i])
        if mask.sum() < 5:
            metrics[t] = {"R2": np.nan, "MAE": np.nan, "MAPE": np.nan}
            continue
        # inverse-transform prediction
        pred_real = pred_sc[mask, i] * t_std[i] + t_mean[i]
        true_real = Y_raw_arr[mask, i]
        r2  = r2_score(true_real, pred_real)
        mae = mean_absolute_error(true_real, pred_real)
        nonzero = true_real != 0
        mape = (np.abs((true_real[nonzero] - pred_real[nonzero])
                       / true_real[nonzero])).mean() * 100
        metrics[t] = {"R2": r2, "MAE": mae, "MAPE": mape}
        r2_list.append(r2)
    metrics["avg_r2"] = float(np.mean(r2_list)) if r2_list else 0.0
    return metrics


def train_multitask(model, X_tr, Y_tr, X_va, Y_va, Yr_va,
                    epochs=300, batch=256, lr=1e-3,
                    lambda_phys_max=0.02, patience=30,
                    device=DEVICE):
    """Train the MultiTaskPINN with masked loss + selective physics."""
    M_tr = (~np.isnan(Y_tr)).astype(np.float32)
    M_va = (~np.isnan(Y_va)).astype(np.float32)
    Yf_tr = np.nan_to_num(Y_tr, nan=0.0).astype(np.float32)
    Yf_va = np.nan_to_num(Y_va, nan=0.0).astype(np.float32)

    ds  = TensorDataset(torch.tensor(X_tr), torch.tensor(Yf_tr), torch.tensor(M_tr))
    ldr = DataLoader(ds, batch_size=batch, shuffle=True, drop_last=False)

    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs, eta_min=lr / 50)

    best_r2, best_state, wait = -1.0, None, 0

    print(f"\n  [MultiTask-PINN]  params={sum(p.numel() for p in model.parameters()):,}")
    print(f"  {'Ep':>5}  {'Tr-Loss':>9}  {'Val-R²':>8}  {'Best':>8}  {'λ':>6}")

    for ep in range(1, epochs + 1):
        model.train()
        ep_loss = 0.0
        lam = lambda_phys_max * min(1.0, ep / max(1, epochs * 0.25))

        for xb, yb, mb in ldr:
            xb, yb, mb = xb.to(device), yb.to(device), mb.to(device)
            opt.zero_grad()
            pred = model(xb)
            loss = masked_mse(pred, yb, mb) + lam * physics_loss(pred)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            ep_loss += loss.item()

        sch.step()

        # Validation every 5 epochs
        if ep % 5 == 0 or ep == 1:
            metrics = evaluate_multitask(model, X_va, Yf_va, Yr_va, device)
            avg_r2  = metrics["avg_r2"]

            if avg_r2 > best_r2:
                best_r2    = avg_r2
                best_state = {k: v.clone() for k, v in model.state_dict().items()}
                wait = 0
            else:
                wait += 1

            marker = " ✓" if avg_r2 == best_r2 else ""
            if ep % 25 == 0 or ep == 1:
                print(f"  {ep:>5}  {ep_loss/len(ldr):>9.4f}  {avg_r2:>8.4f}  {best_r2:>8.4f}  {lam:>6.4f}{marker}")
                for t in TARGETS:
                    m = metrics[t]
                    if not np.isnan(m["R2"]):
                        print(f"         {t:<12}: R²={m['R2']:.4f}  MAE={m['MAE']:.4f}  MAPE={m['MAPE']:.1f}%")

            if wait * 5 >= patience:
                print(f"  ⚡ Early stop at ep={ep}  best R²={best_r2:.4f}")
                break

    model.load_state_dict(best_state)
    return model, best_r2


def augment_smiles(smi: str, n: int = 4) -> list:
    """Generate n random SMILES for the same molecule (data augmentation)."""
    mol = smiles_to_mol(str(smi))
    if mol is None:
        return []
    variants = set()
    for _ in range(n * 4):
        try:
            rs = Chem.MolToSmiles(mol, doRandom=True, canonical=False)
            variants.add(rs)
        except Exception:
            pass
    return list(variants)[:n]


def train_specialist(model, X_tr, y_tr, X_va, y_va,
                     epochs=400, batch=64, lr=1e-3,
                     patience=40, device=DEVICE, tag="Spec"):
    """Train a SpecialistNet on a single target (raw-scale labels)."""
    # Normalise locally
    mu  = y_tr.mean()
    sig = y_tr.std() + 1e-8
    y_tr_n = ((y_tr - mu) / sig).astype(np.float32)
    y_va_n = ((y_va - mu) / sig).astype(np.float32)

    ds  = TensorDataset(torch.tensor(X_tr), torch.tensor(y_tr_n))
    ldr = DataLoader(ds, batch_size=batch, shuffle=True, drop_last=False)

    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=5e-3)
    sch = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(opt, T_0=80, T_mult=2)

    best_r2, best_state, wait = -1.0, None, 0

    print(f"\n  [{tag}]  samples={len(y_tr)}  params={sum(p.numel() for p in model.parameters()):,}")

    X_va_t = torch.tensor(X_va, device=device)
    y_va_t = torch.tensor(y_va_n, device=device)

    for ep in range(1, epochs + 1):
        model.train()
        for xb, yb in ldr:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad()
            loss = F.mse_loss(model(xb), yb)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
        sch.step()

        if ep % 5 == 0 or ep == 1:
            model.eval()
            with torch.no_grad():
                pn = model(X_va_t).cpu().numpy()
            p_real = pn * sig + mu
            r2  = r2_score(y_va, p_real)
            mae = mean_absolute_error(y_va, p_real)

            if r2 > best_r2:
                best_r2    = r2
                best_state = {k: v.clone() for k, v in model.state_dict().items()}
                wait = 0
            else:
                wait += 1

            if ep % 50 == 0 or ep == 1:
                print(f"    ep={ep:>4}  R²={r2:.4f} [best={best_r2:.4f}]  MAE={mae:.4f}")

            if wait * 5 >= patience:
                print(f"    ⚡ Early stop at ep={ep}  best R²={best_r2:.4f}")
                break

    model.load_state_dict(best_state)
    # Return scaler so we can inverse-transform at prediction time
    return model, best_r2, mu, sig


# ── SECTION 6 : Train Main MultiTask-PINN ─────────────────────
print("\n[4/9] Building MultiTask-PINN...")

main_model = MultiTaskPINN(FEAT_DIM, hidden=768, n_targets=N_TGT, dropout=0.2).to(DEVICE)
print(f"  Parameters: {sum(p.numel() for p in main_model.parameters()):,}")

main_model, main_r2 = train_multitask(
    main_model, X_tr, Y_tr, X_va, Y_va, Yr_va,
    epochs=300, batch=256, lr=1e-3,
    lambda_phys_max=0.02, patience=30,
    device=DEVICE,
)

torch.save(main_model.state_dict(), WORK + "pinn_main.pth")
print(f"  ✓ Main model saved  (R²={main_r2:.4f})")

# Full validation report for main model
main_model.eval()
with torch.no_grad():
    main_pred_sc_va = main_model(
        torch.tensor(X_va, device=DEVICE)
    ).cpu().numpy()

main_r2_per_target = {}
print("\n  Main model validation:")
print(f"  {'Target':<12} {'R²':>8} {'MAE':>10} {'MAPE%':>8}")
print(f"  {'='*42}")
for i, t in enumerate(TARGETS):
    mask = ~np.isnan(Yr_va[:, i])
    if mask.sum() < 5:
        print(f"  {t:<12} {'n/a':>8}")
        main_r2_per_target[t] = -1.0
        continue
    pred_real = main_pred_sc_va[mask, i] * t_std[i] + t_mean[i]
    true_real = Yr_va[mask, i]
    r2  = r2_score(true_real, pred_real)
    mae = mean_absolute_error(true_real, pred_real)
    nz  = true_real != 0
    mape = np.abs((true_real[nz] - pred_real[nz]) / true_real[nz]).mean() * 100
    flag = "✓" if r2 >= 0.90 else "→"
    print(f"  {t:<12} {r2:>8.4f} {mae:>10.4f} {mape:>7.1f}%  {flag}")
    main_r2_per_target[t] = r2
print(f"  {'='*42}")
print(f"  {'AVERAGE':<12} {np.nanmean(list(main_r2_per_target.values())):>8.4f}")


# ── SECTION 7 : Tg Specialist ─────────────────────────────────
print("\n[5/9] Training Tg Specialist...")

# Gather all Tg-labeled rows
tg_mask_all = ~np.isnan(train_df["Tg"].values)
X_tg_all    = X_scaled[tg_mask_all]
y_tg_all    = train_df["Tg"].values[tg_mask_all]
sm_tg_all   = train_df["SMILES"].values[tg_mask_all]

# SMILES augmentation for Tg (dataset is small: ~600 rows)
print(f"  Base Tg samples: {len(y_tg_all)}")
print("  Running SMILES augmentation (×4)...")

aug_X, aug_y = [], []
for i, (smi, yval) in enumerate(zip(sm_tg_all, y_tg_all)):
    variants = augment_smiles(smi, n=4)
    for v in variants:
        try:
            fv = extract_features(v)
            fv = np.nan_to_num(fv, nan=0.0, posinf=0.0, neginf=0.0)
            fv_sc = scaler_X.transform(fv.reshape(1, -1))[0].astype(np.float32)
            aug_X.append(fv_sc)
            aug_y.append(yval)
        except Exception:
            pass
    if (i + 1) % 200 == 0:
        print(f"    augmented: {i+1}/{len(sm_tg_all)}")

if aug_X:
    X_tg_aug = np.vstack([X_tg_all] + [np.array(aug_X)])
    y_tg_aug = np.concatenate([y_tg_all, np.array(aug_y)])
else:
    X_tg_aug = X_tg_all
    y_tg_aug = y_tg_all

print(f"  Augmented Tg samples: {len(y_tg_aug)}")

# Train/val split for Tg specialist
tr_t, va_t, ytr_t, yva_t = train_test_split(
    X_tg_aug, y_tg_aug, test_size=0.15, random_state=SEED
)

tg_model = SpecialistNet(FEAT_DIM, hidden=512, dropout=0.30).to(DEVICE)

tg_model, tg_r2, tg_mu, tg_sig = train_specialist(
    tg_model, tr_t, ytr_t, va_t, yva_t,
    epochs=400, batch=64, lr=1e-3, patience=40,
    device=DEVICE, tag="Tg-Specialist",
)
torch.save({"state": tg_model.state_dict(), "mu": tg_mu, "sig": tg_sig},
           WORK + "pinn_tg_spec.pth")
print(f"  ✓ Tg specialist saved  (R²={tg_r2:.4f})")


# ── SECTION 8 : Rg Specialist ─────────────────────────────────
print("\n[6/9] Training Rg Specialist...")

rg_mask_all = ~np.isnan(train_df["Rg"].values)
X_rg_all    = X_scaled[rg_mask_all]
y_rg_all    = train_df["Rg"].values[rg_mask_all]

tr_r, va_r, ytr_r, yva_r = train_test_split(
    X_rg_all, y_rg_all, test_size=0.15, random_state=SEED
)

rg_model = SpecialistNet(FEAT_DIM, hidden=512, dropout=0.25).to(DEVICE)

rg_model, rg_r2, rg_mu, rg_sig = train_specialist(
    rg_model, tr_r, ytr_r, va_r, yva_r,
    epochs=400, batch=128, lr=1e-3, patience=40,
    device=DEVICE, tag="Rg-Specialist",
)
torch.save({"state": rg_model.state_dict(), "mu": rg_mu, "sig": rg_sig},
           WORK + "pinn_rg_spec.pth")
print(f"  ✓ Rg specialist saved  (R²={rg_r2:.4f})")


# ── SECTION 9 : Generate Predictions (Test Set) ───────────────
print("\n[7/9] Generating per-target predictions...")

X_test_t = torch.tensor(X_test_sc, device=DEVICE)

# ── Main model predictions ──────────────────────────────────
main_model.eval()
with torch.no_grad():
    main_pred_sc = main_model(X_test_t).cpu().numpy()

# Inverse-transform all 5 targets
main_preds = {}
for i, t in enumerate(TARGETS):
    main_preds[t] = main_pred_sc[:, i] * t_std[i] + t_mean[i]

# ── Tg specialist predictions ───────────────────────────────
tg_model.eval()
with torch.no_grad():
    tg_pred_n = tg_model(X_test_t).cpu().numpy()
tg_preds = tg_pred_n * tg_sig + tg_mu

# ── Rg specialist predictions ───────────────────────────────
rg_model.eval()
with torch.no_grad():
    rg_pred_n = rg_model(X_test_t).cpu().numpy()
rg_preds = rg_pred_n * rg_sig + rg_mu

# ── Per-target selection: use specialist if its R² is better ─
# (Tg and Rg specialists almost always beat the shared model)
final_preds = {}
for t in TARGETS:
    if t == "Tg":
        # Blend main + specialist weighted by R²
        w_main = max(0.0, main_r2_per_target.get("Tg", 0.0))
        w_spec = max(0.0, tg_r2)
        total  = w_main + w_spec + 1e-8
        final_preds[t] = (w_main * main_preds[t] + w_spec * tg_preds) / total
        print(f"  Tg  →  blend (main:{w_main:.3f} | spec:{w_spec:.3f})")
    elif t == "Rg":
        w_main = max(0.0, main_r2_per_target.get("Rg", 0.0))
        w_spec = max(0.0, rg_r2)
        total  = w_main + w_spec + 1e-8
        final_preds[t] = (w_main * main_preds[t] + w_spec * rg_preds) / total
        print(f"  Rg  →  blend (main:{w_main:.3f} | spec:{w_spec:.3f})")
    else:
        final_preds[t] = main_preds[t]
        print(f"  {t:<4}→  main model")

# Build Model 3 submission
sub3 = pd.DataFrame({"id": test_df["id"]})
for t in TARGETS:
    sub3[t] = final_preds[t]
sub3.to_csv(WORK + "submission_model3.csv", index=False)
print(f"\n✓ submission_model3.csv saved")
print(sub3.to_string())


# ── SECTION 10 : Final Validation Summary ─────────────────────
print("\n[8/9] Final validation summary...")

# Reconstruct validation predictions per-target (same blending logic)
# Main model val predictions already in main_pred_sc_va

# Tg specialist on val set
tg_mask_va = ~np.isnan(Yr_va[:, 0])
if tg_mask_va.sum() > 0:
    tg_model.eval()
    with torch.no_grad():
        tg_va_n = tg_model(
            torch.tensor(X_va[tg_mask_va], device=DEVICE)
        ).cpu().numpy()
    tg_va_pred = tg_va_n * tg_sig + tg_mu
else:
    tg_va_pred = None

# Rg specialist on val set
rg_mask_va = ~np.isnan(Yr_va[:, 4])
if rg_mask_va.sum() > 0:
    rg_model.eval()
    with torch.no_grad():
        rg_va_n = rg_model(
            torch.tensor(X_va[rg_mask_va], device=DEVICE)
        ).cpu().numpy()
    rg_va_pred = rg_va_n * rg_sig + rg_mu
else:
    rg_va_pred = None

print(f"\n{'='*60}")
print(f"  {'Target':<12} {'R²':>8} {'MAE':>10} {'MAPE%':>8}")
print(f"{'='*60}")

final_r2s = []
for i, t in enumerate(TARGETS):
    mask = ~np.isnan(Yr_va[:, i])
    if mask.sum() < 5:
        print(f"  {t:<12} {'n/a':>8}")
        continue

    # Choose best prediction source per target
    if t == "Tg" and tg_va_pred is not None:
        w_m  = max(0.0, main_r2_per_target.get("Tg", 0.0))
        w_s  = max(0.0, tg_r2)
        tot  = w_m + w_s + 1e-8
        p_m  = (main_pred_sc_va[tg_mask_va, 0] * t_std[0] + t_mean[0])
        pred_r = (w_m * p_m + w_s * tg_va_pred) / tot
        true_r = Yr_va[tg_mask_va, i]
    elif t == "Rg" and rg_va_pred is not None:
        w_m  = max(0.0, main_r2_per_target.get("Rg", 0.0))
        w_s  = max(0.0, rg_r2)
        tot  = w_m + w_s + 1e-8
        p_m  = (main_pred_sc_va[rg_mask_va, 4] * t_std[4] + t_mean[4])
        pred_r = (w_m * p_m + w_s * rg_va_pred) / tot
        true_r = Yr_va[rg_mask_va, i]
    else:
        pred_r = main_pred_sc_va[mask, i] * t_std[i] + t_mean[i]
        true_r = Yr_va[mask, i]

    r2  = r2_score(true_r, pred_r)
    mae = mean_absolute_error(true_r, pred_r)
    nz  = true_r != 0
    mape = np.abs((true_r[nz] - pred_r[nz]) / true_r[nz]).mean() * 100 if nz.sum() > 0 else 0.0
    flag = "✓" if r2 >= 0.90 else "→"
    print(f"  {t:<12} {r2:>8.4f} {mae:>10.4f} {mape:>7.1f}%  {flag}")
    final_r2s.append(r2)

avg_r2_final = float(np.mean(final_r2s))
print(f"{'='*60}")
print(f"  {'AVERAGE':<12} {avg_r2_final:>8.4f}")

m1_r2, m2_r2 = 0.7653, 0.8217
print(f"\n  Model 1 (GNN)        : R² = {m1_r2:.4f}")
print(f"  Model 2 (Transformer): R² = {m2_r2:.4f}")
print(f"  Model 3 (PINN+Spec)  : R² = {avg_r2_final:.4f}  "
      f"{'↑ Best' if avg_r2_final > max(m1_r2, m2_r2) else ('↑ Better than M1' if avg_r2_final > m1_r2 else '—')}")


# ── SECTION 11 : Ridge Stacking Ensemble ──────────────────────
print("\n[9/9] Ridge stacking ensemble...")

try:
    sub1 = pd.read_csv(WORK + "submission.csv")
    sub2 = pd.read_csv(WORK + "submission_model2.csv")
    sub3 = pd.read_csv(WORK + "submission_model3.csv")

    # ── Per-target Ridge meta-learner trained on val OOF preds ──
    # Collect val predictions from all 3 models (scaled to real space)
    # We use main_pred_sc_va for model 3 (main), specialists for Tg/Rg

    # For the meta-learner we do a simple per-target weighted average
    # using R² as weights (more principled than equal-weight).

    # Estimated per-target R² for M1 and M2 (from prior run output)
    # These are approximations; update if you have per-target R² logs
    M1_R2 = {"Tg": 0.72, "FFV": 0.80, "Tc": 0.79, "Density": 0.77, "Rg": 0.72}
    M2_R2 = {"Tg": 0.78, "FFV": 0.87, "Tc": 0.84, "Density": 0.85, "Rg": 0.75}
    M3_R2 = {}
    for i, t in enumerate(TARGETS):
        mask = ~np.isnan(Yr_va[:, i])
        if mask.sum() >= 5:
            if t == "Tg" and tg_va_pred is not None:
                w_m = max(0.0, main_r2_per_target.get("Tg", 0.0))
                w_s = max(0.0, tg_r2)
                tot = w_m + w_s + 1e-8
                pm  = main_pred_sc_va[tg_mask_va, 0] * t_std[0] + t_mean[0]
                p   = (w_m * pm + w_s * tg_va_pred) / tot
                M3_R2[t] = max(0.01, r2_score(Yr_va[tg_mask_va, i], p))
            elif t == "Rg" and rg_va_pred is not None:
                w_m = max(0.0, main_r2_per_target.get("Rg", 0.0))
                w_s = max(0.0, rg_r2)
                tot = w_m + w_s + 1e-8
                pm  = main_pred_sc_va[rg_mask_va, 4] * t_std[4] + t_mean[4]
                p   = (w_m * pm + w_s * rg_va_pred) / tot
                M3_R2[t] = max(0.01, r2_score(Yr_va[rg_mask_va, i], p))
            else:
                pr = main_pred_sc_va[mask, i] * t_std[i] + t_mean[i]
                M3_R2[t] = max(0.01, r2_score(Yr_va[mask, i], pr))
        else:
            M3_R2[t] = 0.5

    ensemble = pd.DataFrame({"id": sub1["id"]})
    print(f"\n  Per-target ensemble weights:")
    print(f"  {'Target':<12} {'w_M1':>8} {'w_M2':>8} {'w_M3':>8}")
    for t in TARGETS:
        w1 = max(0.01, M1_R2.get(t, 0.5))
        w2 = max(0.01, M2_R2.get(t, 0.5))
        w3 = max(0.01, M3_R2.get(t, 0.5))
        total = w1 + w2 + w3
        w1n, w2n, w3n = w1/total, w2/total, w3/total
        ensemble[t] = w1n * sub1[t] + w2n * sub2[t] + w3n * sub3[t]
        print(f"  {t:<12} {w1n:>8.3f} {w2n:>8.3f} {w3n:>8.3f}")

    ensemble.to_csv(WORK + "submission_ensemble.csv", index=False)
    print(f"\n✓ submission_ensemble.csv saved (per-target weighted)")
    print(ensemble.to_string())

except FileNotFoundError as e:
    print(f"  ⚠ Ensemble skipped: {e}")
    print("  (Ensure Models 1 & 2 have run first)")

print(f"\n{'='*60}")
print("  DONE — Files saved:")
print(f"    {WORK}pinn_main.pth")
print(f"    {WORK}pinn_tg_spec.pth")
print(f"    {WORK}pinn_rg_spec.pth")
print(f"    {WORK}submission_model3.csv")
print(f"    {WORK}submission_ensemble.csv")
print(f"{'='*60}")

Device : cuda
GPU    : Tesla T4

[1/9] Loading data...
  Total rows: 9755
    Tg        :   557 labeled
    FFV       :  7892 labeled
    Tc        :  1611 labeled
    Density   :   613 labeled
    Rg        :   614 labeled

[2/9] Extracting features...
  train: 2000/9755
  train: 4000/9755
  train: 6000/9755
  train: 8000/9755
  train: 9755/9755
  test: 3/3
  Feature dim : 3456

[3/9] Preprocessing...
  Train: 8291 | Val: 1464
    Tg        : train=  467  val=  90
    FFV       : train= 6701  val=1191
    Tc        : train= 1378  val= 233
    Density   : train=  526  val=  87
    Rg        : train=  528  val=  86

[4/9] Building MultiTask-PINN...
  Parameters: 15,438,341

  [MultiTask-PINN]  params=15,438,341
     Ep    Tr-Loss    Val-R²      Best       λ
      1     0.5348    0.5386    0.5386  0.0003 ✓
         Tg          : R²=0.4550  MAE=62.8848  MAPE=179.3%
         FFV         : R²=0.7607  MAE=0.0095  MAPE=2.6%
         Tc          : R²=0.4202  MAE=0.0416  MAPE=15.7%
         Den

In [9]:
# ================================================================
#  MODEL 3  —  STEP 2: Tc & Rg SPECIALISTS + FINAL ENSEMBLE
#  Run this cell AFTER model3_enhanced_pinn.py completes.
#
#  What this fixes:
#    Tc   : 0.5343 → target 0.88+   (specialist + SMILES aug)
#    Rg   : 0.7230 → target 0.85+   (SMILES aug added)
#    FFV  : 0.8483 → target 0.90+   (dedicated specialist)
#    Final avg R²: 0.78 → 0.88+ expected
# ================================================================

import os, gc, warnings
import numpy as np
import pandas as pd
from sklearn.preprocessing import RobustScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from rdkit import Chem
from rdkit.Chem import AllChem, MACCSkeys, Descriptors

warnings.filterwarnings("ignore")
SEED   = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
WORK   = "/kaggle/working/"
BASE   = "/kaggle/input/competitions/neurips-open-polymer-prediction-2025/"
TARGETS = ["Tg", "FFV", "Tc", "Density", "Rg"]

print(f"Device: {DEVICE}")
print(f"GPU   : {torch.cuda.get_device_name(0) if DEVICE.type == 'cuda' else 'n/a'}")


# ── Load cached features & scalers from Step 1 ────────────────
print("\n[1/8] Loading data + cached features from Step 1...")

train_raw = pd.read_csv(BASE + "train.csv")
test_df   = pd.read_csv(BASE + "test.csv")

d1 = pd.read_csv(BASE + "train_supplement/dataset1.csv").rename(columns={"TC_mean": "Tc"})
d3 = pd.read_csv(BASE + "train_supplement/dataset3.csv")
d4 = pd.read_csv(BASE + "train_supplement/dataset4.csv")

def supp_rows(df, col):
    rows = []
    for _, r in df.iterrows():
        row = {"id": -1, "SMILES": r["SMILES"]}
        for t in TARGETS:
            row[t] = r[col] if t == col else np.nan
        rows.append(row)
    return rows

supp_df  = pd.DataFrame(supp_rows(d1, "Tc") + supp_rows(d3, "Tg") + supp_rows(d4, "FFV"))
train_df = pd.concat([train_raw, supp_df], ignore_index=True)

# Load cached scaled features
X_all_sc  = np.load(WORK + "feats_train_v2.npy")
X_test_raw = np.load(WORK + "feats_test_v2.npy")

X_all_sc  = np.nan_to_num(X_all_sc,  nan=0.0, posinf=0.0, neginf=0.0)
X_test_raw = np.nan_to_num(X_test_raw, nan=0.0, posinf=0.0, neginf=0.0)

FEAT_DIM = X_all_sc.shape[1]

# Re-fit the same RobustScaler on the full training data
scaler_X = RobustScaler()
X_scaled  = scaler_X.fit_transform(X_all_sc).astype(np.float32)
X_test_sc = scaler_X.transform(X_test_raw).astype(np.float32)

Y_raw = train_df[TARGETS].values.astype(np.float32)

print(f"  Feature dim  : {FEAT_DIM}")
print(f"  Train rows   : {len(train_df)}")
for t in TARGETS:
    i = TARGETS.index(t)
    print(f"    {t:<10}: {(~np.isnan(Y_raw[:,i])).sum():>5} labeled")


# ── Model architecture (same as Step 1) ───────────────────────

class ResBlock(nn.Module):
    def __init__(self, dim, dropout=0.25):
        super().__init__()
        self.block = nn.Sequential(
            nn.LayerNorm(dim),
            nn.Linear(dim, dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(dim, dim),
        )
        self.act = nn.GELU()
    def forward(self, x):
        return self.act(x + self.block(x))


class SpecialistNet(nn.Module):
    def __init__(self, feat_dim, hidden=512, dropout=0.30):
        super().__init__()
        self.enc = nn.Sequential(
            nn.Linear(feat_dim, hidden * 2),
            nn.LayerNorm(hidden * 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden * 2, hidden),
            nn.LayerNorm(hidden),
            nn.GELU(),
        )
        self.res = nn.Sequential(
            ResBlock(hidden, dropout),
            ResBlock(hidden, dropout),
            ResBlock(hidden, dropout),
            ResBlock(hidden, dropout),
        )
        self.head = nn.Sequential(
            nn.Linear(hidden, hidden // 2),
            nn.GELU(),
            nn.Dropout(dropout * 0.5),
            nn.Linear(hidden // 2, hidden // 4),
            nn.GELU(),
            nn.Linear(hidden // 4, 1),
        )
    def forward(self, x):
        return self.head(self.res(self.enc(x))).squeeze(-1)


# ── SMILES helpers ─────────────────────────────────────────────
_DESC_NAMES = [name for name, _ in Descriptors.descList]

def smiles_to_mol(smi):
    for v in [smi.replace("*", "[At]"), smi.replace("*", ""), smi]:
        try:
            mol = Chem.MolFromSmiles(v)
            if mol is not None:
                return mol
        except Exception:
            pass
    return None

def extract_features(smi):
    mol = smiles_to_mol(str(smi))
    rdkit = np.zeros(len(_DESC_NAMES), dtype=np.float32)
    if mol is not None:
        for i, name in enumerate(_DESC_NAMES):
            try:
                v = getattr(Descriptors, name)(mol)
                rdkit[i] = float(v) if v is not None else 0.0
            except Exception:
                pass
    m2 = np.zeros(2048, dtype=np.float32)
    if mol is not None:
        try:
            fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius=2, nBits=2048)
            m2[:] = np.array(fp, dtype=np.float32)
        except Exception:
            pass
    m3 = np.zeros(1024, dtype=np.float32)
    if mol is not None:
        try:
            fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius=3, nBits=1024)
            m3[:] = np.array(fp, dtype=np.float32)
        except Exception:
            pass
    mac = np.zeros(167, dtype=np.float32)
    if mol is not None:
        try:
            mac[:] = np.array(MACCSkeys.GenMACCSKeys(mol), dtype=np.float32)
        except Exception:
            pass
    return np.concatenate([rdkit, m2, m3, mac])

def augment_smiles(smi, n=4):
    mol = smiles_to_mol(str(smi))
    if mol is None:
        return []
    variants = set()
    for _ in range(n * 5):
        try:
            rs = Chem.MolToSmiles(mol, doRandom=True, canonical=False)
            variants.add(rs)
        except Exception:
            pass
    return list(variants)[:n]


# ── Specialist training function ───────────────────────────────

def train_specialist(model, X_tr, y_tr, X_va, y_va,
                     epochs=500, batch=64, lr=1e-3,
                     patience=50, device=DEVICE, tag="Spec"):
    """Train single-target SpecialistNet. Returns (model, r2, mu, sig)."""
    mu  = y_tr.mean()
    sig = y_tr.std() + 1e-8
    y_tr_n = ((y_tr - mu) / sig).astype(np.float32)
    y_va_n = ((y_va - mu) / sig).astype(np.float32)

    ds  = TensorDataset(torch.tensor(X_tr), torch.tensor(y_tr_n))
    ldr = DataLoader(ds, batch_size=batch, shuffle=True, drop_last=False)

    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=5e-3)
    sch = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(opt, T_0=80, T_mult=2)

    best_r2, best_state, wait = -1.0, None, 0

    X_va_t = torch.tensor(X_va, device=device)
    y_va_t = torch.tensor(y_va_n)

    n_params = sum(p.numel() for p in model.parameters())
    print(f"\n  [{tag}]  train={len(y_tr)}  val={len(y_va)}  params={n_params:,}")

    for ep in range(1, epochs + 1):
        model.train()
        for xb, yb in ldr:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad()
            loss = F.mse_loss(model(xb), yb)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
        sch.step()

        if ep % 5 == 0 or ep == 1:
            model.eval()
            with torch.no_grad():
                pn = model(X_va_t).cpu().numpy()
            p_real = pn * sig + mu
            r2  = r2_score(y_va, p_real)
            mae = mean_absolute_error(y_va, p_real)

            if r2 > best_r2:
                best_r2    = r2
                best_state = {k: v.clone() for k, v in model.state_dict().items()}
                wait = 0
                marker = " ✓"
            else:
                wait += 1
                marker = ""

            if ep % 50 == 0 or ep == 1:
                print(f"    ep={ep:>4}  R²={r2:.4f} [best={best_r2:.4f}]  MAE={mae:.4f}{marker}")

            if wait * 5 >= patience:
                print(f"    ⚡ Early stop ep={ep}  best R²={best_r2:.4f}")
                break

    model.load_state_dict(best_state)
    return model, best_r2, mu, sig





# ─────────────────────────────────────────────────────────────
# SECTION 2: Tc SPECIALIST  (the biggest bottleneck: R²=0.53)
# ─────────────────────────────────────────────────────────────
print("\n[2/8] Training Tc Specialist...")

tc_mask  = ~np.isnan(Y_raw[:, 2])
X_tc_base = X_scaled[tc_mask]
y_tc_base = Y_raw[tc_mask, 2]
sm_tc     = train_df["SMILES"].values[tc_mask]

print(f"  Base Tc samples: {len(y_tc_base)}")
print("  Running SMILES augmentation (×4)...")

extra_X_tc, extra_y_tc = [], []
for i, (smi, yval) in enumerate(zip(sm_tc, y_tc_base)):
    for v in augment_smiles(smi, n=4):
        try:
            fv = extract_features(v)
            fv = np.nan_to_num(fv, nan=0.0, posinf=0.0, neginf=0.0)
            fv_sc = scaler_X.transform(fv.reshape(1, -1))[0].astype(np.float32)
            extra_X_tc.append(fv_sc)
            extra_y_tc.append(yval)
        except Exception:
            pass
    if (i + 1) % 400 == 0:
        print(f"    Tc aug: {i+1}/{len(sm_tc)}")

if extra_X_tc:
    X_tc_all = np.vstack([X_tc_base, np.array(extra_X_tc)])
    y_tc_all = np.concatenate([y_tc_base, np.array(extra_y_tc)])
else:
    X_tc_all = X_tc_base
    y_tc_all = y_tc_base

print(f"  Augmented Tc samples: {len(y_tc_all)}")

X_tc_tr, X_tc_va, y_tc_tr, y_tc_va = train_test_split(
    X_tc_all, y_tc_all, test_size=0.15, random_state=SEED
)

tc_model = SpecialistNet(FEAT_DIM, hidden=512, dropout=0.25).to(DEVICE)
tc_model, tc_r2, tc_mu, tc_sig = train_specialist(
    tc_model, X_tc_tr, y_tc_tr, X_tc_va, y_tc_va,
    epochs=500, batch=128, lr=1e-3, patience=50,
    device=DEVICE, tag="Tc-Specialist",
)
torch.save({"state": tc_model.state_dict(), "mu": tc_mu, "sig": tc_sig},
           WORK + "pinn_tc_spec.pth")
print(f"  ✓ Tc specialist saved  (R²={tc_r2:.4f})")


# ─────────────────────────────────────────────────────────────
# SECTION 3: Rg SPECIALIST v2  (add SMILES augmentation)
# ─────────────────────────────────────────────────────────────
print("\n[3/8] Training Rg Specialist v2 (with augmentation)...")

rg_mask   = ~np.isnan(Y_raw[:, 4])
X_rg_base = X_scaled[rg_mask]
y_rg_base = Y_raw[rg_mask, 4]
sm_rg     = train_df["SMILES"].values[rg_mask]

print(f"  Base Rg samples: {len(y_rg_base)}")
print("  Running SMILES augmentation (×5)...")

extra_X_rg, extra_y_rg = [], []
for i, (smi, yval) in enumerate(zip(sm_rg, y_rg_base)):
    for v in augment_smiles(smi, n=5):
        try:
            fv = extract_features(v)
            fv = np.nan_to_num(fv, nan=0.0, posinf=0.0, neginf=0.0)
            fv_sc = scaler_X.transform(fv.reshape(1, -1))[0].astype(np.float32)
            extra_X_rg.append(fv_sc)
            extra_y_rg.append(yval)
        except Exception:
            pass
    if (i + 1) % 200 == 0:
        print(f"    Rg aug: {i+1}/{len(sm_rg)}")

if extra_X_rg:
    X_rg_all = np.vstack([X_rg_base, np.array(extra_X_rg)])
    y_rg_all = np.concatenate([y_rg_base, np.array(extra_y_rg)])
else:
    X_rg_all = X_rg_base
    y_rg_all = y_rg_base

print(f"  Augmented Rg samples: {len(y_rg_all)}")

X_rg_tr, X_rg_va, y_rg_tr, y_rg_va = train_test_split(
    X_rg_all, y_rg_all, test_size=0.15, random_state=SEED
)

rg_model_v2 = SpecialistNet(FEAT_DIM, hidden=512, dropout=0.25).to(DEVICE)
rg_model_v2, rg_v2_r2, rg_mu, rg_sig = train_specialist(
    rg_model_v2, X_rg_tr, y_rg_tr, X_rg_va, y_rg_va,
    epochs=500, batch=64, lr=1e-3, patience=50,
    device=DEVICE, tag="Rg-Specialist-v2",
)
torch.save({"state": rg_model_v2.state_dict(), "mu": rg_mu, "sig": rg_sig},
           WORK + "pinn_rg_spec_v2.pth")
print(f"  ✓ Rg specialist v2 saved  (R²={rg_v2_r2:.4f})")


# ─────────────────────────────────────────────────────────────
# SECTION 4: FFV SPECIALIST  (7892 labels, push from 0.85→0.90)
# ─────────────────────────────────────────────────────────────
print("\n[4/8] Training FFV Specialist...")

ffv_mask   = ~np.isnan(Y_raw[:, 1])
X_ffv_all  = X_scaled[ffv_mask]
y_ffv_all  = Y_raw[ffv_mask, 1]

print(f"  FFV samples: {len(y_ffv_all)}  (no augmentation needed)")

X_ffv_tr, X_ffv_va, y_ffv_tr, y_ffv_va = train_test_split(
    X_ffv_all, y_ffv_all, test_size=0.15, random_state=SEED
)

ffv_model = SpecialistNet(FEAT_DIM, hidden=512, dropout=0.20).to(DEVICE)
ffv_model, ffv_r2, ffv_mu, ffv_sig = train_specialist(
    ffv_model, X_ffv_tr, y_ffv_tr, X_ffv_va, y_ffv_va,
    epochs=300, batch=256, lr=1e-3, patience=40,
    device=DEVICE, tag="FFV-Specialist",
)
torch.save({"state": ffv_model.state_dict(), "mu": ffv_mu, "sig": ffv_sig},
           WORK + "pinn_ffv_spec.pth")
print(f"  ✓ FFV specialist saved  (R²={ffv_r2:.4f})")


# ─────────────────────────────────────────────────────────────
# SECTION 5: Density SPECIALIST  (optional boost)
# ─────────────────────────────────────────────────────────────
print("\n[5/8] Training Density Specialist...")

dn_mask   = ~np.isnan(Y_raw[:, 3])
X_dn_base = X_scaled[dn_mask]
y_dn_base = Y_raw[dn_mask, 3]
sm_dn     = train_df["SMILES"].values[dn_mask]

print(f"  Base Density samples: {len(y_dn_base)}")
print("  Running SMILES augmentation (×4)...")

extra_X_dn, extra_y_dn = [], []
for i, (smi, yval) in enumerate(zip(sm_dn, y_dn_base)):
    for v in augment_smiles(smi, n=4):
        try:
            fv = extract_features(v)
            fv = np.nan_to_num(fv, nan=0.0, posinf=0.0, neginf=0.0)
            fv_sc = scaler_X.transform(fv.reshape(1, -1))[0].astype(np.float32)
            extra_X_dn.append(fv_sc)
            extra_y_dn.append(yval)
        except Exception:
            pass
    if (i + 1) % 200 == 0:
        print(f"    Density aug: {i+1}/{len(sm_dn)}")

if extra_X_dn:
    X_dn_all = np.vstack([X_dn_base, np.array(extra_X_dn)])
    y_dn_all = np.concatenate([y_dn_base, np.array(extra_y_dn)])
else:
    X_dn_all = X_dn_base
    y_dn_all = y_dn_base

print(f"  Augmented Density samples: {len(y_dn_all)}")

X_dn_tr, X_dn_va, y_dn_tr, y_dn_va = train_test_split(
    X_dn_all, y_dn_all, test_size=0.15, random_state=SEED
)

dn_model = SpecialistNet(FEAT_DIM, hidden=512, dropout=0.25).to(DEVICE)
dn_model, dn_r2, dn_mu, dn_sig = train_specialist(
    dn_model, X_dn_tr, y_dn_tr, X_dn_va, y_dn_va,
    epochs=500, batch=64, lr=1e-3, patience=50,
    device=DEVICE, tag="Density-Specialist",
)
torch.save({"state": dn_model.state_dict(), "mu": dn_mu, "sig": dn_sig},
           WORK + "pinn_density_spec.pth")
print(f"  ✓ Density specialist saved  (R²={dn_r2:.4f})")


# ─────────────────────────────────────────────────────────────
# SECTION 6: Load Tg specialist from Step 1 & main model preds
# ─────────────────────────────────────────────────────────────
print("\n[6/8] Loading Step 1 specialists...")

# Load Tg specialist
tg_ckpt = torch.load(WORK + "pinn_tg_spec.pth", map_location=DEVICE)
tg_model = SpecialistNet(FEAT_DIM, hidden=512, dropout=0.30).to(DEVICE)
tg_model.load_state_dict(tg_ckpt["state"])
tg_mu, tg_sig = float(tg_ckpt["mu"]), float(tg_ckpt["sig"])
tg_model.eval()

# Load old Rg specialist (for comparison / fallback)
rg_ckpt_v1 = torch.load(WORK + "pinn_rg_spec.pth", map_location=DEVICE)
rg_model_v1 = SpecialistNet(FEAT_DIM, hidden=512, dropout=0.25).to(DEVICE)
rg_model_v1.load_state_dict(rg_ckpt_v1["state"])
rg_mu_v1, rg_sig_v1 = float(rg_ckpt_v1["mu"]), float(rg_ckpt_v1["sig"])
rg_model_v1.eval()

# Load main PINN
from types import SimpleNamespace

class MultiTaskPINN_Load(nn.Module):
    """Minimal loader — same arch as Step 1 main model."""
    class _ResBlock(nn.Module):
        def __init__(self, dim, dropout=0.2):
            super().__init__()
            self.block = nn.Sequential(
                nn.LayerNorm(dim), nn.Linear(dim, dim), nn.GELU(),
                nn.Dropout(dropout), nn.Linear(dim, dim),
            )
            self.act = nn.GELU()
        def forward(self, x): return self.act(x + self.block(x))

    def __init__(self, feat_dim, hidden=768, n_targets=5, dropout=0.2):
        super().__init__()
        self.proj = nn.Sequential(
            nn.Linear(feat_dim, hidden * 2), nn.LayerNorm(hidden * 2),
            nn.GELU(), nn.Dropout(dropout), nn.Linear(hidden * 2, hidden),
            nn.LayerNorm(hidden), nn.GELU(),
        )
        self.backbone = nn.Sequential(*[self._ResBlock(hidden, dropout) for _ in range(6)])
        self.heads = nn.ModuleList([
            nn.Sequential(
                nn.Linear(hidden, hidden // 2), nn.GELU(),
                nn.Dropout(dropout * 0.5), nn.Linear(hidden // 2, hidden // 4),
                nn.GELU(), nn.Linear(hidden // 4, 1),
            ) for _ in range(n_targets)
        ])
    def forward(self, x):
        h = self.backbone(self.proj(x))
        return torch.cat([head(h) for head in self.heads], dim=1)

main_ckpt = torch.load(WORK + "pinn_main.pth", map_location=DEVICE)
main_model = MultiTaskPINN_Load(FEAT_DIM, hidden=768, n_targets=5, dropout=0.2).to(DEVICE)
main_model.load_state_dict(main_ckpt)
main_model.eval()

# Per-target mean/std from Step 1 training data (recompute)
Y_raw_all = train_df[TARGETS].values.astype(np.float32)
t_mean = np.zeros(5, dtype=np.float32)
t_std  = np.ones(5, dtype=np.float32)
for i in range(5):
    mask = ~np.isnan(Y_raw_all[:, i])
    vals = Y_raw_all[mask, i]
    t_mean[i] = vals.mean()
    t_std[i]  = vals.std() + 1e-8

print("  ✓ All Step 1 models loaded")


# ─────────────────────────────────────────────────────────────
# SECTION 7: Test set predictions from ALL specialists
# ─────────────────────────────────────────────────────────────
print("\n[7/8] Generating final test predictions...")

X_test_t = torch.tensor(X_test_sc, device=DEVICE)

def predict_spec(model, mu, sig, X_t):
    model.eval()
    with torch.no_grad():
        return model(X_t).cpu().numpy() * sig + mu

# Main model predictions (all 5 targets, z-space → real)
with torch.no_grad():
    main_sc = main_model(X_test_t).cpu().numpy()
main_preds = {t: main_sc[:, i] * t_std[i] + t_mean[i] for i, t in enumerate(TARGETS)}

# Specialist predictions
tg_preds  = predict_spec(tg_model,     tg_mu,  tg_sig,  X_test_t)
tc_preds  = predict_spec(tc_model,     tc_mu,  tc_sig,  X_test_t)
rg_preds  = predict_spec(rg_model_v2,  rg_mu,  rg_sig,  X_test_t)
ffv_preds = predict_spec(ffv_model,    ffv_mu, ffv_sig, X_test_t)
dn_preds  = predict_spec(dn_model,     dn_mu,  dn_sig,  X_test_t)

# R² values from val sets (Step 1 val approximations for main model)
MAIN_R2 = {
    "Tg":      0.6498,   # from Step 1 output
    "FFV":     0.8483,
    "Tc":      0.5343,
    "Density": 0.8279,
    "Rg":      0.6947,
}
SPEC_R2 = {
    "Tg":      0.9730,   # from Step 1
    "FFV":     ffv_r2,
    "Tc":      tc_r2,
    "Density": dn_r2,
    "Rg":      rg_v2_r2,
}

print(f"\n  Specialist R² summary:")
print(f"  {'Target':<12} {'Main':>8} {'Spec':>8}  {'Winner'}")
print(f"  {'='*40}")
for t in TARGETS:
    m, s = MAIN_R2[t], SPEC_R2[t]
    winner = "SPEC ✓" if s > m else "MAIN"
    print(f"  {t:<12} {m:>8.4f} {s:>8.4f}  {winner}")

# Weighted blend: R²-proportional per target
def blend(main_p, spec_p, r2_main, r2_spec):
    wm = max(0.0, r2_main)
    ws = max(0.0, r2_spec)
    tot = wm + ws + 1e-8
    return (wm * main_p + ws * spec_p) / tot

final_preds = {
    "Tg":      blend(main_preds["Tg"],      tg_preds,  MAIN_R2["Tg"],      SPEC_R2["Tg"]),
    "FFV":     blend(main_preds["FFV"],     ffv_preds,  MAIN_R2["FFV"],     SPEC_R2["FFV"]),
    "Tc":      blend(main_preds["Tc"],      tc_preds,   MAIN_R2["Tc"],      SPEC_R2["Tc"]),
    "Density": blend(main_preds["Density"], dn_preds,   MAIN_R2["Density"], SPEC_R2["Density"]),
    "Rg":      blend(main_preds["Rg"],      rg_preds,   MAIN_R2["Rg"],      SPEC_R2["Rg"]),
}

sub3 = pd.DataFrame({"id": test_df["id"]})
for t in TARGETS:
    sub3[t] = final_preds[t]
sub3.to_csv(WORK + "submission_model3.csv", index=False)
print(f"\n✓ submission_model3.csv (updated) saved")
print(sub3.to_string())


# ─────────────────────────────────────────────────────────────
# SECTION 8: Per-target weighted ensemble across all 3 models
# ─────────────────────────────────────────────────────────────
print("\n[8/8] Final per-target weighted ensemble...")

try:
    sub1 = pd.read_csv(WORK + "submission.csv")
    sub2 = pd.read_csv(WORK + "submission_model2.csv")

    # Approximate per-target R² for M1 and M2 (update if you have exact logs)
    M1_R2 = {"Tg": 0.72, "FFV": 0.80, "Tc": 0.79, "Density": 0.77, "Rg": 0.72}
    M2_R2 = {"Tg": 0.78, "FFV": 0.87, "Tc": 0.84, "Density": 0.85, "Rg": 0.75}
    M3_R2 = SPEC_R2   # use specialist R² as M3's per-target score

    ensemble = pd.DataFrame({"id": sub1["id"]})
    print(f"\n  Final per-target blend weights:")
    print(f"  {'Target':<12} {'w_M1':>8} {'w_M2':>8} {'w_M3':>8}  {'M3 R²':>8}")
    print(f"  {'='*55}")
    for t in TARGETS:
        w1 = max(0.01, M1_R2.get(t, 0.5))
        w2 = max(0.01, M2_R2.get(t, 0.5))
        w3 = max(0.01, M3_R2.get(t, 0.5))
        total = w1 + w2 + w3
        w1n, w2n, w3n = w1/total, w2/total, w3/total
        ensemble[t] = w1n * sub1[t] + w2n * sub2[t] + w3n * sub3[t]
        print(f"  {t:<12} {w1n:>8.3f} {w2n:>8.3f} {w3n:>8.3f}  {M3_R2[t]:>8.4f}")

    ensemble.to_csv(WORK + "submission_ensemble.csv", index=False)
    print(f"\n✓ submission_ensemble.csv (final) saved")
    print(ensemble.to_string())

except FileNotFoundError as e:
    print(f"  ⚠ Models 1/2 not found: {e}")
    print("  Saving Model 3 standalone as ensemble fallback.")
    sub3.to_csv(WORK + "submission_ensemble.csv", index=False)

# ── Final summary ──────────────────────────────────────────
print(f"\n{'='*60}")
print("  STEP 2 COMPLETE — Specialist R² summary:")
print(f"{'='*60}")
expected_avg = np.mean(list(SPEC_R2.values()))
for t in TARGETS:
    flag = "✓" if SPEC_R2[t] >= 0.90 else "→"
    print(f"  {t:<12} {SPEC_R2[t]:>8.4f}  {flag}")
print(f"  {'─'*30}")
print(f"  {'AVERAGE':<12} {expected_avg:>8.4f}")
print(f"\n  Files saved:")
print(f"    {WORK}pinn_tc_spec.pth")
print(f"    {WORK}pinn_rg_spec_v2.pth")
print(f"    {WORK}pinn_ffv_spec.pth")
print(f"    {WORK}pinn_density_spec.pth")
print(f"    {WORK}submission_model3.csv  (updated)")
print(f"    {WORK}submission_ensemble.csv  (final)")
print(f"{'='*60}")

Device: cuda
GPU   : Tesla T4

[1/8] Loading data + cached features from Step 1...
  Feature dim  : 3456
  Train rows   : 9755
    Tg        :   557 labeled
    FFV       :  7892 labeled
    Tc        :  1611 labeled
    Density   :   613 labeled
    Rg        :   614 labeled

[2/8] Training Tc Specialist...
  Base Tc samples: 1611
  Running SMILES augmentation (×4)...
    Tc aug: 400/1611
    Tc aug: 800/1611
    Tc aug: 1200/1611
    Tc aug: 1600/1611
  Augmented Tc samples: 8052

  [Tc-Specialist]  train=6844  val=1208  params=6,337,537
    ep=   1  R²=0.8735 [best=0.8735]  MAE=0.0203 ✓
    ep=  50  R²=0.9815 [best=0.9815]  MAE=0.0054 ✓
    ep= 100  R²=0.9832 [best=0.9868]  MAE=0.0056
    ep= 150  R²=0.9845 [best=0.9870]  MAE=0.0060
    ⚡ Early stop ep=170  best R²=0.9870
  ✓ Tc specialist saved  (R²=0.9870)

[3/8] Training Rg Specialist v2 (with augmentation)...
  Base Rg samples: 614
  Running SMILES augmentation (×5)...
    Rg aug: 200/614
    Rg aug: 400/614
    Rg aug: 600/614


UnpicklingError: Weights only load failed. This file can still be loaded, to do so you have two options, [1mdo those steps only if you trust the source of the checkpoint[0m. 
	(1) In PyTorch 2.6, we changed the default value of the `weights_only` argument in `torch.load` from `False` to `True`. Re-running `torch.load` with `weights_only` set to `False` will likely succeed, but it can result in arbitrary code execution. Do it only if you got the file from a trusted source.
	(2) Alternatively, to load with `weights_only=True` please check the recommended steps in the following error message.
	WeightsUnpickler error: Unsupported global: GLOBAL numpy._core.multiarray.scalar was not an allowed global by default. Please use `torch.serialization.add_safe_globals([numpy._core.multiarray.scalar])` or the `torch.serialization.safe_globals([numpy._core.multiarray.scalar])` context manager to allowlist this global if you trust this class/function.

Check the documentation of torch.load to learn 

In [10]:
# ================================================================
#  MODEL 3  —  STEP 2: Tc & Rg SPECIALISTS + FINAL ENSEMBLE
#  Run this cell AFTER model3_enhanced_pinn.py completes.
#
#  What this fixes:
#    Tc   : 0.5343 → target 0.88+   (specialist + SMILES aug)
#    Rg   : 0.7230 → target 0.85+   (SMILES aug added)
#    FFV  : 0.8483 → target 0.90+   (dedicated specialist)
#    Final avg R²: 0.78 → 0.88+ expected
# ================================================================

import os, gc, warnings
import numpy as np
import pandas as pd
from sklearn.preprocessing import RobustScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from rdkit import Chem
from rdkit.Chem import AllChem, MACCSkeys, Descriptors

warnings.filterwarnings("ignore")
SEED   = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
WORK   = "/kaggle/working/"
BASE   = "/kaggle/input/competitions/neurips-open-polymer-prediction-2025/"
TARGETS = ["Tg", "FFV", "Tc", "Density", "Rg"]

print(f"Device: {DEVICE}")
print(f"GPU   : {torch.cuda.get_device_name(0) if DEVICE.type == 'cuda' else 'n/a'}")


# ── Load cached features & scalers from Step 1 ────────────────
print("\n[1/8] Loading data + cached features from Step 1...")

train_raw = pd.read_csv(BASE + "train.csv")
test_df   = pd.read_csv(BASE + "test.csv")

d1 = pd.read_csv(BASE + "train_supplement/dataset1.csv").rename(columns={"TC_mean": "Tc"})
d3 = pd.read_csv(BASE + "train_supplement/dataset3.csv")
d4 = pd.read_csv(BASE + "train_supplement/dataset4.csv")

def supp_rows(df, col):
    rows = []
    for _, r in df.iterrows():
        row = {"id": -1, "SMILES": r["SMILES"]}
        for t in TARGETS:
            row[t] = r[col] if t == col else np.nan
        rows.append(row)
    return rows

supp_df  = pd.DataFrame(supp_rows(d1, "Tc") + supp_rows(d3, "Tg") + supp_rows(d4, "FFV"))
train_df = pd.concat([train_raw, supp_df], ignore_index=True)

# Load cached scaled features
X_all_sc  = np.load(WORK + "feats_train_v2.npy")
X_test_raw = np.load(WORK + "feats_test_v2.npy")

X_all_sc  = np.nan_to_num(X_all_sc,  nan=0.0, posinf=0.0, neginf=0.0)
X_test_raw = np.nan_to_num(X_test_raw, nan=0.0, posinf=0.0, neginf=0.0)

FEAT_DIM = X_all_sc.shape[1]

# Re-fit the same RobustScaler on the full training data
scaler_X = RobustScaler()
X_scaled  = scaler_X.fit_transform(X_all_sc).astype(np.float32)
X_test_sc = scaler_X.transform(X_test_raw).astype(np.float32)

Y_raw = train_df[TARGETS].values.astype(np.float32)

print(f"  Feature dim  : {FEAT_DIM}")
print(f"  Train rows   : {len(train_df)}")
for t in TARGETS:
    i = TARGETS.index(t)
    print(f"    {t:<10}: {(~np.isnan(Y_raw[:,i])).sum():>5} labeled")


# ── Model architecture (same as Step 1) ───────────────────────

class ResBlock(nn.Module):
    def __init__(self, dim, dropout=0.25):
        super().__init__()
        self.block = nn.Sequential(
            nn.LayerNorm(dim),
            nn.Linear(dim, dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(dim, dim),
        )
        self.act = nn.GELU()
    def forward(self, x):
        return self.act(x + self.block(x))


class SpecialistNet(nn.Module):
    def __init__(self, feat_dim, hidden=512, dropout=0.30):
        super().__init__()
        self.enc = nn.Sequential(
            nn.Linear(feat_dim, hidden * 2),
            nn.LayerNorm(hidden * 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden * 2, hidden),
            nn.LayerNorm(hidden),
            nn.GELU(),
        )
        self.res = nn.Sequential(
            ResBlock(hidden, dropout),
            ResBlock(hidden, dropout),
            ResBlock(hidden, dropout),
            ResBlock(hidden, dropout),
        )
        self.head = nn.Sequential(
            nn.Linear(hidden, hidden // 2),
            nn.GELU(),
            nn.Dropout(dropout * 0.5),
            nn.Linear(hidden // 2, hidden // 4),
            nn.GELU(),
            nn.Linear(hidden // 4, 1),
        )
    def forward(self, x):
        return self.head(self.res(self.enc(x))).squeeze(-1)


# ── SMILES helpers ─────────────────────────────────────────────
_DESC_NAMES = [name for name, _ in Descriptors.descList]

def smiles_to_mol(smi):
    for v in [smi.replace("*", "[At]"), smi.replace("*", ""), smi]:
        try:
            mol = Chem.MolFromSmiles(v)
            if mol is not None:
                return mol
        except Exception:
            pass
    return None

def extract_features(smi):
    mol = smiles_to_mol(str(smi))
    rdkit = np.zeros(len(_DESC_NAMES), dtype=np.float32)
    if mol is not None:
        for i, name in enumerate(_DESC_NAMES):
            try:
                v = getattr(Descriptors, name)(mol)
                rdkit[i] = float(v) if v is not None else 0.0
            except Exception:
                pass
    m2 = np.zeros(2048, dtype=np.float32)
    if mol is not None:
        try:
            fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius=2, nBits=2048)
            m2[:] = np.array(fp, dtype=np.float32)
        except Exception:
            pass
    m3 = np.zeros(1024, dtype=np.float32)
    if mol is not None:
        try:
            fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius=3, nBits=1024)
            m3[:] = np.array(fp, dtype=np.float32)
        except Exception:
            pass
    mac = np.zeros(167, dtype=np.float32)
    if mol is not None:
        try:
            mac[:] = np.array(MACCSkeys.GenMACCSKeys(mol), dtype=np.float32)
        except Exception:
            pass
    return np.concatenate([rdkit, m2, m3, mac])

def augment_smiles(smi, n=4):
    mol = smiles_to_mol(str(smi))
    if mol is None:
        return []
    variants = set()
    for _ in range(n * 5):
        try:
            rs = Chem.MolToSmiles(mol, doRandom=True, canonical=False)
            variants.add(rs)
        except Exception:
            pass
    return list(variants)[:n]


# ── Specialist training function ───────────────────────────────

def train_specialist(model, X_tr, y_tr, X_va, y_va,
                     epochs=500, batch=64, lr=1e-3,
                     patience=50, device=DEVICE, tag="Spec"):
    """Train single-target SpecialistNet. Returns (model, r2, mu, sig)."""
    mu  = y_tr.mean()
    sig = y_tr.std() + 1e-8
    y_tr_n = ((y_tr - mu) / sig).astype(np.float32)
    y_va_n = ((y_va - mu) / sig).astype(np.float32)

    ds  = TensorDataset(torch.tensor(X_tr), torch.tensor(y_tr_n))
    ldr = DataLoader(ds, batch_size=batch, shuffle=True, drop_last=False)

    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=5e-3)
    sch = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(opt, T_0=80, T_mult=2)

    best_r2, best_state, wait = -1.0, None, 0

    X_va_t = torch.tensor(X_va, device=device)
    y_va_t = torch.tensor(y_va_n)

    n_params = sum(p.numel() for p in model.parameters())
    print(f"\n  [{tag}]  train={len(y_tr)}  val={len(y_va)}  params={n_params:,}")

    for ep in range(1, epochs + 1):
        model.train()
        for xb, yb in ldr:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad()
            loss = F.mse_loss(model(xb), yb)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
        sch.step()

        if ep % 5 == 0 or ep == 1:
            model.eval()
            with torch.no_grad():
                pn = model(X_va_t).cpu().numpy()
            p_real = pn * sig + mu
            r2  = r2_score(y_va, p_real)
            mae = mean_absolute_error(y_va, p_real)

            if r2 > best_r2:
                best_r2    = r2
                best_state = {k: v.clone() for k, v in model.state_dict().items()}
                wait = 0
                marker = " ✓"
            else:
                wait += 1
                marker = ""

            if ep % 50 == 0 or ep == 1:
                print(f"    ep={ep:>4}  R²={r2:.4f} [best={best_r2:.4f}]  MAE={mae:.4f}{marker}")

            if wait * 5 >= patience:
                print(f"    ⚡ Early stop ep={ep}  best R²={best_r2:.4f}")
                break

    model.load_state_dict(best_state)
    return model, best_r2, mu, sig





# ─────────────────────────────────────────────────────────────
# SECTION 2: Tc SPECIALIST  (the biggest bottleneck: R²=0.53)
# ─────────────────────────────────────────────────────────────
print("\n[2/8] Training Tc Specialist...")

tc_mask  = ~np.isnan(Y_raw[:, 2])
X_tc_base = X_scaled[tc_mask]
y_tc_base = Y_raw[tc_mask, 2]
sm_tc     = train_df["SMILES"].values[tc_mask]

print(f"  Base Tc samples: {len(y_tc_base)}")
print("  Running SMILES augmentation (×4)...")

extra_X_tc, extra_y_tc = [], []
for i, (smi, yval) in enumerate(zip(sm_tc, y_tc_base)):
    for v in augment_smiles(smi, n=4):
        try:
            fv = extract_features(v)
            fv = np.nan_to_num(fv, nan=0.0, posinf=0.0, neginf=0.0)
            fv_sc = scaler_X.transform(fv.reshape(1, -1))[0].astype(np.float32)
            extra_X_tc.append(fv_sc)
            extra_y_tc.append(yval)
        except Exception:
            pass
    if (i + 1) % 400 == 0:
        print(f"    Tc aug: {i+1}/{len(sm_tc)}")

if extra_X_tc:
    X_tc_all = np.vstack([X_tc_base, np.array(extra_X_tc)])
    y_tc_all = np.concatenate([y_tc_base, np.array(extra_y_tc)])
else:
    X_tc_all = X_tc_base
    y_tc_all = y_tc_base

print(f"  Augmented Tc samples: {len(y_tc_all)}")

X_tc_tr, X_tc_va, y_tc_tr, y_tc_va = train_test_split(
    X_tc_all, y_tc_all, test_size=0.15, random_state=SEED
)

tc_model = SpecialistNet(FEAT_DIM, hidden=512, dropout=0.25).to(DEVICE)
tc_model, tc_r2, tc_mu, tc_sig = train_specialist(
    tc_model, X_tc_tr, y_tc_tr, X_tc_va, y_tc_va,
    epochs=500, batch=128, lr=1e-3, patience=50,
    device=DEVICE, tag="Tc-Specialist",
)
torch.save({"state": tc_model.state_dict(), "mu": tc_mu, "sig": tc_sig},
           WORK + "pinn_tc_spec.pth")
print(f"  ✓ Tc specialist saved  (R²={tc_r2:.4f})")


# ─────────────────────────────────────────────────────────────
# SECTION 3: Rg SPECIALIST v2  (add SMILES augmentation)
# ─────────────────────────────────────────────────────────────
print("\n[3/8] Training Rg Specialist v2 (with augmentation)...")

rg_mask   = ~np.isnan(Y_raw[:, 4])
X_rg_base = X_scaled[rg_mask]
y_rg_base = Y_raw[rg_mask, 4]
sm_rg     = train_df["SMILES"].values[rg_mask]

print(f"  Base Rg samples: {len(y_rg_base)}")
print("  Running SMILES augmentation (×5)...")

extra_X_rg, extra_y_rg = [], []
for i, (smi, yval) in enumerate(zip(sm_rg, y_rg_base)):
    for v in augment_smiles(smi, n=5):
        try:
            fv = extract_features(v)
            fv = np.nan_to_num(fv, nan=0.0, posinf=0.0, neginf=0.0)
            fv_sc = scaler_X.transform(fv.reshape(1, -1))[0].astype(np.float32)
            extra_X_rg.append(fv_sc)
            extra_y_rg.append(yval)
        except Exception:
            pass
    if (i + 1) % 200 == 0:
        print(f"    Rg aug: {i+1}/{len(sm_rg)}")

if extra_X_rg:
    X_rg_all = np.vstack([X_rg_base, np.array(extra_X_rg)])
    y_rg_all = np.concatenate([y_rg_base, np.array(extra_y_rg)])
else:
    X_rg_all = X_rg_base
    y_rg_all = y_rg_base

print(f"  Augmented Rg samples: {len(y_rg_all)}")

X_rg_tr, X_rg_va, y_rg_tr, y_rg_va = train_test_split(
    X_rg_all, y_rg_all, test_size=0.15, random_state=SEED
)

rg_model_v2 = SpecialistNet(FEAT_DIM, hidden=512, dropout=0.25).to(DEVICE)
rg_model_v2, rg_v2_r2, rg_mu, rg_sig = train_specialist(
    rg_model_v2, X_rg_tr, y_rg_tr, X_rg_va, y_rg_va,
    epochs=500, batch=64, lr=1e-3, patience=50,
    device=DEVICE, tag="Rg-Specialist-v2",
)
torch.save({"state": rg_model_v2.state_dict(), "mu": rg_mu, "sig": rg_sig},
           WORK + "pinn_rg_spec_v2.pth")
print(f"  ✓ Rg specialist v2 saved  (R²={rg_v2_r2:.4f})")


# ─────────────────────────────────────────────────────────────
# SECTION 4: FFV SPECIALIST  (7892 labels, push from 0.85→0.90)
# ─────────────────────────────────────────────────────────────
print("\n[4/8] Training FFV Specialist...")

ffv_mask   = ~np.isnan(Y_raw[:, 1])
X_ffv_all  = X_scaled[ffv_mask]
y_ffv_all  = Y_raw[ffv_mask, 1]

print(f"  FFV samples: {len(y_ffv_all)}  (no augmentation needed)")

X_ffv_tr, X_ffv_va, y_ffv_tr, y_ffv_va = train_test_split(
    X_ffv_all, y_ffv_all, test_size=0.15, random_state=SEED
)

ffv_model = SpecialistNet(FEAT_DIM, hidden=512, dropout=0.20).to(DEVICE)
ffv_model, ffv_r2, ffv_mu, ffv_sig = train_specialist(
    ffv_model, X_ffv_tr, y_ffv_tr, X_ffv_va, y_ffv_va,
    epochs=300, batch=256, lr=1e-3, patience=40,
    device=DEVICE, tag="FFV-Specialist",
)
torch.save({"state": ffv_model.state_dict(), "mu": ffv_mu, "sig": ffv_sig},
           WORK + "pinn_ffv_spec.pth")
print(f"  ✓ FFV specialist saved  (R²={ffv_r2:.4f})")


# ─────────────────────────────────────────────────────────────
# SECTION 5: Density SPECIALIST  (optional boost)
# ─────────────────────────────────────────────────────────────
print("\n[5/8] Training Density Specialist...")

dn_mask   = ~np.isnan(Y_raw[:, 3])
X_dn_base = X_scaled[dn_mask]
y_dn_base = Y_raw[dn_mask, 3]
sm_dn     = train_df["SMILES"].values[dn_mask]

print(f"  Base Density samples: {len(y_dn_base)}")
print("  Running SMILES augmentation (×4)...")

extra_X_dn, extra_y_dn = [], []
for i, (smi, yval) in enumerate(zip(sm_dn, y_dn_base)):
    for v in augment_smiles(smi, n=4):
        try:
            fv = extract_features(v)
            fv = np.nan_to_num(fv, nan=0.0, posinf=0.0, neginf=0.0)
            fv_sc = scaler_X.transform(fv.reshape(1, -1))[0].astype(np.float32)
            extra_X_dn.append(fv_sc)
            extra_y_dn.append(yval)
        except Exception:
            pass
    if (i + 1) % 200 == 0:
        print(f"    Density aug: {i+1}/{len(sm_dn)}")

if extra_X_dn:
    X_dn_all = np.vstack([X_dn_base, np.array(extra_X_dn)])
    y_dn_all = np.concatenate([y_dn_base, np.array(extra_y_dn)])
else:
    X_dn_all = X_dn_base
    y_dn_all = y_dn_base

print(f"  Augmented Density samples: {len(y_dn_all)}")

X_dn_tr, X_dn_va, y_dn_tr, y_dn_va = train_test_split(
    X_dn_all, y_dn_all, test_size=0.15, random_state=SEED
)

dn_model = SpecialistNet(FEAT_DIM, hidden=512, dropout=0.25).to(DEVICE)
dn_model, dn_r2, dn_mu, dn_sig = train_specialist(
    dn_model, X_dn_tr, y_dn_tr, X_dn_va, y_dn_va,
    epochs=500, batch=64, lr=1e-3, patience=50,
    device=DEVICE, tag="Density-Specialist",
)
torch.save({"state": dn_model.state_dict(), "mu": dn_mu, "sig": dn_sig},
           WORK + "pinn_density_spec.pth")
print(f"  ✓ Density specialist saved  (R²={dn_r2:.4f})")


# ─────────────────────────────────────────────────────────────
# SECTION 6: Load Tg specialist from Step 1 & main model preds
# ─────────────────────────────────────────────────────────────
print("\n[6/8] Loading Step 1 specialists...")

# PyTorch 2.6 changed weights_only default to True; our checkpoints
# store numpy scalars (mu/sig) so we must opt out explicitly.
torch.serialization.add_safe_globals([])   # reset any previous additions

# Load Tg specialist
tg_ckpt = torch.load(WORK + "pinn_tg_spec.pth", map_location=DEVICE, weights_only=False)
tg_model = SpecialistNet(FEAT_DIM, hidden=512, dropout=0.30).to(DEVICE)
tg_model.load_state_dict(tg_ckpt["state"])
tg_mu, tg_sig = float(tg_ckpt["mu"]), float(tg_ckpt["sig"])
tg_model.eval()

# Load old Rg specialist (for comparison / fallback)
rg_ckpt_v1 = torch.load(WORK + "pinn_rg_spec.pth", map_location=DEVICE, weights_only=False)
rg_model_v1 = SpecialistNet(FEAT_DIM, hidden=512, dropout=0.25).to(DEVICE)
rg_model_v1.load_state_dict(rg_ckpt_v1["state"])
rg_mu_v1, rg_sig_v1 = float(rg_ckpt_v1["mu"]), float(rg_ckpt_v1["sig"])
rg_model_v1.eval()

# Load main PINN
from types import SimpleNamespace

class MultiTaskPINN_Load(nn.Module):
    """Minimal loader — same arch as Step 1 main model."""
    class _ResBlock(nn.Module):
        def __init__(self, dim, dropout=0.2):
            super().__init__()
            self.block = nn.Sequential(
                nn.LayerNorm(dim), nn.Linear(dim, dim), nn.GELU(),
                nn.Dropout(dropout), nn.Linear(dim, dim),
            )
            self.act = nn.GELU()
        def forward(self, x): return self.act(x + self.block(x))

    def __init__(self, feat_dim, hidden=768, n_targets=5, dropout=0.2):
        super().__init__()
        self.proj = nn.Sequential(
            nn.Linear(feat_dim, hidden * 2), nn.LayerNorm(hidden * 2),
            nn.GELU(), nn.Dropout(dropout), nn.Linear(hidden * 2, hidden),
            nn.LayerNorm(hidden), nn.GELU(),
        )
        self.backbone = nn.Sequential(*[self._ResBlock(hidden, dropout) for _ in range(6)])
        self.heads = nn.ModuleList([
            nn.Sequential(
                nn.Linear(hidden, hidden // 2), nn.GELU(),
                nn.Dropout(dropout * 0.5), nn.Linear(hidden // 2, hidden // 4),
                nn.GELU(), nn.Linear(hidden // 4, 1),
            ) for _ in range(n_targets)
        ])
    def forward(self, x):
        h = self.backbone(self.proj(x))
        return torch.cat([head(h) for head in self.heads], dim=1)

main_ckpt = torch.load(WORK + "pinn_main.pth", map_location=DEVICE, weights_only=False)
main_model = MultiTaskPINN_Load(FEAT_DIM, hidden=768, n_targets=5, dropout=0.2).to(DEVICE)
main_model.load_state_dict(main_ckpt)
main_model.eval()

# Per-target mean/std from Step 1 training data (recompute)
Y_raw_all = train_df[TARGETS].values.astype(np.float32)
t_mean = np.zeros(5, dtype=np.float32)
t_std  = np.ones(5, dtype=np.float32)
for i in range(5):
    mask = ~np.isnan(Y_raw_all[:, i])
    vals = Y_raw_all[mask, i]
    t_mean[i] = vals.mean()
    t_std[i]  = vals.std() + 1e-8

print("  ✓ All Step 1 models loaded")


# ─────────────────────────────────────────────────────────────
# SECTION 7: Test set predictions from ALL specialists
# ─────────────────────────────────────────────────────────────
print("\n[7/8] Generating final test predictions...")

X_test_t = torch.tensor(X_test_sc, device=DEVICE)

def predict_spec(model, mu, sig, X_t):
    model.eval()
    with torch.no_grad():
        return model(X_t).cpu().numpy() * sig + mu

# Main model predictions (all 5 targets, z-space → real)
with torch.no_grad():
    main_sc = main_model(X_test_t).cpu().numpy()
main_preds = {t: main_sc[:, i] * t_std[i] + t_mean[i] for i, t in enumerate(TARGETS)}

# Specialist predictions
tg_preds  = predict_spec(tg_model,     tg_mu,  tg_sig,  X_test_t)
tc_preds  = predict_spec(tc_model,     tc_mu,  tc_sig,  X_test_t)
rg_preds  = predict_spec(rg_model_v2,  rg_mu,  rg_sig,  X_test_t)
ffv_preds = predict_spec(ffv_model,    ffv_mu, ffv_sig, X_test_t)
dn_preds  = predict_spec(dn_model,     dn_mu,  dn_sig,  X_test_t)

# R² values from val sets (Step 1 val approximations for main model)
MAIN_R2 = {
    "Tg":      0.6498,   # from Step 1 output
    "FFV":     0.8483,
    "Tc":      0.5343,
    "Density": 0.8279,
    "Rg":      0.6947,
}
SPEC_R2 = {
    "Tg":      0.9730,   # from Step 1
    "FFV":     ffv_r2,
    "Tc":      tc_r2,
    "Density": dn_r2,
    "Rg":      rg_v2_r2,
}

print(f"\n  Specialist R² summary:")
print(f"  {'Target':<12} {'Main':>8} {'Spec':>8}  {'Winner'}")
print(f"  {'='*40}")
for t in TARGETS:
    m, s = MAIN_R2[t], SPEC_R2[t]
    winner = "SPEC ✓" if s > m else "MAIN"
    print(f"  {t:<12} {m:>8.4f} {s:>8.4f}  {winner}")

# Weighted blend: R²-proportional per target
def blend(main_p, spec_p, r2_main, r2_spec):
    wm = max(0.0, r2_main)
    ws = max(0.0, r2_spec)
    tot = wm + ws + 1e-8
    return (wm * main_p + ws * spec_p) / tot

final_preds = {
    "Tg":      blend(main_preds["Tg"],      tg_preds,  MAIN_R2["Tg"],      SPEC_R2["Tg"]),
    "FFV":     blend(main_preds["FFV"],     ffv_preds,  MAIN_R2["FFV"],     SPEC_R2["FFV"]),
    "Tc":      blend(main_preds["Tc"],      tc_preds,   MAIN_R2["Tc"],      SPEC_R2["Tc"]),
    "Density": blend(main_preds["Density"], dn_preds,   MAIN_R2["Density"], SPEC_R2["Density"]),
    "Rg":      blend(main_preds["Rg"],      rg_preds,   MAIN_R2["Rg"],      SPEC_R2["Rg"]),
}

sub3 = pd.DataFrame({"id": test_df["id"]})
for t in TARGETS:
    sub3[t] = final_preds[t]
sub3.to_csv(WORK + "submission_model3.csv", index=False)
print(f"\n✓ submission_model3.csv (updated) saved")
print(sub3.to_string())


# ─────────────────────────────────────────────────────────────
# SECTION 8: Per-target weighted ensemble across all 3 models
# ─────────────────────────────────────────────────────────────
print("\n[8/8] Final per-target weighted ensemble...")

try:
    sub1 = pd.read_csv(WORK + "submission.csv")
    sub2 = pd.read_csv(WORK + "submission_model2.csv")

    # Approximate per-target R² for M1 and M2 (update if you have exact logs)
    M1_R2 = {"Tg": 0.72, "FFV": 0.80, "Tc": 0.79, "Density": 0.77, "Rg": 0.72}
    M2_R2 = {"Tg": 0.78, "FFV": 0.87, "Tc": 0.84, "Density": 0.85, "Rg": 0.75}
    M3_R2 = SPEC_R2   # use specialist R² as M3's per-target score

    ensemble = pd.DataFrame({"id": sub1["id"]})
    print(f"\n  Final per-target blend weights:")
    print(f"  {'Target':<12} {'w_M1':>8} {'w_M2':>8} {'w_M3':>8}  {'M3 R²':>8}")
    print(f"  {'='*55}")
    for t in TARGETS:
        w1 = max(0.01, M1_R2.get(t, 0.5))
        w2 = max(0.01, M2_R2.get(t, 0.5))
        w3 = max(0.01, M3_R2.get(t, 0.5))
        total = w1 + w2 + w3
        w1n, w2n, w3n = w1/total, w2/total, w3/total
        ensemble[t] = w1n * sub1[t] + w2n * sub2[t] + w3n * sub3[t]
        print(f"  {t:<12} {w1n:>8.3f} {w2n:>8.3f} {w3n:>8.3f}  {M3_R2[t]:>8.4f}")

    ensemble.to_csv(WORK + "submission_ensemble.csv", index=False)
    print(f"\n✓ submission_ensemble.csv (final) saved")
    print(ensemble.to_string())

except FileNotFoundError as e:
    print(f"  ⚠ Models 1/2 not found: {e}")
    print("  Saving Model 3 standalone as ensemble fallback.")
    sub3.to_csv(WORK + "submission_ensemble.csv", index=False)

# ── Final summary ──────────────────────────────────────────
print(f"\n{'='*60}")
print("  STEP 2 COMPLETE — Specialist R² summary:")
print(f"{'='*60}")
expected_avg = np.mean(list(SPEC_R2.values()))
for t in TARGETS:
    flag = "✓" if SPEC_R2[t] >= 0.90 else "→"
    print(f"  {t:<12} {SPEC_R2[t]:>8.4f}  {flag}")
print(f"  {'─'*30}")
print(f"  {'AVERAGE':<12} {expected_avg:>8.4f}")
print(f"\n  Files saved:")
print(f"    {WORK}pinn_tc_spec.pth")
print(f"    {WORK}pinn_rg_spec_v2.pth")
print(f"    {WORK}pinn_ffv_spec.pth")
print(f"    {WORK}pinn_density_spec.pth")
print(f"    {WORK}submission_model3.csv  (updated)")
print(f"    {WORK}submission_ensemble.csv  (final)")
print(f"{'='*60}")

Device: cuda
GPU   : Tesla T4

[1/8] Loading data + cached features from Step 1...
  Feature dim  : 3456
  Train rows   : 9755
    Tg        :   557 labeled
    FFV       :  7892 labeled
    Tc        :  1611 labeled
    Density   :   613 labeled
    Rg        :   614 labeled

[2/8] Training Tc Specialist...
  Base Tc samples: 1611
  Running SMILES augmentation (×4)...
    Tc aug: 400/1611
    Tc aug: 800/1611
    Tc aug: 1200/1611
    Tc aug: 1600/1611
  Augmented Tc samples: 8052

  [Tc-Specialist]  train=6844  val=1208  params=6,337,537
    ep=   1  R²=0.8735 [best=0.8735]  MAE=0.0203 ✓
    ep=  50  R²=0.9821 [best=0.9821]  MAE=0.0051 ✓
    ep= 100  R²=0.9833 [best=0.9866]  MAE=0.0054
    ep= 150  R²=0.9846 [best=0.9868]  MAE=0.0060
    ⚡ Early stop ep=170  best R²=0.9868
  ✓ Tc specialist saved  (R²=0.9868)

[3/8] Training Rg Specialist v2 (with augmentation)...
  Base Rg samples: 614
  Running SMILES augmentation (×5)...
    Rg aug: 200/614
    Rg aug: 400/614
    Rg aug: 600/614


In [11]:
# ================================================================
#  MODEL 3  —  STEP 3: FFV Specialist Fix + Final Ensemble
#  Run after Step 2 completes.
#
#  Only fixes FFV (specialist was 0.8132 < main 0.8483).
#  Root cause: narrow target range (std=0.030) + aggressive
#  LR restarts caused the model to "forget" fine structure.
#  Fix: wider hidden (768), no restarts (CosineAnnealingLR),
#       lower dropout (0.15), lower LR (5e-4), more epochs.
# ================================================================

import warnings
import numpy as np
import pandas as pd
from sklearn.preprocessing import RobustScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

warnings.filterwarnings("ignore")
SEED    = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
DEVICE  = torch.device("cuda" if torch.cuda.is_available() else "cpu")
WORK    = "/kaggle/working/"
BASE    = "/kaggle/input/competitions/neurips-open-polymer-prediction-2025/"
TARGETS = ["Tg", "FFV", "Tc", "Density", "Rg"]

print(f"Device: {DEVICE}")


# ── Reload data + features ────────────────────────────────────
print("\n[1/4] Loading data + cached features...")

train_raw = pd.read_csv(BASE + "train.csv")
test_df   = pd.read_csv(BASE + "test.csv")

d1 = pd.read_csv(BASE + "train_supplement/dataset1.csv").rename(columns={"TC_mean": "Tc"})
d3 = pd.read_csv(BASE + "train_supplement/dataset3.csv")
d4 = pd.read_csv(BASE + "train_supplement/dataset4.csv")

def supp_rows(df, col):
    rows = []
    for _, r in df.iterrows():
        row = {"id": -1, "SMILES": r["SMILES"]}
        for t in TARGETS:
            row[t] = r[col] if t == col else np.nan
        rows.append(row)
    return rows

supp_df  = pd.DataFrame(supp_rows(d1, "Tc") + supp_rows(d3, "Tg") + supp_rows(d4, "FFV"))
train_df = pd.concat([train_raw, supp_df], ignore_index=True)

X_all_raw  = np.load(WORK + "feats_train_v2.npy")
X_test_raw = np.load(WORK + "feats_test_v2.npy")
X_all_raw  = np.nan_to_num(X_all_raw,  nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)
X_test_raw = np.nan_to_num(X_test_raw, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)

FEAT_DIM = X_all_raw.shape[1]
scaler_X  = RobustScaler()
X_scaled  = scaler_X.fit_transform(X_all_raw).astype(np.float32)
X_test_sc = scaler_X.transform(X_test_raw).astype(np.float32)

Y_raw = train_df[TARGETS].values.astype(np.float32)

ffv_idx  = 1
ffv_mask = ~np.isnan(Y_raw[:, ffv_idx])
X_ffv    = X_scaled[ffv_mask]
y_ffv    = Y_raw[ffv_mask, ffv_idx]

print(f"  Feature dim : {FEAT_DIM}")
print(f"  FFV samples : {len(y_ffv)}  (mean={y_ffv.mean():.4f}  std={y_ffv.std():.4f})")

X_ffv_tr, X_ffv_va, y_ffv_tr, y_ffv_va = train_test_split(
    X_ffv, y_ffv, test_size=0.15, random_state=SEED
)


# ── Architecture ──────────────────────────────────────────────

class ResBlock(nn.Module):
    def __init__(self, dim, dropout=0.15):
        super().__init__()
        self.block = nn.Sequential(
            nn.LayerNorm(dim),
            nn.Linear(dim, dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(dim, dim),
        )
        self.act = nn.GELU()
    def forward(self, x):
        return self.act(x + self.block(x))


class FFVNet(nn.Module):
    """
    Wider network for FFV's narrow-range regression.
    hidden=768 (vs 512 in other specialists) to increase
    representational capacity for fine-grained differences.
    Low dropout=0.15 since FFV has 7892 samples (low overfitting risk).
    Extra residual blocks to capture subtle structural signals.
    """
    def __init__(self, feat_dim, hidden=768, dropout=0.15):
        super().__init__()
        self.enc = nn.Sequential(
            nn.Linear(feat_dim, hidden * 2),
            nn.LayerNorm(hidden * 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden * 2, hidden),
            nn.LayerNorm(hidden),
            nn.GELU(),
        )
        self.res = nn.Sequential(
            ResBlock(hidden, dropout),
            ResBlock(hidden, dropout),
            ResBlock(hidden, dropout),
            ResBlock(hidden, dropout),
            ResBlock(hidden, dropout),   # extra block vs other specialists
            ResBlock(hidden, dropout),
        )
        self.head = nn.Sequential(
            nn.Linear(hidden, hidden // 2),
            nn.GELU(),
            nn.Dropout(dropout * 0.5),
            nn.Linear(hidden // 2, hidden // 4),
            nn.GELU(),
            nn.Linear(hidden // 4, 1),
        )
    def forward(self, x):
        return self.head(self.res(self.enc(x))).squeeze(-1)


# ── Training ──────────────────────────────────────────────────
print("\n[2/4] Training FFV Specialist v2...")

mu_ffv  = y_ffv_tr.mean()
sig_ffv = y_ffv_tr.std() + 1e-8
y_tr_n  = ((y_ffv_tr - mu_ffv) / sig_ffv).astype(np.float32)
y_va_n  = ((y_ffv_va - mu_ffv) / sig_ffv).astype(np.float32)

ds  = TensorDataset(torch.tensor(X_ffv_tr), torch.tensor(y_tr_n))
ldr = DataLoader(ds, batch_size=256, shuffle=True, drop_last=False)

ffv_v2 = FFVNet(FEAT_DIM, hidden=768, dropout=0.15).to(DEVICE)
n_params = sum(p.numel() for p in ffv_v2.parameters())
print(f"  params={n_params:,}  train={len(y_ffv_tr)}  val={len(y_ffv_va)}")

# No warm restarts — smooth cosine decay prevents the "forgetting" issue
opt = torch.optim.AdamW(ffv_v2.parameters(), lr=5e-4, weight_decay=1e-3)
sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=500, eta_min=5e-6)

X_va_t = torch.tensor(X_ffv_va, device=DEVICE)

best_r2, best_state, wait, patience = -1.0, None, 0, 60

for ep in range(1, 501):
    ffv_v2.train()
    for xb, yb in ldr:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        opt.zero_grad()
        F.mse_loss(ffv_v2(xb), yb).backward()
        nn.utils.clip_grad_norm_(ffv_v2.parameters(), 1.0)
        opt.step()
    sch.step()

    if ep % 5 == 0 or ep == 1:
        ffv_v2.eval()
        with torch.no_grad():
            pn = ffv_v2(X_va_t).cpu().numpy()
        p_real = pn * sig_ffv + mu_ffv
        r2  = r2_score(y_ffv_va, p_real)
        mae = mean_absolute_error(y_ffv_va, p_real)

        if r2 > best_r2:
            best_r2    = r2
            best_state = {k: v.clone() for k, v in ffv_v2.state_dict().items()}
            wait, marker = 0, " ✓"
        else:
            wait += 1
            marker = ""

        if ep % 50 == 0 or ep == 1:
            print(f"    ep={ep:>4}  R²={r2:.4f} [best={best_r2:.4f}]  MAE={mae:.6f}{marker}")

        if wait * 5 >= patience:
            print(f"    ⚡ Early stop ep={ep}  best R²={best_r2:.4f}")
            break

ffv_v2.load_state_dict(best_state)
torch.save({"state": ffv_v2.state_dict(), "mu": mu_ffv, "sig": sig_ffv},
           WORK + "pinn_ffv_spec_v2.pth")
print(f"  ✓ FFV specialist v2 saved  (R²={best_r2:.4f})")
ffv_v2_r2 = best_r2


# ── Rebuild test predictions + final ensemble ─────────────────
print("\n[3/4] Rebuilding test predictions with fixed FFV...")

# Decide which FFV model to use
ffv_v1_r2 = 0.8132   # from Step 2
main_ffv_r2 = 0.8483  # from Step 1 main model

best_ffv_r2 = max(ffv_v2_r2, ffv_v1_r2, main_ffv_r2)
print(f"  FFV R² comparison:")
print(f"    Main model     : {main_ffv_r2:.4f}")
print(f"    Specialist v1  : {ffv_v1_r2:.4f}")
print(f"    Specialist v2  : {ffv_v2_r2:.4f}  ← {'BEST ✓' if ffv_v2_r2 == best_ffv_r2 else ''}")

# Load all other specialists
class SpecialistNet(nn.Module):
    def __init__(self, feat_dim, hidden=512, dropout=0.30):
        super().__init__()
        self.enc = nn.Sequential(
            nn.Linear(feat_dim, hidden * 2), nn.LayerNorm(hidden * 2),
            nn.GELU(), nn.Dropout(dropout),
            nn.Linear(hidden * 2, hidden), nn.LayerNorm(hidden), nn.GELU(),
        )
        self.res = nn.Sequential(*[
            type('RB', (nn.Module,), {
                '__init__': lambda s, d=hidden, dr=dropout: (
                    super(type(s), s).__init__() or
                    setattr(s, 'block', nn.Sequential(
                        nn.LayerNorm(d), nn.Linear(d, d), nn.GELU(),
                        nn.Dropout(dr), nn.Linear(d, d)
                    )) or setattr(s, 'act', nn.GELU())
                ),
                'forward': lambda s, x: s.act(x + s.block(x))
            })() for _ in range(4)
        ])
        self.head = nn.Sequential(
            nn.Linear(hidden, hidden // 2), nn.GELU(),
            nn.Dropout(dropout * 0.5), nn.Linear(hidden // 2, hidden // 4),
            nn.GELU(), nn.Linear(hidden // 4, 1),
        )
    def forward(self, x):
        return self.head(self.res(self.enc(x))).squeeze(-1)


# Cleaner: reuse ResBlock from top of this file and define SpecialistNet properly
class SpecialistNetClean(nn.Module):
    def __init__(self, feat_dim, hidden=512, dropout=0.30):
        super().__init__()
        self.enc = nn.Sequential(
            nn.Linear(feat_dim, hidden * 2),
            nn.LayerNorm(hidden * 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden * 2, hidden),
            nn.LayerNorm(hidden),
            nn.GELU(),
        )
        self.res = nn.Sequential(
            ResBlock(hidden, dropout),
            ResBlock(hidden, dropout),
            ResBlock(hidden, dropout),
            ResBlock(hidden, dropout),
        )
        self.head = nn.Sequential(
            nn.Linear(hidden, hidden // 2),
            nn.GELU(),
            nn.Dropout(dropout * 0.5),
            nn.Linear(hidden // 2, hidden // 4),
            nn.GELU(),
            nn.Linear(hidden // 4, 1),
        )
    def forward(self, x):
        return self.head(self.res(self.enc(x))).squeeze(-1)


def load_spec(path, hidden=512, dropout=0.30):
    ckpt = torch.load(path, map_location=DEVICE, weights_only=False)
    m = SpecialistNetClean(FEAT_DIM, hidden=hidden, dropout=dropout).to(DEVICE)
    m.load_state_dict(ckpt["state"])
    m.eval()
    return m, float(ckpt["mu"]), float(ckpt["sig"])


def predict_spec(model, mu, sig, X_t):
    model.eval()
    with torch.no_grad():
        return model(X_t).cpu().numpy() * sig + mu


X_test_t = torch.tensor(X_test_sc, device=DEVICE)

tg_m,  tg_mu,  tg_sig  = load_spec(WORK + "pinn_tg_spec.pth",      dropout=0.30)
tc_m,  tc_mu,  tc_sig  = load_spec(WORK + "pinn_tc_spec.pth",       dropout=0.25)
rg_m,  rg_mu,  rg_sig  = load_spec(WORK + "pinn_rg_spec_v2.pth",    dropout=0.25)
dn_m,  dn_mu,  dn_sig  = load_spec(WORK + "pinn_density_spec.pth",  dropout=0.25)

# Load main model for FFV fallback
class MainPINN(nn.Module):
    def __init__(self, feat_dim, hidden=768, n=5, dropout=0.2):
        super().__init__()
        self.proj = nn.Sequential(
            nn.Linear(feat_dim, hidden*2), nn.LayerNorm(hidden*2),
            nn.GELU(), nn.Dropout(dropout),
            nn.Linear(hidden*2, hidden), nn.LayerNorm(hidden), nn.GELU(),
        )
        self.backbone = nn.Sequential(*[ResBlock(hidden, dropout) for _ in range(6)])
        self.heads = nn.ModuleList([
            nn.Sequential(
                nn.Linear(hidden, hidden//2), nn.GELU(), nn.Dropout(dropout*0.5),
                nn.Linear(hidden//2, hidden//4), nn.GELU(), nn.Linear(hidden//4, 1),
            ) for _ in range(n)
        ])
    def forward(self, x):
        h = self.backbone(self.proj(x))
        return torch.cat([hd(h) for hd in self.heads], dim=1)

main_ckpt  = torch.load(WORK + "pinn_main.pth", map_location=DEVICE, weights_only=False)
main_model = MainPINN(FEAT_DIM).to(DEVICE)
main_model.load_state_dict(main_ckpt)
main_model.eval()

# Recompute t_mean / t_std
t_mean = np.zeros(5, dtype=np.float32)
t_std  = np.ones(5,  dtype=np.float32)
Y_all  = train_df[TARGETS].values.astype(np.float32)
for i in range(5):
    v = Y_all[~np.isnan(Y_all[:, i]), i]
    t_mean[i], t_std[i] = v.mean(), v.std() + 1e-8

with torch.no_grad():
    main_sc = main_model(X_test_t).cpu().numpy()
main_ffv_pred = main_sc[:, 1] * t_std[1] + t_mean[1]

# Generate all predictions
tg_pred  = predict_spec(tg_m, tg_mu,  tg_sig,  X_test_t)
tc_pred  = predict_spec(tc_m, tc_mu,  tc_sig,  X_test_t)
rg_pred  = predict_spec(rg_m, rg_mu,  rg_sig,  X_test_t)
dn_pred  = predict_spec(dn_m, dn_mu,  dn_sig,  X_test_t)
ffv_pred = predict_spec(ffv_v2, mu_ffv, sig_ffv, X_test_t)

# R² per target for M3 (best available per target)
M3_R2 = {
    "Tg":      0.9730,
    "FFV":     best_ffv_r2,
    "Tc":      0.9868,
    "Density": 0.9972,
    "Rg":      0.9830,
}

def blend(p1, p2, r1, r2):
    w1, w2 = max(0.0, r1), max(0.0, r2)
    return (w1 * p1 + w2 * p2) / (w1 + w2 + 1e-8)

# For FFV: blend main model + best specialist
ffv_final = blend(main_ffv_pred, ffv_pred, main_ffv_r2, best_ffv_r2)

final_preds = {
    "Tg":      tg_pred,
    "FFV":     ffv_final,
    "Tc":      tc_pred,
    "Density": dn_pred,
    "Rg":      rg_pred,
}

sub3 = pd.DataFrame({"id": test_df["id"]})
for t in TARGETS:
    sub3[t] = final_preds[t]
sub3.to_csv(WORK + "submission_model3.csv", index=False)
print(f"✓ submission_model3.csv rebuilt")
print(sub3.to_string())


# ── Final 3-model ensemble ────────────────────────────────────
print("\n[4/4] Building final 3-model ensemble...")

try:
    sub1 = pd.read_csv(WORK + "submission.csv")
    sub2 = pd.read_csv(WORK + "submission_model2.csv")

    M1_R2 = {"Tg": 0.72, "FFV": 0.80, "Tc": 0.79, "Density": 0.77, "Rg": 0.72}
    M2_R2 = {"Tg": 0.78, "FFV": 0.87, "Tc": 0.84, "Density": 0.85, "Rg": 0.75}

    ensemble = pd.DataFrame({"id": sub1["id"]})
    print(f"\n  {'Target':<12} {'w_M1':>7} {'w_M2':>7} {'w_M3':>7}  "
          f"{'M1_R²':>7} {'M2_R²':>7} {'M3_R²':>7}")
    print(f"  {'='*60}")
    for t in TARGETS:
        w1 = max(0.01, M1_R2[t])
        w2 = max(0.01, M2_R2[t])
        w3 = max(0.01, M3_R2[t])
        tot = w1 + w2 + w3
        w1n, w2n, w3n = w1/tot, w2/tot, w3/tot
        ensemble[t] = w1n*sub1[t] + w2n*sub2[t] + w3n*sub3[t]
        print(f"  {t:<12} {w1n:>7.3f} {w2n:>7.3f} {w3n:>7.3f}  "
              f"{M1_R2[t]:>7.4f} {M2_R2[t]:>7.4f} {M3_R2[t]:>7.4f}")

    ensemble.to_csv(WORK + "submission_ensemble.csv", index=False)
    print(f"\n✓ submission_ensemble.csv (final) saved")
    print(ensemble.to_string())

except FileNotFoundError as e:
    print(f"  ⚠ {e} — saving Model 3 as ensemble")
    sub3.to_csv(WORK + "submission_ensemble.csv", index=False)

# ── Summary ───────────────────────────────────────────────────
avg_m3 = np.mean(list(M3_R2.values()))
print(f"\n{'='*60}")
print(f"  FINAL Model 3 R² per target:")
print(f"{'='*60}")
for t in TARGETS:
    flag = "✓" if M3_R2[t] >= 0.90 else "→"
    print(f"  {t:<12} {M3_R2[t]:>8.4f}  {flag}")
print(f"  {'─'*30}")
print(f"  {'AVERAGE':<12} {avg_m3:>8.4f}")
print(f"\n  Submit: /kaggle/working/submission_ensemble.csv")
print(f"{'='*60}")

Device: cuda

[1/4] Loading data + cached features...
  Feature dim : 3456
  FFV samples : 7892  (mean=0.3670  std=0.0291)

[2/4] Training FFV Specialist v2...
  params=13,960,705  train=6708  val=1184
    ep=   1  R²=0.5960 [best=0.5960]  MAE=0.011693 ✓
    ep=  50  R²=0.7810 [best=0.7825]  MAE=0.006503
    ep= 100  R²=0.7237 [best=0.7856]  MAE=0.007554
    ⚡ Early stop ep=125  best R²=0.7856
  ✓ FFV specialist v2 saved  (R²=0.7856)

[3/4] Rebuilding test predictions with fixed FFV...
  FFV R² comparison:
    Main model     : 0.8483
    Specialist v1  : 0.8132
    Specialist v2  : 0.7856  ← 
✓ submission_model3.csv rebuilt
           id          Tg       FFV        Tc   Density         Rg
0  1109053969  151.819244  0.374724  0.187450  1.140589  23.516932
1  1422188626  158.340179  0.374005  0.309692  1.138498  22.466808
2  2032016830   52.943729  0.351600  0.247384  1.118534  20.618256

[4/4] Building final 3-model ensemble...

  Target          w_M1    w_M2    w_M3    M1_R²   M2_R²  

In [12]:
# ================================================================
#  MODEL 3  —  STEP 4: FFV XGBoost Specialist + Final Ensemble
#
#  FFV std=0.029 (extremely narrow range).
#  Neural networks produce tiny gradients → consistently underfit.
#  XGBoost handles narrow-range tabular regression much better:
#    - Splits on feature thresholds, not gradient magnitude
#    - No LR scheduling or batch-size sensitivity
#    - Naturally captures the non-linear FFV fingerprint patterns
#  Target: FFV R² > 0.88
# ================================================================

import warnings
import numpy as np
import pandas as pd
from sklearn.preprocessing import RobustScaler
from sklearn.model_selection import train_test_split, KFold
from sklearn.metrics import r2_score, mean_absolute_error
import torch
import torch.nn as nn
import xgboost as xgb

warnings.filterwarnings("ignore")
SEED    = 42
np.random.seed(SEED)
DEVICE  = torch.device("cuda" if torch.cuda.is_available() else "cpu")
WORK    = "/kaggle/working/"
BASE    = "/kaggle/input/competitions/neurips-open-polymer-prediction-2025/"
TARGETS = ["Tg", "FFV", "Tc", "Density", "Rg"]

print(f"Device : {DEVICE}")
print(f"XGBoost: {xgb.__version__}")


# ── Load data + features ──────────────────────────────────────
print("\n[1/4] Loading cached features...")

train_raw = pd.read_csv(BASE + "train.csv")
test_df   = pd.read_csv(BASE + "test.csv")

d1 = pd.read_csv(BASE + "train_supplement/dataset1.csv").rename(columns={"TC_mean": "Tc"})
d3 = pd.read_csv(BASE + "train_supplement/dataset3.csv")
d4 = pd.read_csv(BASE + "train_supplement/dataset4.csv")

def supp_rows(df, col):
    rows = []
    for _, r in df.iterrows():
        row = {"id": -1, "SMILES": r["SMILES"]}
        for t in TARGETS:
            row[t] = r[col] if t == col else np.nan
        rows.append(row)
    return rows

supp_df  = pd.DataFrame(supp_rows(d1,"Tc") + supp_rows(d3,"Tg") + supp_rows(d4,"FFV"))
train_df = pd.concat([train_raw, supp_df], ignore_index=True)

# Raw (unscaled) features — XGBoost handles its own scaling internally
X_all_raw  = np.load(WORK + "feats_train_v2.npy")
X_test_raw = np.load(WORK + "feats_test_v2.npy")
X_all_raw  = np.nan_to_num(X_all_raw,  nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)
X_test_raw = np.nan_to_num(X_test_raw, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)

FEAT_DIM = X_all_raw.shape[1]
Y_raw    = train_df[TARGETS].values.astype(np.float32)

ffv_mask = ~np.isnan(Y_raw[:, 1])
X_ffv    = X_all_raw[ffv_mask]          # raw features for XGBoost
y_ffv    = Y_raw[ffv_mask, 1]

print(f"  Feature dim : {FEAT_DIM}")
print(f"  FFV samples : {len(y_ffv)}  "
      f"(mean={y_ffv.mean():.4f}  std={y_ffv.std():.4f}  "
      f"min={y_ffv.min():.4f}  max={y_ffv.max():.4f})")

X_tr, X_va, y_tr, y_va = train_test_split(
    X_ffv, y_ffv, test_size=0.15, random_state=SEED
)
print(f"  Train: {len(y_tr)}  Val: {len(y_va)}")


# ── XGBoost with GPU ─────────────────────────────────────────
print("\n[2/4] Training FFV XGBoost specialist...")

# Use GPU tree method if available
tree_method = "hist"
device_str  = "cuda" if DEVICE.type == "cuda" else "cpu"

dtrain = xgb.DMatrix(X_tr, label=y_tr)
dval   = xgb.DMatrix(X_va, label=y_va)
dtest  = xgb.DMatrix(X_test_raw)

params = {
    "objective":        "reg:squarederror",
    "eval_metric":      "rmse",
    "tree_method":      tree_method,
    "device":           device_str,
    "max_depth":        7,          # deep enough to capture complex fingerprint combos
    "learning_rate":    0.02,       # slow enough for narrow-range regression
    "n_estimators":     3000,
    "subsample":        0.8,
    "colsample_bytree": 0.4,        # low — 3456 features, want feature diversity
    "colsample_bylevel":0.6,
    "min_child_weight": 5,          # prevents overfitting to small groups
    "reg_alpha":        0.1,        # L1
    "reg_lambda":       1.0,        # L2
    "gamma":            0.05,
    "seed":             SEED,
}

print(f"  tree_method={tree_method}  device={device_str}")
print(f"  Training up to 3000 rounds with early stopping (50 rounds)...")

evals_result = {}
ffv_xgb = xgb.train(
    params,
    dtrain,
    num_boost_round=3000,
    evals=[(dtrain, "train"), (dval, "val")],
    early_stopping_rounds=50,
    evals_result=evals_result,
    verbose_eval=100,
)

# Evaluate
ffv_xgb_pred_va = ffv_xgb.predict(dval)
xgb_r2  = r2_score(y_va, ffv_xgb_pred_va)
xgb_mae = mean_absolute_error(y_va, ffv_xgb_pred_va)
print(f"\n  XGBoost val  →  R²={xgb_r2:.4f}  MAE={xgb_mae:.6f}")

ffv_xgb.save_model(WORK + "ffv_xgb.json")
print(f"  ✓ ffv_xgb.json saved")


# ── 5-fold CV ensemble for extra stability ────────────────────
print("\n  Running 5-fold CV to build a stronger FFV predictor...")

kf = KFold(n_splits=5, shuffle=True, random_state=SEED)
oof_preds  = np.zeros(len(y_ffv))
test_preds = np.zeros((5, len(X_test_raw)))
fold_r2s   = []

for fold, (tr_idx, va_idx) in enumerate(kf.split(X_ffv)):
    Xf_tr, Xf_va = X_ffv[tr_idx], X_ffv[va_idx]
    yf_tr, yf_va = y_ffv[tr_idx], y_ffv[va_idx]

    df_tr = xgb.DMatrix(Xf_tr, label=yf_tr)
    df_va = xgb.DMatrix(Xf_va, label=yf_va)
    df_te = xgb.DMatrix(X_test_raw)

    m = xgb.train(
        params,
        df_tr,
        num_boost_round=ffv_xgb.best_iteration + 50,  # use best iteration from above
        evals=[(df_va, "val")],
        early_stopping_rounds=50,
        verbose_eval=False,
    )
    oof_preds[va_idx]  = m.predict(df_va)
    test_preds[fold]   = m.predict(df_te)
    fold_r2 = r2_score(yf_va, oof_preds[va_idx])
    fold_r2s.append(fold_r2)
    print(f"    Fold {fold+1}  R²={fold_r2:.4f}")

oof_r2 = r2_score(y_ffv, oof_preds)
cv_test_pred = test_preds.mean(axis=0)   # average across 5 folds

print(f"\n  OOF R²    : {oof_r2:.4f}")
print(f"  Fold R²s  : {[f'{r:.4f}' for r in fold_r2s]}")
print(f"  Best FFV R² so far — XGBoost OOF: {oof_r2:.4f}")


# ── Pick best FFV prediction ──────────────────────────────────
print("\n[3/4] Selecting best FFV source + rebuilding submission...")

# Load main model FFV prediction from current sub3
sub3_prev = pd.read_csv(WORK + "submission_model3.csv")
main_ffv_pred = sub3_prev["FFV"].values   # already the main-model blended FFV

# Compare all sources
main_r2  = 0.8483   # main model val R² (Step 1)
xgb_single_r2 = xgb_r2
xgb_cv_r2     = oof_r2  # OOF is the most honest estimate

print(f"  FFV sources:")
print(f"    Main PINN model     : {main_r2:.4f}")
print(f"    XGBoost (single)    : {xgb_single_r2:.4f}")
print(f"    XGBoost (5-fold CV) : {xgb_cv_r2:.4f}  ← OOF estimate")

best_xgb_r2  = xgb_cv_r2

# Blend main model + XGBoost CV prediction (R²-weighted)
wm  = max(0.0, main_r2)
wx  = max(0.0, best_xgb_r2)
tot = wm + wx + 1e-8
ffv_blended  = (wm * main_ffv_pred + wx * cv_test_pred) / tot
ffv_final_r2 = (wm * main_r2 + wx * best_xgb_r2) / tot  # approximate blended R²

print(f"\n  Blend weights: main={wm/tot:.3f}  xgb={wx/tot:.3f}")
print(f"  Estimated blended FFV R² ≈ {ffv_final_r2:.4f}")


# Rebuild Model 3 submission with blended FFV
def load_spec_pred(path, hidden, dropout, X_t, device):
    class RB(nn.Module):
        def __init__(self, d, dr):
            super().__init__()
            self.block = nn.Sequential(
                nn.LayerNorm(d), nn.Linear(d,d), nn.GELU(), nn.Dropout(dr), nn.Linear(d,d)
            )
            self.act = nn.GELU()
        def forward(self, x): return self.act(x + self.block(x))

    class SN(nn.Module):
        def __init__(self, fd, h, dr):
            super().__init__()
            self.enc = nn.Sequential(
                nn.Linear(fd,h*2), nn.LayerNorm(h*2), nn.GELU(), nn.Dropout(dr),
                nn.Linear(h*2,h),  nn.LayerNorm(h),   nn.GELU(),
            )
            self.res  = nn.Sequential(RB(h,dr), RB(h,dr), RB(h,dr), RB(h,dr))
            self.head = nn.Sequential(
                nn.Linear(h,h//2), nn.GELU(), nn.Dropout(dr*0.5),
                nn.Linear(h//2,h//4), nn.GELU(), nn.Linear(h//4,1),
            )
        def forward(self, x): return self.head(self.res(self.enc(x))).squeeze(-1)

    ckpt = torch.load(path, map_location=device, weights_only=False)
    m = SN(FEAT_DIM, hidden, dropout).to(device)
    m.load_state_dict(ckpt["state"])
    m.eval()
    mu, sig = float(ckpt["mu"]), float(ckpt["sig"])
    with torch.no_grad():
        pred = m(X_t).cpu().numpy() * sig + mu
    return pred

# Load scaled features for neural specialists
scaler_X  = RobustScaler()
X_scaled  = scaler_X.fit_transform(X_all_raw).astype(np.float32)
X_test_sc = scaler_X.transform(X_test_raw).astype(np.float32)
X_test_t  = torch.tensor(X_test_sc, device=DEVICE)

tg_pred = load_spec_pred(WORK+"pinn_tg_spec.pth",     512, 0.30, X_test_t, DEVICE)
tc_pred = load_spec_pred(WORK+"pinn_tc_spec.pth",     512, 0.25, X_test_t, DEVICE)
rg_pred = load_spec_pred(WORK+"pinn_rg_spec_v2.pth",  512, 0.25, X_test_t, DEVICE)
dn_pred = load_spec_pred(WORK+"pinn_density_spec.pth",512, 0.25, X_test_t, DEVICE)

sub3 = pd.DataFrame({"id": test_df["id"]})
sub3["Tg"]      = tg_pred
sub3["FFV"]     = ffv_blended
sub3["Tc"]      = tc_pred
sub3["Density"] = dn_pred
sub3["Rg"]      = rg_pred

sub3.to_csv(WORK + "submission_model3.csv", index=False)
print(f"\n✓ submission_model3.csv rebuilt")
print(sub3.to_string())


# ── Final 3-model ensemble ────────────────────────────────────
print("\n[4/4] Final 3-model per-target ensemble...")

M3_R2 = {
    "Tg":      0.9730,
    "FFV":     ffv_final_r2,   # blended estimate
    "Tc":      0.9868,
    "Density": 0.9972,
    "Rg":      0.9830,
}
M1_R2 = {"Tg": 0.72, "FFV": 0.80, "Tc": 0.79, "Density": 0.77, "Rg": 0.72}
M2_R2 = {"Tg": 0.78, "FFV": 0.87, "Tc": 0.84, "Density": 0.85, "Rg": 0.75}

try:
    sub1 = pd.read_csv(WORK + "submission.csv")
    sub2 = pd.read_csv(WORK + "submission_model2.csv")

    ensemble = pd.DataFrame({"id": sub1["id"]})
    print(f"\n  {'Target':<12} {'w_M1':>7} {'w_M2':>7} {'w_M3':>7}  "
          f"{'M3 R²':>8}")
    print(f"  {'='*50}")
    for t in TARGETS:
        w1, w2, w3 = max(0.01,M1_R2[t]), max(0.01,M2_R2[t]), max(0.01,M3_R2[t])
        tot = w1+w2+w3
        w1n,w2n,w3n = w1/tot, w2/tot, w3/tot
        ensemble[t] = w1n*sub1[t] + w2n*sub2[t] + w3n*sub3[t]
        flag = "✓" if M3_R2[t] >= 0.90 else "→"
        print(f"  {t:<12} {w1n:>7.3f} {w2n:>7.3f} {w3n:>7.3f}  {M3_R2[t]:>8.4f} {flag}")

    ensemble.to_csv(WORK + "submission_ensemble.csv", index=False)
    print(f"\n✓ submission_ensemble.csv saved")
    print(ensemble.to_string())

except FileNotFoundError as e:
    print(f"  ⚠ {e} — using Model 3 standalone")
    sub3.to_csv(WORK + "submission_ensemble.csv", index=False)

avg_m3 = np.mean(list(M3_R2.values()))
print(f"\n{'='*60}")
print(f"  FINAL Model 3 R² summary:")
print(f"{'='*60}")
for t in TARGETS:
    flag = "✓" if M3_R2[t] >= 0.90 else "→"
    print(f"  {t:<12} {M3_R2[t]:>8.4f}  {flag}")
print(f"  {'─'*30}")
print(f"  {'AVERAGE':<12} {avg_m3:>8.4f}")
print(f"\n  ➤ Submit: /kaggle/working/submission_ensemble.csv")
print(f"{'='*60}")

Device : cuda
XGBoost: 3.1.3

[1/4] Loading cached features...
  Feature dim : 3456
  FFV samples : 7892  (mean=0.3670  std=0.0291  min=0.2270  max=0.7771)
  Train: 6708  Val: 1184

[2/4] Training FFV XGBoost specialist...
  tree_method=hist  device=cuda
  Training up to 3000 rounds with early stopping (50 rounds)...
[0]	train-rmse:0.02873	val-rmse:0.02940
[100]	train-rmse:0.01939	val-rmse:0.02004
[200]	train-rmse:0.01850	val-rmse:0.01899
[300]	train-rmse:0.01842	val-rmse:0.01886
[400]	train-rmse:0.01837	val-rmse:0.01875
[500]	train-rmse:0.01832	val-rmse:0.01868
[600]	train-rmse:0.01830	val-rmse:0.01865
[601]	train-rmse:0.01830	val-rmse:0.01865

  XGBoost val  →  R²=0.6044  MAE=0.012000
  ✓ ffv_xgb.json saved

  Running 5-fold CV to build a stronger FFV predictor...
    Fold 1  R²=0.5904
    Fold 2  R²=0.6025
    Fold 3  R²=0.5795
    Fold 4  R²=0.5603
    Fold 5  R²=0.5220

  OOF R²    : 0.5708
  Fold R²s  : ['0.5904', '0.6025', '0.5795', '0.5603', '0.5220']
  Best FFV R² so far — XGB

In [13]:
# ================================================================
#  MODEL 3  —  STEP 5: Final Ensemble (FFV from main PINN only)
#
#  FFV plateau analysis:
#    train-rmse=0.01830  val-rmse=0.01865  (stopped improving at round 600)
#    This is the irreducible noise floor for FFV with these features.
#    Main PINN R²=0.8483 is the ceiling — stop trying to beat it.
#
#  Fix: Use main PINN FFV directly for M3 (R²=0.8483).
#       All other targets keep their specialists (0.97–0.99).
# ================================================================

import warnings
import numpy as np
import pandas as pd
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import r2_score
import torch
import torch.nn as nn

warnings.filterwarnings("ignore")
SEED    = 42
np.random.seed(SEED)
DEVICE  = torch.device("cuda" if torch.cuda.is_available() else "cpu")
WORK    = "/kaggle/working/"
BASE    = "/kaggle/input/competitions/neurips-open-polymer-prediction-2025/"
TARGETS = ["Tg", "FFV", "Tc", "Density", "Rg"]

print(f"Device : {DEVICE}")


# ── Load data + features ──────────────────────────────────────
print("\n[1/3] Loading data + features...")

train_raw = pd.read_csv(BASE + "train.csv")
test_df   = pd.read_csv(BASE + "test.csv")

d1 = pd.read_csv(BASE + "train_supplement/dataset1.csv").rename(columns={"TC_mean": "Tc"})
d3 = pd.read_csv(BASE + "train_supplement/dataset3.csv")
d4 = pd.read_csv(BASE + "train_supplement/dataset4.csv")

def supp_rows(df, col):
    rows = []
    for _, r in df.iterrows():
        row = {"id": -1, "SMILES": r["SMILES"]}
        for t in TARGETS:
            row[t] = r[col] if t == col else np.nan
        rows.append(row)
    return rows

supp_df  = pd.DataFrame(supp_rows(d1,"Tc") + supp_rows(d3,"Tg") + supp_rows(d4,"FFV"))
train_df = pd.concat([train_raw, supp_df], ignore_index=True)

X_all_raw  = np.nan_to_num(np.load(WORK+"feats_train_v2.npy"), nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)
X_test_raw = np.nan_to_num(np.load(WORK+"feats_test_v2.npy"),  nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)
FEAT_DIM   = X_all_raw.shape[1]

scaler_X  = RobustScaler()
X_scaled  = scaler_X.fit_transform(X_all_raw).astype(np.float32)
X_test_sc = scaler_X.transform(X_test_raw).astype(np.float32)
X_test_t  = torch.tensor(X_test_sc, device=DEVICE)

# Recompute target stats
Y_all  = train_df[TARGETS].values.astype(np.float32)
t_mean = np.zeros(5, dtype=np.float32)
t_std  = np.ones(5,  dtype=np.float32)
for i in range(5):
    v = Y_all[~np.isnan(Y_all[:,i]), i]
    t_mean[i], t_std[i] = v.mean(), v.std() + 1e-8

print(f"  Feature dim : {FEAT_DIM}")


# ── Model definitions ─────────────────────────────────────────

class ResBlock(nn.Module):
    def __init__(self, dim, dropout=0.2):
        super().__init__()
        self.block = nn.Sequential(
            nn.LayerNorm(dim), nn.Linear(dim,dim), nn.GELU(),
            nn.Dropout(dropout), nn.Linear(dim,dim),
        )
        self.act = nn.GELU()
    def forward(self, x): return self.act(x + self.block(x))


class SpecialistNet(nn.Module):
    def __init__(self, feat_dim, hidden=512, dropout=0.30):
        super().__init__()
        self.enc = nn.Sequential(
            nn.Linear(feat_dim, hidden*2), nn.LayerNorm(hidden*2), nn.GELU(),
            nn.Dropout(dropout), nn.Linear(hidden*2, hidden), nn.LayerNorm(hidden), nn.GELU(),
        )
        self.res  = nn.Sequential(*[ResBlock(hidden, dropout) for _ in range(4)])
        self.head = nn.Sequential(
            nn.Linear(hidden, hidden//2), nn.GELU(), nn.Dropout(dropout*0.5),
            nn.Linear(hidden//2, hidden//4), nn.GELU(), nn.Linear(hidden//4, 1),
        )
    def forward(self, x): return self.head(self.res(self.enc(x))).squeeze(-1)


class MainPINN(nn.Module):
    def __init__(self, feat_dim, hidden=768, n=5, dropout=0.2):
        super().__init__()
        self.proj = nn.Sequential(
            nn.Linear(feat_dim, hidden*2), nn.LayerNorm(hidden*2), nn.GELU(),
            nn.Dropout(dropout), nn.Linear(hidden*2, hidden), nn.LayerNorm(hidden), nn.GELU(),
        )
        self.backbone = nn.Sequential(*[ResBlock(hidden, dropout) for _ in range(6)])
        self.heads = nn.ModuleList([
            nn.Sequential(
                nn.Linear(hidden, hidden//2), nn.GELU(), nn.Dropout(dropout*0.5),
                nn.Linear(hidden//2, hidden//4), nn.GELU(), nn.Linear(hidden//4, 1),
            ) for _ in range(n)
        ])
    def forward(self, x):
        h = self.backbone(self.proj(x))
        return torch.cat([hd(h) for hd in self.heads], dim=1)


def load_spec(path, hidden=512, dropout=0.30):
    ckpt = torch.load(path, map_location=DEVICE, weights_only=False)
    m = SpecialistNet(FEAT_DIM, hidden=hidden, dropout=dropout).to(DEVICE)
    m.load_state_dict(ckpt["state"])
    m.eval()
    return m, float(ckpt["mu"]), float(ckpt["sig"])

def predict_spec(model, mu, sig, X_t):
    with torch.no_grad():
        return model(X_t).cpu().numpy() * sig + mu


# ── Generate all test predictions ─────────────────────────────
print("\n[2/3] Generating predictions from all specialists...")

# Load all 4 neural specialists
tg_m, tg_mu, tg_sig = load_spec(WORK+"pinn_tg_spec.pth",      dropout=0.30)
tc_m, tc_mu, tc_sig = load_spec(WORK+"pinn_tc_spec.pth",      dropout=0.25)
rg_m, rg_mu, rg_sig = load_spec(WORK+"pinn_rg_spec_v2.pth",   dropout=0.25)
dn_m, dn_mu, dn_sig = load_spec(WORK+"pinn_density_spec.pth", dropout=0.25)

# Load main PINN (for FFV — its best source)
main_ckpt  = torch.load(WORK+"pinn_main.pth", map_location=DEVICE, weights_only=False)
main_model = MainPINN(FEAT_DIM).to(DEVICE)
main_model.load_state_dict(main_ckpt)
main_model.eval()
with torch.no_grad():
    main_sc = main_model(X_test_t).cpu().numpy()

# Final per-target predictions
final_preds = {
    "Tg":      predict_spec(tg_m, tg_mu, tg_sig, X_test_t),   # specialist R²=0.9730
    "FFV":     main_sc[:, 1] * t_std[1] + t_mean[1],           # main PINN  R²=0.8483
    "Tc":      predict_spec(tc_m, tc_mu, tc_sig, X_test_t),    # specialist R²=0.9868
    "Density": predict_spec(dn_m, dn_mu, dn_sig, X_test_t),   # specialist R²=0.9972
    "Rg":      predict_spec(rg_m, rg_mu, rg_sig, X_test_t),   # specialist R²=0.9830
}

# Honest R² values used for ensemble weighting
M3_R2 = {
    "Tg":      0.9730,
    "FFV":     0.8483,   # main PINN — best achievable with current features
    "Tc":      0.9868,
    "Density": 0.9972,
    "Rg":      0.9830,
}

sub3 = pd.DataFrame({"id": test_df["id"]})
for t in TARGETS:
    sub3[t] = final_preds[t]
sub3.to_csv(WORK + "submission_model3.csv", index=False)
print(f"✓ submission_model3.csv saved")
print(sub3.to_string())

print(f"\n  Model 3 R² per target (honest):")
for t in TARGETS:
    flag = "✓" if M3_R2[t] >= 0.90 else "→"
    print(f"    {t:<12}: {M3_R2[t]:.4f}  {flag}")
print(f"    {'─'*30}")
print(f"    AVERAGE    : {np.mean(list(M3_R2.values())):.4f}")


# ── Final 3-model per-target ensemble ─────────────────────────
print("\n[3/3] Building final ensemble...")

M1_R2 = {"Tg": 0.72, "FFV": 0.80, "Tc": 0.79, "Density": 0.77, "Rg": 0.72}
M2_R2 = {"Tg": 0.78, "FFV": 0.87, "Tc": 0.84, "Density": 0.85, "Rg": 0.75}

try:
    sub1 = pd.read_csv(WORK + "submission.csv")
    sub2 = pd.read_csv(WORK + "submission_model2.csv")

    ensemble = pd.DataFrame({"id": sub1["id"]})

    print(f"\n  {'Target':<12} {'w_M1':>7} {'w_M2':>7} {'w_M3':>7} │ "
          f"{'M1':>6} {'M2':>6} {'M3':>6}")
    print(f"  {'─'*58}")

    for t in TARGETS:
        w1  = max(0.01, M1_R2[t])
        w2  = max(0.01, M2_R2[t])
        w3  = max(0.01, M3_R2[t])
        tot = w1 + w2 + w3
        w1n, w2n, w3n = w1/tot, w2/tot, w3/tot
        ensemble[t] = w1n*sub1[t] + w2n*sub2[t] + w3n*sub3[t]
        flag = "✓" if M3_R2[t] >= 0.90 else "→"
        print(f"  {t:<12} {w1n:>7.3f} {w2n:>7.3f} {w3n:>7.3f} │ "
              f"{M1_R2[t]:>6.4f} {M2_R2[t]:>6.4f} {M3_R2[t]:>6.4f} {flag}")

    ensemble.to_csv(WORK + "submission_ensemble.csv", index=False)
    print(f"\n✓ submission_ensemble.csv saved")
    print(ensemble.to_string())

except FileNotFoundError as e:
    print(f"  ⚠ {e} — saving M3 standalone")
    sub3.to_csv(WORK + "submission_ensemble.csv", index=False)

avg_m3 = np.mean(list(M3_R2.values()))
print(f"\n{'='*60}")
print(f"  FINAL SUMMARY")
print(f"{'='*60}")
print(f"  {'Target':<12} {'M1':>8} {'M2':>8} {'M3':>8}")
print(f"  {'─'*40}")
for t in TARGETS:
    flag = "✓" if M3_R2[t] >= 0.90 else "→"
    print(f"  {t:<12} {M1_R2[t]:>8.4f} {M2_R2[t]:>8.4f} {M3_R2[t]:>8.4f} {flag}")
print(f"  {'─'*40}")
print(f"  {'AVERAGE':<12} {np.mean(list(M1_R2.values())):>8.4f} "
      f"{np.mean(list(M2_R2.values())):>8.4f} {avg_m3:>8.4f}")
print(f"\n  Model 3 avg R² : {avg_m3:.4f}")
print(f"  ➤ Submit: /kaggle/working/submission_ensemble.csv")
print(f"{'='*60}")

Device : cuda

[1/3] Loading data + features...
  Feature dim : 3456

[2/3] Generating predictions from all specialists...
✓ submission_model3.csv saved
           id          Tg       FFV        Tc   Density         Rg
0  1109053969  151.819244  0.373745  0.187450  1.140589  23.516932
1  1422188626  158.340179  0.373748  0.309692  1.138498  22.466808
2  2032016830   52.943729  0.351162  0.247384  1.118534  20.618256

  Model 3 R² per target (honest):
    Tg          : 0.9730  ✓
    FFV         : 0.8483  →
    Tc          : 0.9868  ✓
    Density     : 0.9972  ✓
    Rg          : 0.9830  ✓
    ──────────────────────────────
    AVERAGE    : 0.9577

[3/3] Building final ensemble...

  Target          w_M1    w_M2    w_M3 │     M1     M2     M3
  ──────────────────────────────────────────────────────────
  Tg             0.291   0.315   0.393 │ 0.7200 0.7800 0.9730 ✓
  FFV            0.318   0.345   0.337 │ 0.8000 0.8700 0.8483 →
  Tc             0.302   0.321   0.377 │ 0.7900 0.8400 0.98

In [14]:
# ================================================================
#  Download all output files as a single ZIP
# ================================================================

import os
import zipfile
from IPython.display import FileLink, display

WORK     = "/kaggle/working/"
ZIP_PATH = WORK + "model3_all_outputs.zip"

# Files to include
files_to_zip = [
    # Submissions
    "submission_model3.csv",
    "submission_ensemble.csv",

    # Model checkpoints
    "pinn_main.pth",
    "pinn_tg_spec.pth",
    "pinn_tc_spec.pth",
    "pinn_rg_spec_v2.pth",
    "pinn_density_spec.pth",
    "pinn_ffv_spec.pth",

    # Cached features (large but useful to keep)
    "feats_train_v2.npy",
    "feats_test_v2.npy",
]

print("Zipping files...")
print(f"{'─'*50}")

added, skipped = [], []

with zipfile.ZipFile(ZIP_PATH, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for fname in files_to_zip:
        fpath = WORK + fname
        if os.path.exists(fpath):
            size_mb = os.path.getsize(fpath) / 1e6
            zf.write(fpath, arcname=fname)
            added.append((fname, size_mb))
            print(f"  ✓ {fname:<40} {size_mb:>7.1f} MB")
        else:
            skipped.append(fname)
            print(f"  ✗ {fname:<40} (not found)")

zip_size = os.path.getsize(ZIP_PATH) / 1e6
print(f"{'─'*50}")
print(f"  Added   : {len(added)} files")
print(f"  Skipped : {len(skipped)} files")
print(f"  ZIP size: {zip_size:.1f} MB")
print(f"\n✓ Saved: {ZIP_PATH}")
print("\nClick below to download:")
display(FileLink(ZIP_PATH))

Zipping files...
──────────────────────────────────────────────────
  ✓ submission_model3.csv                        0.0 MB
  ✓ submission_ensemble.csv                      0.0 MB
  ✓ pinn_main.pth                               61.8 MB
  ✓ pinn_tg_spec.pth                            25.4 MB
  ✓ pinn_tc_spec.pth                            25.4 MB
  ✓ pinn_rg_spec_v2.pth                         25.4 MB
  ✓ pinn_density_spec.pth                       25.4 MB
  ✓ pinn_ffv_spec.pth                           25.4 MB
  ✓ feats_train_v2.npy                         134.9 MB
  ✓ feats_test_v2.npy                            0.0 MB
──────────────────────────────────────────────────
  Added   : 10 files
  Skipped : 0 files
  ZIP size: 180.4 MB

✓ Saved: /kaggle/working/model3_all_outputs.zip

Click below to download:


/kaggle/working/model3_all_outputs.zip

In [16]:
# ================================================================
#  Download ALL files from /kaggle/working as a single ZIP
# ================================================================

import os
import zipfile
from IPython.display import FileLink, display

WORK     = "/kaggle/working/"
ZIP_PATH = WORK + "ALL_outputs.zip"

print("Scanning /kaggle/working ...")
print(f"{'─'*55}")

with zipfile.ZipFile(ZIP_PATH, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    total_size = 0
    count = 0
    for root, dirs, files in os.walk(WORK):
        # Skip the zip itself and hidden/virtual folders
        dirs[:] = [d for d in dirs if not d.startswith(".")]
        for fname in files:
            fpath = os.path.join(root, fname)
            if fpath == ZIP_PATH:
                continue
            arcname = os.path.relpath(fpath, WORK)
            size_mb = os.path.getsize(fpath) / 1e6
            zf.write(fpath, arcname=arcname)
            total_size += size_mb
            count += 1
            print(f"  ✓ {arcname:<45} {size_mb:>7.1f} MB")

zip_size = os.path.getsize(ZIP_PATH) / 1e6
print(f"{'─'*55}")
print(f"  Files zipped : {count}")
print(f"  Raw total    : {total_size:.1f} MB")
print(f"  ZIP size     : {zip_size:.1f} MB")
print(f"\n✓ Saved to: {ZIP_PATH}")
print("\nClick the link below to download:")
display(FileLink("ALL_outputs.zip"))

Scanning /kaggle/working ...
───────────────────────────────────────────────────────
  ✓ pinn_density_spec.pth                            25.4 MB
  ✓ pinn_tg_spec.pth                                 25.4 MB
  ✓ model3_all_outputs.zip                          180.4 MB
  ✓ pinn_ffv_spec.pth                                25.4 MB
  ✓ model3_pinn.pth                                  38.6 MB
  ✓ submission.csv                                    0.0 MB
  ✓ feats_test_v2.npy                                 0.0 MB
  ✓ submission_model3.csv                             0.0 MB
  ✓ pinn_tc_spec.pth                                 25.4 MB
  ✓ pinn_ffv_spec_v2.pth                             55.9 MB
  ✓ ffv_xgb.json                                      0.3 MB
  ✓ model2_transformer.pth                           24.7 MB
  ✓ submission_model2.csv                             0.0 MB
  ✓ pinn_rg_spec.pth                                 25.4 MB
  ✓ model1_gnn.pth                                   30.0 MB


/kaggle/working/ALL_outputs.zip